# 01 — Pruning YOLO26s dengan Target GFLOPs
Notebook standalone khusus **YOLO26s** untuk pruning Poltrada V3. Pruning terstruktur memakai LAMP + DepGraph.
Target 1,02×, 1,05×, 1,10×, 1,20×, 1,30×, 1,40×, 1,50×, 1,60×, 1,70× adalah GFLOPs baseline dibagi GFLOPs pruned.
Latency median baseline dibagi latency median pruned dilaporkan terpisah.
Contoh MP-YOLO memakai 1,7×. Adaptasi YOLO26 ini mempertahankan dual head dan dimensi antarmuka,
serta memakai ulang baseline yang sama untuk setiap kandidat. Training, recovery, dan KD tidak dijalankan.
[MP-YOLO](https://github.com/Wang-jj-zs/MP-YOLO/blob/master/MP-YOLO/compress.py) ·
[LAMP](https://arxiv.org/abs/2010.07611) · [DepGraph](https://arxiv.org/abs/2301.12900)

Pada Kaggle aktifkan GPU, Internet untuk instalasi/unduh artifact, dan tambahkan **Poltrada V3 YOLO26 Clean**.
Sediakan secret `WANDB_API_KEY` untuk mengunduh checkpoint atau isi path checkpoint lokal model ini.
SHA-256 checkpoint YOLO26s v62 dikunci. Logging W&B dan unggah artefak aktif; raw dipublikasikan sebelum recovery.
Default `MODEL_VARIANTS=["s"]` dan `MODE="full_val"` menjalankan model ini
pada seluruh validation Poltrada V3. Varian lainnya mempunyai notebook sendiri.
Untuk pemeriksaan awal Kaggle, pilih `MODE="smoke"` sebelum Run All;
smoke mengevaluasi subset tetap hingga 128 citra val. `test` memerlukan lock dari full_val.
Hasil Kaggle dan mAP Poltrada V3 harus berasal dari eksekusi nyata.

## 02 — Konfigurasi YOLO26s
Default eksperimen memakai YOLO26s v62, seed 42, FP32 640 px, batch validasi 16,
serta target GFLOPs 1,02×/1,05×/1,10×/1,20×/1,30×/1,40×/1,50×/1,60×/1,70×. Kontrol extended/P5 opsional melalui RUN_DIAGNOSTIC_CONTROLS; default nonaktif.
Setiap konfigurasi memuat baseline asli variannya. Pruning menjalankan nol epoch training.
Pilih mode, varian, sumber checkpoint, dan dataset pada sel berikut.

| Mode | Data dan tujuan |
| --- | --- |
| `smoke` | Maksimal 128 citra val dengan subset tetap untuk memeriksa pipeline |
| `full_val` | Seluruh val model ini; ekspor kandidat untuk perbandingan N/S |
| `test` | Split independen untuk baseline dan student yang sudah dikunci |

In [ ]:
import os
from pathlib import Path

MODEL_VARIANTS = ["s"]
WANDB_ENTITY = "ilhamrafiqin-brawijaya-university"
WANDB_PROJECT = "yolom26-kd-indotraffic"
MODEL_ARTIFACTS = {
    "n": "ilhamrafiqin-brawijaya-university/yolom26-kd-indotraffic/03_baseline_yolo26n-9uu3meve-checkpoints:v84",
    "s": "ilhamrafiqin-brawijaya-university/yolom26-kd-indotraffic/02_baseline_yolo26s-q5kp9k9d-checkpoints:v62",
}
LOCAL_CHECKPOINT_PATHS = {"n": None, "s": None}
LOCAL_EXPECTED_SHA256S = {"n": None, "s": None}
CHECKPOINT_SELECTOR = "best.pt"
DATASET_ROOT = Path("/kaggle/input/datasets/ilhamrafiqin/poltrada-v3-yolo26-clean")
DATA_YAML = DATASET_ROOT / "data.yaml"
OUTPUT_DIR = Path("/kaggle/working/pruning/lamp_gflops_yolo26s") if Path("/kaggle").exists() else Path("outputs/pruning/lamp_gflops_yolo26s")
PRUNING_POLICY = "extended"
PRUNING_RATIOS = [0.10, 0.20, 0.30, 0.40, 0.50]
GLOBAL_PRUNING = True
ITERATIVE_STEPS = 1
MAX_PRUNING_RATIO = 0.50
ROUND_TO = None
IMPORTANCE_P = 2
SEED = 42
IMGSZ = 640
DEVICE = "auto"
RUN_FULL_VALIDATION = True
RUN_LATENCY_BENCHMARK = True
BATCH = 16
WORKERS = 2
LATENCY_WARMUP = 50
LATENCY_REPEATS = 200
RELOAD_TIMEOUT_SECONDS = 300
INSTALL_DEPENDENCIES = True
WANDB_TRACKING = True
UPLOAD_WANDB_ARTIFACTS = True
EDA_DPI = 300
EDA_MAX_LAYERS_PER_PAGE = 32
EDA_DISPLAY_RATIO = 0.50
RECOVERY_ENABLED = False
FINETUNING_ENABLED = False
KD_ENABLED = False

# full_val: eksperimen model ini. Gabungkan ekspor N/S untuk memilih satu student akhir.
MODE = "full_val"
# MODE = "smoke"
# MODE = "test"  # Memerlukan STUDENT_LOCK_PATH dari full_val.
GFLOPS_TARGETS = (1.02, 1.05, 1.10, 1.20, 1.30, 1.40, 1.50, 1.60, 1.70)
RUN_DIAGNOSTIC_CONTROLS = False
RESUME_SUITE_DIR = None
DATASET_MANIFEST_PATH = None
EXPORT_DATASET_MANIFEST = True
DATASET_MANIFEST_ONLY = False
SOURCE_CHECKPOINT_SHA256 = {
    "n": "380a087fd621e8a43fe2b9b4d6b1bb6fb00a4d0443cc5afe81ef3efdf0775f0f",
    "s": "5bc3ae61f8da1c9f03916922de10146805c27c64a11e46edd766a00d0f6f18fd",
}
LOCAL_EXPECTED_SHA256S = SOURCE_CHECKPOINT_SHA256.copy()
DATASET_ID = "ilhamrafiqin/poltrada-v3-yolo26-clean"
EXPECTED_DATASET_FINGERPRINT = None
STUDENT_LOCK_PATH = None
TEST_STUDENT_CHECKPOINT_PATH = None
TEST_BASELINE_CHECKPOINT_PATH = None
ALLOW_ARTIFACT_DOWNLOAD = True
# Setelah kedua full_val selesai, lampirkan seluruh folder output varian lain.
# Isi dengan path student_candidates.json varian lain; sel terakhir dapat dijalankan ulang.
COMPARISON_CANDIDATE_PATHS = ()
# () menampilkan overview semua konfigurasi. Detail dapat diperluas tanpa mengulang pruning.
EDA_CONFIGURATIONS = ()
EDA_DETAIL_CONFIGURATIONS = ()

PLATFORM = 'kaggle'
NOTEBOOK_VARIANT = 's'
IMPLEMENTATION_SHA256 = 'afe8fb4516e8a0a79137dc18aca87894180a356e98de79c061ddadb5ce63eb9a'

## 03 — Environment dan Dependensi
Notebook memakai versi Ultralytics dan Torch-Pruning yang dikunci serta mencatat lingkungan tiap suite.

In [ ]:
import csv
import hashlib
import importlib.metadata as metadata
import inspect
import json
import math
import os
import platform
import random
import subprocess
import sys
import time
import traceback
import uuid
import warnings
from contextlib import contextmanager
from copy import deepcopy
from datetime import datetime, timezone
from types import SimpleNamespace

PINNED = {"ultralytics": "8.4.155", "wandb": "0.28.1", "torch-pruning": "1.6.1"}
REQUIRED_CHECKS = (
    "baseline_loaded", "baseline_forward", "dependency_graph", "structural_channels_removed",
    "parameters_reduced", "pruned_forward", "protected_head_unchanged", "checkpoint_saved",
    "checkpoint_reloaded", "reload_consistent", "detection_prediction",
)


def write_json(path, value):
    """Tulis JSON standar; NaN/Inf ditolak, bukan disamarkan sebagai metrik."""
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")


def sha256_file(path):
    """Hash berkas tanpa membaca seluruh checkpoint ke RAM sekaligus."""
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

In [ ]:
# Backward-compatible N helper aliases; public inputs are the per-model mappings.
WANDB_ARTIFACT = MODEL_ARTIFACTS["n"]
LOCAL_CHECKPOINT_PATH = None
LOCAL_EXPECTED_SHA256 = None


def model_spec(model_variant="n"):
    if model_variant not in {"n", "s"}:
        raise ValueError("Model variant harus n atau s.")
    local = LOCAL_CHECKPOINT_PATHS.get(model_variant)
    expected_hash = LOCAL_EXPECTED_SHA256S.get(model_variant)
    if model_variant == "n" and LOCAL_CHECKPOINT_PATH is not None:
        local, expected_hash = LOCAL_CHECKPOINT_PATH, LOCAL_EXPECTED_SHA256
    return {"model_variant": model_variant, "model_name": f"YOLO26{model_variant}",
            "artifact": MODEL_ARTIFACTS[model_variant], "local_checkpoint": local,
            "expected_sha256": expected_hash}


def policies_to_run(policy):
    if policy not in {"conservative", "extended", "both"}:
        raise ValueError("PRUNING_POLICY harus conservative, extended, atau both.")
    return ["conservative", "extended"] if policy == "both" else [policy]


def configuration(policy=None, ratio=None, model_variant=None):
    keys = ("MODEL_VARIANTS MODEL_ARTIFACTS LOCAL_CHECKPOINT_PATHS LOCAL_EXPECTED_SHA256S CHECKPOINT_SELECTOR "
            "DATASET_ROOT DATA_YAML OUTPUT_DIR PRUNING_POLICY PRUNING_RATIOS GLOBAL_PRUNING "
            "ITERATIVE_STEPS MAX_PRUNING_RATIO ROUND_TO IMPORTANCE_P SEED IMGSZ DEVICE "
            "RUN_FULL_VALIDATION RUN_LATENCY_BENCHMARK BATCH WORKERS INSTALL_DEPENDENCIES "
            "WANDB_TRACKING UPLOAD_WANDB_ARTIFACTS LATENCY_WARMUP LATENCY_REPEATS "
            "RELOAD_TIMEOUT_SECONDS RECOVERY_ENABLED FINETUNING_ENABLED KD_ENABLED "
            "EDA_DPI EDA_MAX_LAYERS_PER_PAGE EDA_DISPLAY_RATIO").split()
    def serializable(value):
        if isinstance(value, Path):
            return str(value)
        if isinstance(value, dict):
            return {key: serializable(item) for key, item in value.items()}
        if isinstance(value, (tuple, list)):
            return [serializable(item) for item in value]
        return value
    result = {key: serializable(globals()[key]) for key in keys}
    if policy is not None:
        result["policy"] = policy
    if ratio is not None:
        result["ratio"] = ratio
    if model_variant is not None:
        result["selected_model"] = serializable(model_spec(model_variant))
    return result


def validate_configuration():
    if not MODEL_VARIANTS or len(set(MODEL_VARIANTS)) != len(MODEL_VARIANTS):
        raise ValueError("MODEL_VARIANTS harus nonkosong dan tanpa duplikat.")
    for model_variant in MODEL_VARIANTS:
        artifact = model_spec(model_variant)["artifact"]
        if artifact.count("/") != 2 or ":" not in artifact:
            raise ValueError("Artifact wajib entity/project/artifact:version.")
    policies_to_run(PRUNING_POLICY)
    if not PRUNING_RATIOS or len(set(PRUNING_RATIOS)) != len(PRUNING_RATIOS):
        raise ValueError("PRUNING_RATIOS harus nonkosong dan tanpa duplikat.")
    if any(isinstance(r, bool) or not isinstance(r, (int, float)) or not math.isfinite(r)
           or not 0 < r <= MAX_PRUNING_RATIO for r in PRUNING_RATIOS):
        raise ValueError("Setiap ratio wajib finite dan 0 < ratio <= MAX_PRUNING_RATIO.")
    if MAX_PRUNING_RATIO != 0.50 or ROUND_TO is not None or not GLOBAL_PRUNING:
        raise ValueError("V2 memakai global ranking, cap 0.50, dan ROUND_TO=None.")
    if ITERATIVE_STEPS != 1 or IMPORTANCE_P != 2:
        raise ValueError("V2 hanya mendukung one-shot dengan p=2.")
    if any((RECOVERY_ENABLED, FINETUNING_ENABLED, KD_ENABLED)):
        raise ValueError("Recovery, fine-tuning, dan KD wajib disabled.")
    if IMGSZ < 32 or IMGSZ % 32 or BATCH < 1 or WORKERS < 0:
        raise ValueError("IMGSZ kelipatan 32, BATCH positif, WORKERS nonnegatif.")
    if EDA_DPI < 72 or EDA_MAX_LAYERS_PER_PAGE < 1 or not 0 < EDA_DISPLAY_RATIO <= 0.5:
        raise ValueError("Konfigurasi EDA DPI/page/ratio invalid.")
    if LATENCY_WARMUP < 1 or LATENCY_REPEATS < 1 or RELOAD_TIMEOUT_SECONDS < 1:
        raise ValueError("Jumlah pengukuran dan timeout wajib positif.")


def write_csv(path, rows):
    if not rows:
        return
    keys = list(dict.fromkeys(key for row in rows for key in row))
    with Path(path).open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=keys)
        writer.writeheader()
        for row in rows:
            writer.writerow({key: json.dumps(value, ensure_ascii=False) if isinstance(value, (list, dict))
                             else value for key, value in row.items()})


def model_state_sha256(core):
    """Buktikan setiap konfigurasi dimulai dari bobot/buffer FP32 yang identik."""
    digest = hashlib.sha256()
    for name, value in sorted(core.state_dict().items()):
        tensor = value.detach().cpu().contiguous()
        digest.update(name.encode())
        digest.update(str((str(tensor.dtype), tuple(tensor.shape))).encode())
        digest.update(tensor.numpy().tobytes())
    return digest.hexdigest()


def metric_comparison(before, after):
    result = {}
    for key, label in (("parameters", "parameter"), ("gflops", "gflops"),
                       ("checkpoint_bytes", "checkpoint")):
        base, pruned = before.get(key), after.get(key)
        result[label + "_reduction_pct"] = 100 * (base - pruned) / base if base else None
    base = (before.get("validation") or {}).get("map50_95")
    pruned = (after.get("validation") or {}).get("map50_95")
    delta = pruned - base if base is not None and pruned is not None else None
    result.update({"delta_map50_95": delta, "delta_map50_95_pp": 100 * delta if delta is not None else None,
                   "relative_map50_95_pct": 100 * delta / base if delta is not None and base else None})
    return result


def lamp_v2_display_name(policy, ratio_tag, model_variant, recovery_epochs=None):
    model_spec(model_variant)
    policy_suffix = "_c" if policy == "conservative" else ""
    recovery_suffix = f"_ft{recovery_epochs}" if recovery_epochs is not None else ""
    return f"prun_lamp2_yolo26{model_variant}{policy_suffix}{recovery_suffix}_pruned-{ratio_tag}"


def run_identity(policy, ratio, suite_id, model_variant="n"):
    tag = f"pr{ratio:g}"
    return {"name": lamp_v2_display_name(policy, tag, model_variant),
            "id": uuid.uuid4().hex, "resume": "never", "group": f"lamp-v2-{suite_id}",
            "job_type": "lamp-structured-pruning-v2",
            "tags": [f"yolo26{model_variant}", "lamp", "v2", "structured", "pruned", policy,
                     "no-recovery", tag, f"seed{SEED}"]}


def safe_error(error):
    message = str(error)
    secret = os.environ.get("WANDB_API_KEY")
    return message.replace(secret, "[REDACTED]") if secret else message

In [ ]:
def ensure_dependencies(output_dir, install=True):
    """Pasang dependency tambahan sambil mengunci Torch/torchvision environment."""
    if sys.version_info < (3, 11):
        raise RuntimeError("Gunakan Python >=3.11.")
    preserved = {name: metadata.version(name) for name in ("torch", "torchvision")}
    needed = []
    for name, expected in PINNED.items():
        try:
            found = metadata.version(name)
        except metadata.PackageNotFoundError:
            found = None
        if found != expected:
            needed.append(f"{name}=={expected}")
    if needed and not install:
        raise RuntimeError(f"Dependency belum cocok: {needed}; aktifkan INSTALL_DEPENDENCIES.")
    if needed:
        constraints = Path(output_dir) / "preserve_torch.txt"
        constraints.write_text("\n".join(f"{k}=={v}" for k, v in preserved.items()) + "\n", encoding="utf-8")
        subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                        "--constraint", str(constraints), *needed], check=True)
    actual = {name: metadata.version(name) for name in preserved}
    if actual != preserved:
        raise RuntimeError(f"Torch/torchvision berubah: {preserved} -> {actual}; restart runtime asli.")
    for name, expected in PINNED.items():
        if metadata.version(name) != expected:
            raise RuntimeError(f"Versi {name} tidak sesuai pin {expected}; restart kernel bila perlu.")

In [ ]:
"""Shared standalone persistence, provenance and W&B helpers for pruning/recovery."""
# ruff: noqa: F821 -- Notebook declarations supply SDK imports and configuration.
import json
import os
import re
import uuid
from contextlib import contextmanager
from datetime import UTC
from pathlib import Path


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
    os.replace(temporary, path)


def write_csv(path, rows):
    import csv
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")
    keys = list(dict.fromkeys(key for row in rows for key in row))
    with temporary.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=keys)
        writer.writeheader()
        writer.writerows({key: json.dumps(value, ensure_ascii=False) if isinstance(value, (list, dict)) else value
                         for key, value in row.items()} for row in rows)
    os.replace(temporary, path)


def workflow_digest(value):
    import hashlib
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str, allow_nan=False).encode()).hexdigest()


def workflow_bind(folder, name, contract):
    """Reject semantic changes on resume, allowing absolute paths to relocate."""
    path = Path(folder) / name
    contract = json.loads(json.dumps(contract, sort_keys=True, default=str, allow_nan=False))
    record = {"contract": contract, "sha256": workflow_digest(contract)}
    if path.is_file() and json.loads(path.read_text(encoding="utf-8")) != record:
        raise ValueError(f"Kontrak resume berbeda: {name}. Gunakan suite baru.")
    write_json(path, record)
    return record


def workflow_save_checkpoint(core, path, provenance):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        existing = YOLO(str(path), task="detect").model.cpu().float().eval()
        if model_state_sha256(existing) != model_state_sha256(core):
            raise ValueError(f"Checkpoint immutable sudah berisi state berbeda: {path}")
        return {"path": str(path.resolve()), "sha256": sha256_file(path), "file_bytes": path.stat().st_size}
    temporary = path.with_name(path.stem + "_" + uuid.uuid4().hex + ".pt")
    saved = save_fp32_checkpoint(core, temporary, provenance)
    os.replace(temporary, path)
    return {**saved, "path": str(path.resolve())}


def workflow_suite_dir(output, resume=None):
    if resume:
        path = Path(resume).expanduser().resolve(strict=True)
        if not path.is_dir():
            raise NotADirectoryError(path)
        return path
    from datetime import datetime
    path = Path(output).expanduser().resolve() / (datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8])
    path.mkdir(parents=True, exist_ok=False)
    return path


def workflow_status(statuses, *, useful=False):
    statuses = list(statuses)
    if statuses and all(status == "PASS" for status in statuses):
        return "PASS"
    if useful or any(status in {"PASS", "PARTIAL"} for status in statuses):
        return "PARTIAL"
    return "FAIL" if any(status == "FAIL" for status in statuses) else "NOT_RUN"


def workflow_run_name(variant, target, recovery=False):
    tag = f"{target:.2f}".replace(".", "p")
    method = globals().get("RECOVERY_METHOD", "finetune")
    recovery_tag = f"_ft{RECOVERY_EPOCHS}" if method == "finetune" else f"_{method}{RECOVERY_EPOCHS}"
    ft = recovery_tag if recovery else ""
    smoke = "_smoke" if MODE == "smoke" else ""
    return f"prun_lamp_gflops_yolo26{variant}_{PLATFORM}{ft}_su{tag}_seed{SEED}{smoke}"


def workflow_login():
    import getpass

    import wandb
    key = os.environ.get("WANDB_API_KEY")
    if not key and PLATFORM == "kaggle":
        try:
            from kaggle_secrets import UserSecretsClient
            key = UserSecretsClient().get_secret("WANDB_API_KEY")
        except (ImportError, RuntimeError, ValueError):
            pass
    if not key:
        key = getpass.getpass("WANDB_API_KEY: ")
    if not key:
        raise ValueError("W&B memerlukan API key untuk publikasi artefak.")
    os.environ["WANDB_API_KEY"] = key
    wandb.login(key=key, relogin=False)


def workflow_version_ref(reference):
    if not isinstance(reference, str) or not re.fullmatch(r"[^/:]+/[^/:]+/[^/:]+:v\d+", reference):
        raise ValueError("Input W&B memerlukan entity/project/artifact:vN (versi tetap).")
    return reference


@contextmanager
def workflow_tracking(folder, variant, target, *, recovery=False, source_ref=None, group=None, registration=False):
    import wandb
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    run = None
    if not WANDB_TRACKING:
        yield None
        return
    identity_path = folder / "wandb_identity.json"
    existed = identity_path.is_file()
    name = workflow_run_name(variant, target, recovery) + ("_register_raw" if registration else "")
    identity = json.loads(identity_path.read_text(encoding="utf-8")) if existed else {
        "id": uuid.uuid4().hex, "name": name, "group": group or f"lamp-gflops-{PLATFORM}-{folder.parent.name}"}
    if identity["name"] != name:
        raise ValueError("Identitas run resume berbeda dari konfigurasi.")
    write_json(identity_path, identity)
    try:
        workflow_login()
        run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, name=name, id=identity["id"],
                         resume="allow" if existed else "never", group=identity["group"],
                         job_type=("register-raw" if registration else
                                   globals().get("RECOVERY_METHOD", "finetune") + "-recovery" if recovery else "pruning-no-recovery"),
                         tags=[PLATFORM, f"yolo26{variant}", "lamp", "depgraph", f"seed{SEED}",
                               "recovery" if recovery else "raw-pruned", MODE],
                         config={"target_speedup": target, "model_variant": variant, "platform": PLATFORM,
                                 "seed": SEED, "mode": MODE})
        if source_ref:
            run.use_artifact(workflow_version_ref(source_ref), type="model")
        yield run
    finally:
        if run is not None:
            run.finish()


class WorkflowUploadPending(RuntimeError):
    """Local computation is committed; only publication needs retrying."""


def workflow_publish(run, folder, label, files, *, metadata_info=None, aliases=None, event=None, kind="model"):
    import wandb
    folder = Path(folder)
    path = folder / "artifact_publications.json"
    publications = json.loads(path.read_text(encoding="utf-8")) if path.is_file() else {}
    files = {name: Path(value).resolve(strict=True) for name, value in files.items()}
    hashes = {name: sha256_file(value) for name, value in files.items()}
    key = label + (":" + event if event else "")
    prior = publications.get(key)
    if prior and prior.get("status") == "CONFIRMED" and prior.get("files_sha256") == hashes:
        return prior
    record = {"status": "NOT_RUN", "files_sha256": hashes, "label": label, "event": event}
    if run is None:
        publications[key] = record
        write_json(path, publications)
        return record
    try:
        artifact = wandb.Artifact(f"{run.name}-{run.id}-{label}", type=kind,
                                 metadata={**(metadata_info or {}), "files_sha256": hashes})
        for name, value in files.items():
            artifact.add_file(str(value), name=name)
        uploaded = run.log_artifact(artifact, aliases=aliases or ["latest"])
        uploaded.wait(timeout=300)
        reference = workflow_version_ref(uploaded.qualified_name)
        record.update(status="CONFIRMED", reference=reference)
    except Exception as error:
        record.update(status="FAIL", error=safe_error(error))
        publications[key] = record
        write_json(path, publications)
        raise WorkflowUploadPending(safe_error(error)) from error
    publications[key] = record
    write_json(path, publications)
    return record


def workflow_report_files(folder):
    folder = Path(folder)
    return {path.relative_to(folder).as_posix(): path for path in folder.rglob("*")
            if path.is_file() and path.suffix in {".json", ".csv", ".yaml", ".md", ".png", ".svg"}
            and "wandb" not in path.relative_to(folder).parts and path.name != "artifact_publications.json"}


def workflow_dataset_manifest(dataset, source_yaml, root, folder):
    """Portable full train/val content identity, never an evaluation train=val placeholder."""
    raw = yaml.safe_load(Path(source_yaml).read_text(encoding="utf-8-sig"))
    names = ordered_names(raw["names"])
    root = Path(root).resolve()
    train = _gflops_split_images(root, raw.get("train"))
    val = _gflops_split_images(root, raw.get("val"))
    if any(image.with_suffix(".npy").is_file() for image in (*train, *val)):
        raise ValueError("Gunakan sumber dataset tanpa cache gambar .npy.")
    train_fp, train_rows = _gflops_fingerprint(root, train, "train", names)
    val_fp, val_rows = _gflops_fingerprint(root, val, "val", names)
    if {row["image_sha256"] for row in train_rows}.intersection(row["image_sha256"] for row in val_rows):
        raise ValueError("Train dan val berbagi konten gambar; recovery tidak boleh memakai train=val.")
    record = {"format_version": 1, "dataset_id": DATASET_ID, "class_names": names,
              "train_fingerprint": train_fp, "val_fingerprint": val_fp,
              "provenance_status": dataset.get("provenance_status", "UNVERIFIED"),
              "train_count": len(train), "val_count": len(val), "source_yaml_sha256": sha256_file(source_yaml)}
    write_json(Path(folder) / "dataset_manifest.json", record)
    return record, train, val

In [ ]:
SUITE_DIR = workflow_suite_dir(OUTPUT_DIR, RESUME_SUITE_DIR)
try:
    ensure_dependencies(SUITE_DIR, install=INSTALL_DEPENDENCIES)
except Exception as error:
    failure = {"status": "FAIL", "stage": "environment", "error": safe_error(error)}
    write_json(SUITE_DIR / "environment_failure.json", failure)
    if not (SUITE_DIR / "suite_manifest.json").is_file():
        write_json(SUITE_DIR / "suite_manifest.json", failure)
    raise

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch_pruning as tp
import ultralytics
import wandb
import yaml
from PIL import Image
from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset
from ultralytics.nn.modules import Bottleneck, C2PSA, Detect
from ultralytics.nn.modules.block import AAttn, Attention
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils.torch_utils import _attention_ops
from thop import profile as thop_profile

In [ ]:
def seed_environment(seed):
    """Samakan RNG dan backend FP32 di notebook dan proses restore."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.set_float32_matmul_precision("highest")


def environment_report():
    """Periksa API aktual, bukan API yang diasumsikan dari contoh YOLOv8."""
    for name, expected in PINNED.items():
        if metadata.version(name) != expected:
            raise RuntimeError(f"Pin {name} tidak terpenuhi.")
    if ultralytics.__version__ != PINNED["ultralytics"]:
        raise RuntimeError("Modul Ultralytics lama masih di kernel; restart runtime.")
    signature = inspect.signature(tp.pruner.BasePruner)
    for key in ("global_pruning", "max_pruning_ratio", "round_to", "output_transform", "forward_fn"):
        if key not in signature.parameters:
            raise RuntimeError(f"BasePruner tidak mendukung {key}.")
    return {
        "python": sys.version, "platform": platform.platform(),
        "packages": {name: metadata.version(name) for name in (*PINNED, "torch", "torchvision", "numpy", "ultralytics-thop")},
        "cuda_available": torch.cuda.is_available(), "cuda_build": torch.version.cuda,
        "device": str(TARGET_DEVICE),
        "gpu": torch.cuda.get_device_name(TARGET_DEVICE) if TARGET_DEVICE.type == "cuda" else None,
        "precision": "FP32", "tf32": False, "cudnn_benchmark": False,
        "base_pruner_signature": str(signature),
        "lamp_signature": str(inspect.signature(tp.importance.LAMPImportance)),
        "lamp_source_sha256": hashlib.sha256(inspect.getsource(tp.importance.LAMPImportance).encode()).hexdigest(),
        "importance_module_sha256": sha256_file(inspect.getfile(tp.importance.LAMPImportance)),
        "flops_method": "THOP full-resolution Conv/BN + separate Ultralytics attention matmul counter; GFLOPs=2*MACs/1e9",
        "thop_profile_signature": str(inspect.signature(thop_profile)),
        "thop_distribution_versions": thop_versions(),
    }


def thop_versions():
    """Catat kedua distribution yang dapat menyediakan namespace thop yang sama."""
    result = {}
    for name in ("thop", "ultralytics-thop"):
        try:
            result[name] = metadata.version(name)
        except metadata.PackageNotFoundError:
            result[name] = None
    return result

In [ ]:
TARGET_DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu") if DEVICE == "auto" else torch.device(DEVICE)
seed_environment(SEED)
ENVIRONMENT = environment_report()
write_json(SUITE_DIR / "environment.json", ENVIRONMENT)
print("Runtime", TARGET_DEVICE, "| FP32 | Seed", SEED)
print("Kernel", sys.executable, "| cwd", Path.cwd(), "| platform", PLATFORM)
print("GPU", ENVIRONMENT.get("gpu"), "| CUDA", torch.version.cuda, "| Output", SUITE_DIR)
display(pd.DataFrame(ENVIRONMENT["packages"].items(), columns=["package", "version"]))

## 04 — Pengambilan Baseline
Periksa sumber dan SHA-256 checkpoint per varian. Sumber yang dipilih dimuat saat eksperimen berjalan.

In [ ]:
def select_checkpoint(root, selector="best.pt"):
    """Pilih checkpoint unik secara eksplisit; tidak menebak dari urutan filesystem."""
    root = Path(root).expanduser().resolve()
    if root.is_file():
        candidates = [root] if root.suffix.lower() == ".pt" else []
        base = root.parent
    else:
        candidates = sorted(p.resolve() for p in root.rglob("*.pt") if p.is_file())
        base = root
    print("Kandidat checkpoint:", [p.relative_to(base).as_posix() for p in candidates])
    if not candidates:
        raise FileNotFoundError(f"Tidak ada file .pt di {root}.")
    if root.is_file():
        return root  # LOCAL_CHECKPOINT_PATH berupa file adalah pilihan eksplisit.
    if selector:
        selector = str(selector).replace("\\", "/")
        matches = [p for p in candidates if p.relative_to(base).as_posix() == selector] if "/" in selector else [p for p in candidates if p.name == selector]
        if len(matches) == 1:
            return matches[0]
        raise ValueError(f"Selector {selector!r} memiliki {len(matches)} kecocokan; pilih path relatif tepat.")
    if len(candidates) == 1 and candidates[0].name == "best.pt":
        return candidates[0]
    raise ValueError("Checkpoint ambigu atau best.pt tidak tersedia; isi CHECKPOINT_SELECTOR secara eksplisit.")


def checkpoint_provenance(path, artifact_name, *, expected_hash=None, resolved_artifact=None):
    """Catat identitas sumber; hash lokal opsional wajib cocok bila diberikan."""
    digest = sha256_file(path)
    if expected_hash is not None and digest.lower() != expected_hash.lower():
        raise ValueError("SHA-256 checkpoint lokal tidak cocok dengan LOCAL_EXPECTED_SHA256.")
    return {"requested_artifact": artifact_name, "resolved_artifact": resolved_artifact,
            "checkpoint_name": Path(path).name, "checkpoint_path": str(Path(path).resolve()),
            "sha256": digest, "file_bytes": Path(path).stat().st_size,
            "source_verification": "wandb_download" if resolved_artifact else ("expected_sha256_matched" if expected_hash else "user_asserted_local_copy")}

In [ ]:
def wandb_session_login():
    workflow_login()


def obtain_baseline(suite_dir, model_variant="n"):
    """Ambil sumber satu kali via API; setiap run eksperimen dibuat terpisah."""
    spec = model_spec(model_variant)
    if spec["local_checkpoint"] is not None:
        path = select_checkpoint(spec["local_checkpoint"], CHECKPOINT_SELECTOR)
        return path, {**checkpoint_provenance(path, spec["artifact"], expected_hash=spec["expected_sha256"]),
                      "model_variant": model_variant}
    if not ALLOW_ARTIFACT_DOWNLOAD:
        raise ValueError("Isi LOCAL_CHECKPOINT_PATHS atau aktifkan ALLOW_ARTIFACT_DOWNLOAD untuk sumber W&B.")
    wandb_session_login()
    artifact = wandb.Api(timeout=30).artifact(spec["artifact"], type="model")
    root = artifact.download(root=str(Path(suite_dir) / "baseline" / "source_artifact"))
    path = select_checkpoint(root, CHECKPOINT_SELECTOR)
    provenance = checkpoint_provenance(path, spec["artifact"], resolved_artifact=artifact.qualified_name)
    provenance.update({"artifact_digest": artifact.digest, "artifact_id": artifact.id,
                       "artifact_version": artifact.version,
                       "artifact_relative_path": path.relative_to(Path(root).resolve()).as_posix(),
                       "model_variant": model_variant})
    return path, provenance

In [ ]:
if MODE != "test":
    display(pd.DataFrame([
        {**model_spec(variant), "source_sha256": SOURCE_CHECKPOINT_SHA256[variant]}
        for variant in MODEL_VARIANTS
    ]).reindex(columns=["model_name", "artifact", "local_checkpoint", "source_sha256"]))

## 05 — Audit Arsitektur YOLO26s
Periksa struktur backbone, neck, dan kedua cabang Detect. Ukur operasi pada model unfused.

In [ ]:
def ordered_names(names):
    """Pertahankan identitas dan urutan indeks kelas."""
    if isinstance(names, dict):
        indexed = {int(k): str(v) for k, v in names.items()}
        if sorted(indexed) != list(range(len(indexed))):
            raise ValueError("Indeks kelas wajib berurutan mulai dari nol.")
        return [indexed[i] for i in range(len(indexed))]
    return [str(value) for value in names]


def assert_class_match(model_names, dataset_names):
    """Jangan remap kelas diam-diam untuk membuat evaluasi berjalan."""
    if ordered_names(model_names) != ordered_names(dataset_names):
        raise ValueError(f"Urutan/nama kelas berbeda: model={ordered_names(model_names)}, dataset={ordered_names(dataset_names)}")


def architecture_audit(core, model_variant="n"):
    """Bangun allowlist hidden channel berdasarkan tipe modul dan hubungan konsumen."""
    if not isinstance(core, DetectionModel):
        raise TypeError("Checkpoint bukan DetectionModel Ultralytics.")
    heads = [(name, module) for name, module in core.named_modules() if isinstance(module, Detect)]
    if len(heads) != 1:
        raise ValueError("Diperlukan tepat satu Detect head.")
    head_name, head = heads[0]
    cfg = core.yaml
    first_conv = next(module for module in core.modules() if isinstance(module, torch.nn.Conv2d))
    if "yolo26" not in str(cfg.get("yaml_file", "")).lower() or cfg.get("reg_max") != 1 or first_conv.out_channels != {"n": 16, "s": 32}[model_variant]:
        raise ValueError("Metadata/struktur tidak cocok dengan YOLO26 N/S standar; audit manual diperlukan.")
    if cfg.get("scale") not in (None, "", model_variant):
        raise ValueError("Checkpoint bukan skala model yang dipilih.")
    branches = ("cv2", "cv3", "one2one_cv2", "one2one_cv3")
    if any(getattr(head, attr, None) is None or len(getattr(head, attr)) != head.nl for attr in branches):
        raise ValueError("Kedua cabang head wajib tersedia; checkpoint fused/head berubah ditolak.")
    names = ordered_names(core.names)
    if len(names) != head.nc or head.no != head.nc + 4 * head.reg_max:
        raise ValueError("Metadata kelas/dimensi Detect tidak konsisten.")
    modules = dict(core.named_modules())
    attention_prefixes = [name for name, m in modules.items() if isinstance(m, (C2PSA, Attention, AAttn))]
    allowed = {}
    for name, module in modules.items():
        if not isinstance(module, Bottleneck) or name.startswith(head_name + "."):
            continue
        if any(name == prefix or name.startswith(prefix + ".") for prefix in attention_prefixes):
            continue
        producer, consumer = module.cv1.conv, module.cv2.conv
        bn = getattr(module.cv1, "bn", None)
        if not isinstance(bn, torch.nn.BatchNorm2d) or producer.groups != 1 or consumer.groups != 1:
            raise ValueError(f"Bottleneck {name} tidak mendukung kebijakan hidden-channel konservatif.")
        if producer.out_channels != consumer.in_channels or producer.out_channels != bn.num_features:
            raise ValueError(f"Dimensi hidden channel {name} tidak konsisten.")
        allowed[name + ".cv1.conv"] = {"root": producer, "bn": bn, "consumer": consumer,
                                        "initial_channels": producer.out_channels, "block": name}
    if not allowed:
        raise ValueError("Tidak ditemukan hidden-channel group yang eligible.")
    backbone_count = len(cfg["backbone"])
    stages = []
    for i, module in enumerate(core.model):
        stages.append({"index": i, "type": type(module).__name__, "from": module.f,
                       "region": "detection_head" if module is head else ("backbone" if i < backbone_count else "neck"),
                       "parameters": sum(p.numel() for p in module.parameters())})
    inventory = [{"name": name, "type": type(module).__name__,
                  "residual_add": bool(getattr(module, "add", False))}
                 for name, module in modules.items()
                 if isinstance(module, (torch.nn.Conv2d, torch.nn.BatchNorm2d, Bottleneck, C2PSA, Attention, AAttn))
                 or type(module).__name__ in {"Concat", "C3k2", "SPPF"}]
    protection = []
    for name, module in modules.items():
        if isinstance(module, torch.nn.Conv2d) and name not in allowed:
            reason = "head one2many/one2one utuh" if name.startswith(head_name + ".") else "dimensi CSP/residual/attention/antartahap dilindungi"
            protection.append({"name": name, "reason": reason,
                               "protected_dimension": "all" if name.startswith(head_name + ".") else "out_channels",
                               "dependent_input_pruning_allowed": any(module is entry["consumer"] for entry in allowed.values())})
    return {"head": head, "head_name": head_name, "eligible": allowed, "modules": modules,
            "report": {"stages": stages, "module_inventory": inventory, "protected_layers": protection,
                       "eligible_layers": list(allowed), "head_feature_sources": list(head.f),
                       "class_names": names, "nc": head.nc, "reg_max": head.reg_max, "nl": head.nl,
                       "end2end": head.end2end, "unknown_policy": "fail explicitly"}}


def raw_branches(output):
    """Ambil raw kedua cabang tanpa mengganti forward atau algoritme deteksi."""
    if not isinstance(output, tuple) or len(output) != 2 or not isinstance(output[1], dict):
        raise ValueError("Output eval YOLO26 harus (decoded, raw_predictions).")
    branches = output[1]
    if set(branches) != {"one2many", "one2one"}:
        raise ValueError(f"Cabang YOLO26 tidak lengkap: {list(branches)}")
    return branches


def trace_outputs(output):
    """Root autograd mencakup boxes/scores kedua cabang, tanpa top-k/NMS."""
    branches = raw_branches(output)
    return [branches[branch][key] for branch in ("one2many", "one2one") for key in ("boxes", "scores")]


def finite_tensors(value):
    """Validasi semua tensor, termasuk feature maps dalam output nested."""
    if isinstance(value, torch.Tensor):
        if not torch.isfinite(value).all():
            raise ValueError("Output model mengandung NaN/Inf.")
    elif isinstance(value, dict):
        for item in value.values():
            finite_tensors(item)
    elif isinstance(value, (tuple, list)):
        for item in value:
            finite_tensors(item)


def forward_snapshot(core, image):
    """Forward FP32, cek kontrak raw/decoded, simpan tensor kecil untuk restore parity."""
    core.eval()
    with torch.no_grad():
        output = core(image)
    finite_tensors(output)
    branches = raw_branches(output)
    head = core.model[-1]
    snapshot = {"decoded": output[0].detach().cpu().clone()}
    anchors = None
    for branch in ("one2many", "one2one"):
        boxes, scores = branches[branch]["boxes"], branches[branch]["scores"]
        if boxes.ndim != 3 or scores.ndim != 3 or boxes.shape[:2] != (image.shape[0], 4 * head.reg_max):
            raise ValueError(f"Bentuk boxes {branch} invalid: {tuple(boxes.shape)}")
        if scores.shape[:2] != (image.shape[0], head.nc) or boxes.shape[2] != scores.shape[2]:
            raise ValueError(f"Bentuk scores {branch} tidak cocok dengan jumlah kelas.")
        if anchors is not None and anchors != boxes.shape[2]:
            raise ValueError("Jumlah anchor kedua cabang berbeda.")
        anchors = boxes.shape[2]
        snapshot[branch + "/boxes"] = boxes.detach().cpu().clone()
        snapshot[branch + "/scores"] = scores.detach().cpu().clone()
    decoded = snapshot["decoded"]
    if head.end2end:
        if decoded.ndim != 3 or decoded.shape[0] != image.shape[0] or decoded.shape[-1] != 6:
            raise ValueError("Output end-to-end bukan [batch, detections, 6].")
        ids = decoded[..., 5]
        if ((ids < 0) | (ids >= head.nc) | (ids != ids.floor())).any():
            raise ValueError("Indeks kelas pada output deteksi invalid.")
    elif decoded.ndim != 3 or decoded.shape[:2] != (image.shape[0], head.nc + 4):
        raise ValueError("Output one-to-many tidak memiliki dimensi nc+4.")
    return snapshot


def model_measurements(core, imgsz):
    """Hitung Conv/BN dan attention matmul tanpa menghilangkan child Conv attention."""
    parameters = sum(p.numel() for p in core.parameters())
    copy = deepcopy(core).eval()
    parameter = next(copy.parameters())
    sample = torch.zeros(1, 3, imgsz, imgsz, device=parameter.device, dtype=torch.float32)
    attention_macs = []
    def count_attention(module, inputs, output):
        counter = SimpleNamespace(total_ops=0, num_heads=module.num_heads, head_dim=module.head_dim,
                                  key_dim=getattr(module, "key_dim", module.head_dim), area=getattr(module, "area", 1))
        _attention_ops(counter, inputs, output)
        attention_macs.append(float(counter.total_ops))
    handles = [module.register_forward_hook(count_attention) for module in copy.modules()
               if isinstance(module, (Attention, AAttn))]
    try:
        with torch.no_grad():
            # A custom Attention handler in classic THOP would hide its child Conv/BN counts.
            macs, _ = thop_profile(copy, inputs=(sample,), verbose=False)
    finally:
        for handle in handles:
            handle.remove()
    gflops = (float(macs) + sum(attention_macs)) * 2 / 1e9
    if not math.isfinite(gflops) or gflops <= 0:
        raise RuntimeError("Profiling GFLOPs gagal/bernilai nol; cek thop dan source dependency.")
    return {"parameters": parameters, "gflops": gflops, "precision": "FP32", "fusion": False}


def head_snapshot(head):
    """Salin parameter/buffer serta kontrak dimensi seluruh head yang dilindungi."""
    return {"state": {key: value.detach().cpu().clone() for key, value in head.state_dict().items()},
            "contract": (head.nc, head.no, head.nl, head.reg_max, head.end2end)}


def assert_head_unchanged(head, before):
    """Pruning tersembunyi tidak boleh menyentuh bobot atau dimensi head."""
    after = head_snapshot(head)
    if after["contract"] != before["contract"] or after["state"].keys() != before["state"].keys():
        raise ValueError("Kontrak/struktur detection head berubah.")
    if any(not torch.equal(after["state"][key], value) for key, value in before["state"].items()):
        raise ValueError("Parameter/buffer detection head berubah.")

In [ ]:
def assert_layer_consistency(core):
    """Dimensi atribut modul harus sesuai dengan tensor parameter/buffer aktual."""
    for name, module in core.named_modules():
        if isinstance(module, torch.nn.Conv2d):
            expected = (module.out_channels, module.in_channels // module.groups, *module.kernel_size)
            if tuple(module.weight.shape) != expected or module.in_channels % module.groups or module.out_channels % module.groups:
                raise ValueError(f"Bentuk Conv invalid pada {name}.")
        elif isinstance(module, torch.nn.BatchNorm2d):
            if any(t.numel() != module.num_features for t in (module.weight, module.bias, module.running_mean, module.running_var)):
                raise ValueError(f"Bentuk BatchNorm invalid pada {name}.")

## 06 — Audit Dependency Graph dan Eligible Layer
Gunakan DepGraph untuk memeriksa dimensi yang terikat, indeks channel asli, dan group eligible.

In [ ]:
"""Standalone LAMP V2 policy source, embedded verbatim in the Kaggle notebook.

This file is an authoring source, not a notebook runtime dependency.
"""
import math
import warnings

import torch
import torch_pruning as tp
from ultralytics.nn.modules import Bottleneck, C2f, C3k2, Detect
from ultralytics.nn.modules.block import C2PSA, Attention, PSABlock
from ultralytics.nn.tasks import DetectionModel


def _v2_trace_outputs(output):
    if not isinstance(output, tuple) or len(output) != 2 or not isinstance(output[1], dict):
        raise ValueError("Output YOLO26 eval harus (decoded, raw).")
    if set(output[1]) != {"one2many", "one2one"}:
        raise ValueError("Graph wajib mencakup raw output kedua cabang.")
    return [output[1][branch][key] for branch in ("one2many", "one2one")
            for key in ("boxes", "scores")]


def _v2_policy_structure(core, model_variant=None):
    if not isinstance(core, DetectionModel):
        raise TypeError("Policy memerlukan DetectionModel YOLO26 unfused.")
    modules = dict(core.named_modules())
    heads = [(name, module) for name, module in modules.items() if isinstance(module, Detect)]
    if len(heads) != 1:
        raise ValueError("Diperlukan satu Detect head.")
    head_name, head = heads[0]
    if any(getattr(head, name, None) is None for name in ("cv2", "cv3", "one2one_cv2", "one2one_cv3")):
        raise ValueError("Detection head kedua cabang harus unfused dan lengkap.")
    first = next(module for module in modules.values() if isinstance(module, torch.nn.Conv2d))
    actual_variant = core.yaml.get("scale")
    if actual_variant in (None, ""):
        actual_variant = {16: "n", 32: "s"}.get(first.out_channels)
    if model_variant is not None and model_variant not in ("n", "s"):
        raise ValueError("Model variant harus n atau s.")
    if ("yolo26" not in str(core.yaml.get("yaml_file", "")).lower()
            or core.yaml.get("reg_max") != 1 or actual_variant not in ("n", "s")
            or first.out_channels != {"n": 16, "s": 32}.get(actual_variant)):
        raise ValueError("V2 memerlukan arsitektur standar YOLO26n/S dengan stem 16/32 yang sesuai.")
    if model_variant is not None and model_variant != actual_variant:
        raise ValueError(f"Model variant tidak cocok: diminta {model_variant}, checkpoint {actual_variant}.")
    envelopes = [(name, module) for name, module in modules.items()
                 if isinstance(module, (Detect, C2PSA, PSABlock, Attention))
                 or type(module).__name__ == "AAttn"]
    protected = {}
    for name, module in modules.items():
        for prefix, envelope in envelopes:
            if name == prefix or name.startswith(prefix + "."):
                protected[module] = "detection_head_all_dimensions" if isinstance(envelope, Detect) else "attention_envelope_all_dimensions"
                break
    split_outputs = {}
    for name, module in modules.items():
        if isinstance(module, (C2f, C3k2)):
            for child in (module.cv1.conv, getattr(module.cv1, "bn", None)):
                if child is not None:
                    split_outputs[child] = "CSP_split_output_dimension"
    conservative = {}
    for name, module in modules.items():
        if not isinstance(module, Bottleneck) or module in protected:
            continue
        producer, consumer = module.cv1.conv, module.cv2.conv
        bn = getattr(module.cv1, "bn", None)
        if (not isinstance(bn, torch.nn.BatchNorm2d) or producer.groups != 1 or consumer.groups != 1
                or producer.out_channels != consumer.in_channels or producer.out_channels != bn.num_features):
            raise ValueError(f"Bottleneck hidden channel tidak didukung: {name}")
        conservative[producer] = {"name": name + ".cv1.conv", "bn": bn, "consumer": consumer}
    if not conservative:
        raise ValueError("Tidak ada kandidat konservatif.")
    return {"model_variant": actual_variant, "modules": modules, "names": {module: name for name, module in modules.items()},
            "head": head, "head_name": head_name, "protected": protected,
            "split_outputs": split_outputs, "conservative": conservative}


def protected_policy_snapshot(core, model_variant=None):
    """Snapshot seluruh bobot, buffer, dan kontrak head/attention yang dilindungi."""
    structure = _v2_policy_structure(core, model_variant)
    result = {}
    for name, module in structure["modules"].items():
        if isinstance(module, (Detect, C2PSA, PSABlock, Attention)) or type(module).__name__ == "AAttn":
            result[name] = {"state": {key: value.detach().cpu().clone() for key, value in module.state_dict().items()},
                            "dimensions": {key: getattr(module, key) for key in ("nc", "no", "nl", "reg_max", "end2end", "c")
                                           if hasattr(module, key)}}
    return result


def assert_policy_protected_unchanged(core, snapshot):
    modules = dict(core.named_modules())
    for name, entry in snapshot.items():
        if name not in modules:
            raise ValueError(f"Envelope terlindungi hilang: {name}")
        module = modules[name]
        after = module.state_dict()
        if after.keys() != entry["state"].keys() or any(
                not torch.equal(value.detach().cpu(), entry["state"][key]) for key, value in after.items()):
            raise ValueError(f"Bobot/buffer envelope terlindungi berubah: {name}")
        if any(getattr(module, key) != value for key, value in entry["dimensions"].items()):
            raise ValueError(f"Kontrak envelope terlindungi berubah: {name}")


def _v2_output_signature(group, graph, names):
    """Identitas satu group adalah seluruh dimensi Conv-output yang terikat."""
    return tuple(sorted(names[dep.target.module] for dep, _ in group
                        if isinstance(dep.target.module, torch.nn.Conv2d)
                        and graph.is_out_channel_pruning_fn(dep.handler)))


def _v2_validate_group(group, graph, structure, policy):
    root = group[0].dep.target.module
    root_name = structure["names"].get(root, "<unknown>")
    entry = structure["conservative"].get(root)
    if policy == "conservative" and entry is None:
        raise ValueError("root_not_conservative_hidden_channel")
    for dep, indices in group:
        module, handler = dep.target.module, dep.handler
        name = structure["names"].get(module, type(module).__name__)
        output = graph.is_out_channel_pruning_fn(handler)
        if module in structure["protected"]:
            raise ValueError(f"{name}: {structure['protected'][module]}")
        if output and module in structure["split_outputs"]:
            raise ValueError(f"{name}: {structure['split_outputs'][module]}")
        if isinstance(module, torch.nn.Conv2d):
            if not output and not graph.is_in_channel_pruning_fn(handler):
                raise ValueError(f"{name}: unknown_conv_pruning_handler")
            if module.groups != 1:
                raise ValueError(f"{name}: grouped_or_depthwise_conv_unsupported")
            if policy == "conservative" and not (
                    (module is root and output) or (module is entry["consumer"] and graph.is_in_channel_pruning_fn(handler))):
                raise ValueError(f"{name}: outside_conservative_hidden_channel")
        elif isinstance(module, torch.nn.BatchNorm2d):
            if policy == "conservative" and module is not entry["bn"]:
                raise ValueError(f"{name}: outside_conservative_batchnorm")
        else:
            kind = type(module).__name__
            if kind not in {"_ElementWiseOp", "_ConcatOp"}:
                raise ValueError(f"{name}: unsupported_operation_{kind}")
            if kind == "_ElementWiseOp":
                operation = str(module)
                supported = ("SiluBackward", "AddBackward", "MaxPool2DWithIndicesBackward", "UpsampleNearest2DBackward", "CloneBackward")
                if not any(token.lower() in operation.lower() for token in supported):
                    raise ValueError(f"{name}: unsupported_elementwise_{operation}")
        # Torch-Pruning maps concat offsets and coupled residual identities. Reject invalid maps.
        mapped = [int(index) for index in indices]
        if len(mapped) != len(set(mapped)) or any(index < 0 for index in mapped):
            raise ValueError(f"{name}: invalid_or_duplicate_index_mapping")
        channels = graph.get_out_channels(module) if output else graph.get_in_channels(module)
        if channels is not None and any(index >= channels for index in mapped):
            raise ValueError(f"{name}: dependency_index_out_of_bounds")
    return root_name


def _v2_channel_ledger(structure):
    """Track each physical model dimension independently in original index space."""
    ledger = {}
    for name, module in structure["modules"].items():
        if isinstance(module, torch.nn.Conv2d):
            dimensions = {"in": module.in_channels, "out": module.out_channels}
        elif isinstance(module, torch.nn.BatchNorm2d):
            dimensions = {"channel": module.num_features}
        else:
            continue
        for dimension, count in dimensions.items():
            ledger[(module, dimension)] = {
                "layer": name, "module_type": type(module).__name__, "dimension": dimension,
                "channels_before": count, "kept": list(range(count)), "group_ids": set()}
    return ledger


def _v2_dependency_dimension(dependency, graph):
    module = dependency.target.module
    if isinstance(module, torch.nn.BatchNorm2d):
        return module, "channel"
    if isinstance(module, torch.nn.Conv2d):
        return module, "out" if graph.is_out_channel_pruning_fn(dependency.handler) else "in"
    return None


def _v2_refresh_channel_report(audit):
    mappings, masks = [], []
    for entry in audit["channel_ledger"].values():
        kept = set(entry["kept"])
        removed = [index for index in range(entry["channels_before"]) if index not in kept]
        row = {key: entry[key] for key in ("layer", "module_type", "dimension", "channels_before")}
        row.update(channels_after=len(entry["kept"]), kept_original_indices=list(entry["kept"]),
                   removed_original_indices=removed, dependency_group_ids=sorted(entry["group_ids"]))
        mappings.append(row)
        masks.append({"layer": row["layer"], "module_type": row["module_type"],
                      "dimension": row["dimension"], "channels_before": row["channels_before"],
                      "channels_after": row["channels_after"],
                      "dependency_group_ids": row["dependency_group_ids"],
                      "kept_mask": [index in kept for index in range(entry["channels_before"])]})
    audit["report"]["channel_mapping"] = mappings
    audit["report"]["channel_masks"] = masks


def _v2_capacity(entry):
    current = entry["root"].out_channels
    capacity = current - 1
    for module, initial in entry["output_dimensions"].items():
        capacity = min(capacity, math.floor(initial * 0.5) - (initial - module.out_channels))
    return max(0, capacity)


def build_policy_pruner(core, image, policy, ratio, model_variant="n"):
    """Discover, audit and canonicalize dependencies before global LAMP ranking."""
    if policy not in ("conservative", "extended"):
        raise ValueError("Policy harus conservative atau extended.")
    if not 0 < float(ratio) <= 0.5:
        raise ValueError("Rasio harus di dalam (0, 0.50].")
    core.eval().requires_grad_(True)
    structure = _v2_policy_structure(core, model_variant)
    importance = tp.importance.LAMPImportance(p=2, group_reduction="mean", bias=False)
    initial_ignored = [module for module in structure["modules"].values()
                       if isinstance(module, torch.nn.Conv2d) and
                       (module in structure["protected"] or module in structure["split_outputs"]
                        or (policy == "conservative" and module not in structure["conservative"]))]
    with torch.enable_grad(), warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter("always")
        pruner = tp.pruner.BasePruner(core, image, importance=importance,
            global_pruning=True, pruning_ratio=float(ratio), iterative_steps=1,
            max_pruning_ratio=0.50, round_to=None, root_module_types=[torch.nn.Conv2d],
            ignored_layers=initial_ignored,
            forward_fn=lambda model, example: model(example), output_transform=_v2_trace_outputs)
    graph = pruner.DG
    eligible, rejected, seen, seen_outputs = {}, [], set(), set()
    # Inspect every Conv root explicitly, including rejected roots. The later canonical
    # scan follows BasePruner's own graph order to preserve V1 ranking semantics.
    for name, module in structure["modules"].items():
        if not isinstance(module, torch.nn.Conv2d):
            continue
        if module in initial_ignored:
            reason = structure["protected"].get(module) or structure["split_outputs"].get(module) or "root_not_conservative_hidden_channel"
            rejected.append({"root": name, "reason": reason})
            continue
        group = graph.get_pruning_group(module, tp.prune_conv_out_channels, list(range(module.out_channels)))
        try:
            _v2_validate_group(group, graph, structure, policy)
        except ValueError as error:
            rejected.append({"root": name, "reason": str(error)})
    rejected_roots = {entry["root"] for entry in rejected}
    ignored = [module for name, module in structure["modules"].items()
               if isinstance(module, torch.nn.Conv2d) and name in rejected_roots]
    for group in graph.get_all_groups(ignored_layers=ignored, root_module_types=[torch.nn.Conv2d]):
        root_name = _v2_validate_group(group, graph, structure, policy)
        signature = _v2_output_signature(group, graph, structure["names"])
        if not signature or signature in seen:
            continue
        overlap = seen_outputs.intersection(signature)
        if overlap:
            raise ValueError(f"Canonical dependency groups overlap output dimensions: {sorted(overlap)}")
        seen.add(signature)
        seen_outputs.update(signature)
        root = group[0].dep.target.module
        score = importance(group)
        if score is None or score.numel() != root.out_channels or not torch.isfinite(score).all():
            raise ValueError(f"Importance LAMP invalid: {root_name}")
        dimensions = {dep.target.module: int(dep.target.module.out_channels)
                      for dep, _ in group if isinstance(dep.target.module, torch.nn.Conv2d)
                      and graph.is_out_channel_pruning_fn(dep.handler)}
        membership = {dep.target.module: ("out" if graph.is_out_channel_pruning_fn(dep.handler) else "in")
                      for dep, _ in group
                      if isinstance(dep.target.module, (torch.nn.Conv2d, torch.nn.BatchNorm2d))}
        capacity = min(root.out_channels - 1, *(math.floor(initial * 0.5) for initial in dimensions.values()))
        permitted = set(torch.argsort(score, stable=True)[:capacity].detach().cpu().tolist())
        eligible[root] = {"name": root_name, "root": root, "signature": signature,
                          "initial_channels": root.out_channels, "output_dimensions": dimensions,
                          "member_dimensions": membership,
                          "initial_lamp_scores": score.detach().cpu().tolist(),
                          "initial_capacity_mask": [index in permitted for index in range(root.out_channels)],
                          "group_id": f"{policy}-group-{len(eligible):03d}",
                          "dependencies": [str(dep) for dep, _ in group]}
    if not eligible:
        raise ValueError("Audit tidak menemukan group aman untuk policy ini.")
    pruner.ignored_layers = ignored
    audit = {"structure": structure, "eligible": eligible, "policy": policy, "ratio": float(ratio),
             "model_variant": structure["model_variant"], "channel_ledger": _v2_channel_ledger(structure),
             "independent_channels_before": sum(entry["initial_channels"] for entry in eligible.values()),
             "physical_conv_output_channels_before": sum(sum(entry["output_dimensions"].values()) for entry in eligible.values())}
    audit["report"] = {"model_variant": structure["model_variant"], "policy": policy,
         "requested_ratio": float(ratio), "max_pruning_ratio": 0.5,
         "importance": "LAMPImportance(p=2, group_reduction='mean', bias=False)",
         "ranking": "BasePruner global one-step; capacity exclusions are +inf sentinels, finite eligible LAMP scores unchanged",
         "group_count": len(eligible), "independent_channels_before": audit["independent_channels_before"],
         "physical_conv_output_channels_before": audit["physical_conv_output_channels_before"],
         "eligible_groups": [{key: value for key, value in entry.items() if key not in ("root", "output_dimensions", "member_dimensions")}
                             for entry in eligible.values()],
         "rejected_groups": rejected, "warnings": [str(item.message) for item in captured],
         "lamp_scores": [{"group_id": entry["group_id"], "root": entry["name"],
                          "channel_idx": index, "score": score,
                          "capacity_eligible": entry["initial_capacity_mask"][index], "removed": False}
                         for entry in eligible.values() for index, score in enumerate(entry["initial_lamp_scores"])],
         "channel_mapping_status": "RECORDED_FROM_LIVE_DEPENDENCY_GRAPH",
         "channel_mapping_index_space": "Original model dimension indices; current indices translated before every successful prune"}
    for entry in eligible.values():
        group = graph.get_pruning_group(entry["root"], tp.prune_conv_out_channels,
                                        list(range(entry["initial_channels"])))
        for dependency, _ in group:
            dimension = _v2_dependency_dimension(dependency, graph)
            if dimension is not None:
                audit["channel_ledger"][dimension]["group_ids"].add(entry["group_id"])
    _v2_refresh_channel_report(audit)

    def capacity_limited_lamp(group):
        _v2_validate_group(group, graph, structure, policy)
        entry = eligible.get(group[0].dep.target.module)
        if entry is None:
            raise ValueError("Proposal berakar di luar group yang diaudit.")
        score = importance(group)
        current = entry["root"].out_channels
        if score is None or score.numel() != current or not torch.isfinite(score).all():
            raise ValueError("Importance LAMP tidak finite sebelum cap.")
        # Bound every coupled output dimension, preserving the original LAMP score.
        capacity = _v2_capacity(entry)
        permitted = torch.argsort(score, stable=True)[:capacity]
        capped = torch.full_like(score, float("inf"))
        capped[permitted] = score[permitted]
        return capped
    pruner.importance = capacity_limited_lamp
    return pruner, audit


def apply_policy_pruning(pruner, audit):
    """Audit each concrete proposal before irreversible tensor shape mutation."""
    history = []
    graph, structure = pruner.DG, audit["structure"]
    # The runtime report retains this live list even if a later proposal raises.
    audit["report"]["pruning_history"] = history
    audit["report"]["pruning_status"] = "IN_PROGRESS"
    for group in pruner.step(interactive=True):
        audit["report"]["last_proposal"] = {
            "root": structure["names"].get(group[0].dep.target.module, "<unknown>"),
            "removed_indices": [int(index) for index in group[0].idxs]}
        name = _v2_validate_group(group, graph, structure, audit["policy"])
        entry = audit["eligible"].get(group[0].dep.target.module)
        if entry is None:
            raise ValueError("Group proposal tidak terdapat pada audit.")
        indices = [int(index) for index in group[0].idxs]
        if not indices:
            continue
        if _v2_output_signature(group, graph, structure["names"]) != entry["signature"]:
            raise ValueError("Dimensi output proposal berbeda dari group canonical yang diaudit.")
        if not graph.check_pruning_group(group):
            raise ValueError(f"Dependency group tidak valid: {name}")
        mapped_dimensions = {}
        for dependency, mapped in group:
            dimension = _v2_dependency_dimension(dependency, graph)
            if dimension is None:
                continue
            # A residual/concat may reach a physical dimension along multiple paths.
            # The ledger records its union once, in the current dimension's space.
            mapped_dimensions.setdefault(dimension, set()).update(int(index) for index in mapped)
        dimension_records = []
        for dimension, mapped in mapped_dimensions.items():
            ledger = audit["channel_ledger"][dimension]
            current_indices = sorted(mapped)
            if any(index < 0 or index >= len(ledger["kept"]) for index in current_indices):
                raise ValueError(f"Ledger dependency index out of bounds: {ledger['layer']}:{dimension[1]}")
            dimension_records.append({"layer": ledger["layer"], "module_type": ledger["module_type"],
                "dimension": dimension[1], "current_indices": current_indices,
                "original_indices": [ledger["kept"][index] for index in current_indices],
                "channels_before": len(ledger["kept"]), "channels_after": len(ledger["kept"]) - len(mapped)})
        physical_removed = 0
        touched = []
        for (module, dimension), mapped in mapped_dimensions.items():
            if isinstance(module, torch.nn.Conv2d) and dimension == "out":
                if module not in entry["output_dimensions"]:
                    raise ValueError("Proposal mengubah output di luar group canonical.")
                initial = entry["output_dimensions"][module]
                remaining = module.out_channels - len(mapped)
                if remaining < 1 or initial - remaining > math.floor(initial * 0.5):
                    raise ValueError(f"Cap 50%/minimum satu channel dilanggar: {structure['names'][module]}")
                physical_removed += len(mapped)
                touched.append({"layer": structure["names"][module], "before": module.out_channels, "after": remaining})
        before = entry["root"].out_channels
        root_ledger = audit["channel_ledger"][(entry["root"], "out")]
        original_root_indices = [root_ledger["kept"][index] for index in indices]
        record = {"root": name, "group_id": entry["group_id"], "removed_indices": indices,
                  "removed_original_indices": original_root_indices, "mapped_dimensions": dimension_records,
                  "channels_before": before, "channels_after": before - len(indices),
                  "physical_conv_output_channels_removed": physical_removed,
                  "coupled_conv_output_count": len(touched),
                  "output_changes": touched, "dependencies": [str(dep) for dep, _ in group]}
        group.prune()
        for (module, dimension), mapped in mapped_dimensions.items():
            ledger = audit["channel_ledger"][(module, dimension)]
            ledger["kept"] = [original for index, original in enumerate(ledger["kept"]) if index not in mapped]
            ledger["group_ids"].add(entry["group_id"])
            actual = (module.in_channels if dimension == "in" else module.out_channels) if isinstance(
                module, torch.nn.Conv2d) else module.num_features
            if len(ledger["kept"]) != actual:
                raise ValueError(f"Ledger/model channel mismatch: {ledger['layer']}:{dimension}")
        removed_set = set(original_root_indices)
        for row in audit["report"]["lamp_scores"]:
            if row["group_id"] == entry["group_id"] and row["channel_idx"] in removed_set:
                row["removed"] = True
        _v2_refresh_channel_report(audit)
        history.append(record)
        audit["report"]["independent_channels_after"] = sum(entry["root"].out_channels for entry in audit["eligible"].values())
        audit["report"]["independent_channels_removed"] = sum(item["channels_before"] - item["channels_after"] for item in history)
        audit["report"]["physical_conv_output_channels_removed"] = sum(item["physical_conv_output_channels_removed"] for item in history)
    if not history:
        raise RuntimeError("Tidak ada channel struktural dihapus.")
    audit["report"]["pruning_status"] = "PASS"
    return history


def measure_layer_inventory(core, image, audit, policy):
    """Per-layer Conv/BN estimates at the actual input size, on an unfused model."""
    shapes, handles = {}, []
    for name, module in core.named_modules():
        if isinstance(module, (torch.nn.Conv2d, torch.nn.BatchNorm2d)):
            def hook(layer, args, output, name=name):
                shapes[name] = list(output.shape)
            handles.append(module.register_forward_hook(hook))
    try:
        with torch.no_grad():
            core(image)
    finally:
        for handle in handles:
            handle.remove()
    structure = audit["structure"]
    membership, member_dimensions = {}, {}
    for entry in audit["eligible"].values():
        for module, dimension in entry["member_dimensions"].items():
            membership.setdefault(module, []).append(entry["group_id"])
            member_dimensions.setdefault(module, set()).add(dimension)
    rejected = {item["root"]: item["reason"] for item in audit["report"]["rejected_groups"]}
    backbone_count = len(core.yaml.get("backbone", []))
    rows = []
    for name, module in core.named_modules():
        if not isinstance(module, (torch.nn.Conv2d, torch.nn.BatchNorm2d)):
            continue
        shape = shapes.get(name, [])
        if isinstance(module, torch.nn.Conv2d):
            inputs, outputs = module.in_channels, module.out_channels
            operations = (2 * math.prod(shape) * (inputs // module.groups) * math.prod(module.kernel_size)) if shape else None
        else:
            inputs = outputs = module.num_features
            operations = 2 * math.prod(shape) if shape else None
        stage = next((i for i, block in enumerate(core.model)
                      if any(child is module for child in block.modules())), None)
        reason = structure["protected"].get(module) or structure["split_outputs"].get(module) or rejected.get(name)
        rows.append({"layer_name": name, "module_type": type(module).__name__,
                     "region": "detection_head" if module in structure["protected"] and str(reason).startswith("detection")
                               else ("backbone" if stage is not None and stage < backbone_count else "neck"),
                     "in_channels": inputs, "out_channels": outputs,
                     "parameters": sum(parameter.numel() for parameter in module.parameters(recurse=False)),
                     "estimated_flops": operations, "flops_estimate_method": "Conv:2*elements*Cin/groups*Kh*Kw; BN:2*elements; functional attention excluded",
                     "policy": policy, "eligible": module in membership,
                     "protection_reason": reason or ("not_in_eligible_dependency_group" if module not in membership else ""),
                     "dependency_group": ";".join(membership.get(module, [])),
                     "eligible_dimensions": ";".join(sorted(member_dimensions.get(module, set()))), "output_shape": shape})
    return rows

In [ ]:
"""Standalone analysis helpers embedded in notebook 03 by its generator.

All figures describe measured structural changes, not activation feature maps.
Missing historical mappings, scores, and validation values are never fabricated.
"""
import csv as _eda_csv
import json as _eda_json
import math as _eda_math
from pathlib import Path as _EdaPath

import matplotlib.pyplot as _eda_plt
import numpy as _eda_np
from matplotlib.colors import ListedColormap as _EdaColors


def _eda_write_json(path, value):
    path = _EdaPath(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(_eda_json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")


def _eda_write_csv(path, rows):
    path = _EdaPath(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    keys = list(dict.fromkeys(key for row in rows for key in row))
    with path.open("w", newline="", encoding="utf-8") as stream:
        writer = _eda_csv.DictWriter(stream, fieldnames=keys)
        writer.writeheader()
        writer.writerows({key: _eda_json.dumps(value, ensure_ascii=False) if isinstance(value, (dict, list)) else value
                         for key, value in row.items()} for row in rows)


def _eda_finite(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool) and _eda_math.isfinite(value)


def _eda_save_figure(fig, path):
    path = _EdaPath(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path.with_suffix(".png"), dpi=int(globals().get("EDA_DPI", 300)), bbox_inches="tight")
    fig.savefig(path.with_suffix(".svg"), bbox_inches="tight")
    _eda_plt.close(fig)
    return {"png": path.with_suffix(".png").name, "svg": path.with_suffix(".svg").name}


def layer_eda_rows(inventory_before, inventory_after):
    """Join actual unfused inventory by module name, preserving baseline order."""
    previous = {row["layer_name"]: row for row in inventory_before}
    current = {row["layer_name"]: row for row in inventory_after}
    if len(previous) != len(inventory_before) or len(current) != len(inventory_after):
        raise ValueError("EDA inventory has duplicate module names.")
    if set(previous) != set(current):
        raise ValueError("EDA before/after module names differ; layer alignment is unsafe.")
    rows = []
    for name, before in previous.items():
        after = current[name]
        if before["module_type"] != after["module_type"]:
            raise ValueError(f"EDA module type changed at {name}.")
        if after["out_channels"] > before["out_channels"] or after["in_channels"] > before["in_channels"]:
            raise ValueError(f"Pruning unexpectedly increased channels at {name}.")
        parts = name.split(".")
        stage = before.get("stage", ".".join(parts[:2]))
        row = {"layer_name": name, "module_type": before["module_type"], "stage": stage,
               "region": before.get("region", "unknown"), "policy": before.get("policy"),
               "eligible": before.get("eligible", False), "protection_reason": before.get("protection_reason", ""),
               "dependency_group": before.get("dependency_group", "")}
        for key in ("in_channels", "out_channels", "parameters", "estimated_flops"):
            row[key + "_before"] = before.get(key)
            row[key + "_after"] = after.get(key)
            base, value = before.get(key), after.get(key)
            row[key + "_removed"] = base - value if _eda_finite(base) and _eda_finite(value) else None
            row[key + "_reduction_pct"] = 100 * (base - value) / base if _eda_finite(base) and _eda_finite(value) and base else None
        if (row["protection_reason"] and str(row["protection_reason"]).startswith(("detection", "attention"))
                and (row["out_channels_removed"] or row["in_channels_removed"])):
            raise ValueError(f"Protected head/attention channel dimension changed at {name}.")
        rows.append(row)
    return rows


def stage_eda_rows(rows):
    """Physical Conv-output sums; these are distinct from independent group channels."""
    stages = {}
    for row in rows:
        key = (str(row["stage"]), row["region"])
        stage = stages.setdefault(key, {"stage": row["stage"], "region": row["region"], "conv_layers": 0,
                                      "physical_conv_outputs_before": 0, "physical_conv_outputs_after": 0,
                                      "parameters_before": 0, "parameters_after": 0,
                                      "estimated_flops_before": 0, "estimated_flops_after": 0})
        if row["module_type"] == "Conv2d":
            stage["conv_layers"] += 1
            for suffix in ("before", "after"):
                stage["physical_conv_outputs_" + suffix] += row["out_channels_" + suffix]
        for metric in ("parameters", "estimated_flops"):
            for suffix in ("before", "after"):
                value = row[metric + "_" + suffix]
                if _eda_finite(value):
                    stage[metric + "_" + suffix] += value
    for row in stages.values():
        row["physical_conv_outputs_removed"] = row["physical_conv_outputs_before"] - row["physical_conv_outputs_after"]
        for key in ("physical_conv_outputs", "parameters", "estimated_flops"):
            base, after = row[key + "_before"], row[key + "_after"]
            row[key + "_reduction_pct"] = 100 * (base - after) / base if base else None
    return list(stages.values())


def make_channel_figure(rows, title="Channel comparison"):
    """Three panels share exact Conv order and Y limits, as in the reference image."""
    if not rows or any(row["module_type"] != "Conv2d" for row in rows):
        raise ValueError("Channel diagram requires a nonempty ordered Conv2d inventory.")
    count = len(rows)
    width = max(16, min(32, count * 0.16))
    fig, axes = _eda_plt.subplots(3, 1, figsize=(width, 12), sharex=True, sharey=True)
    indices = _eda_np.arange(count)
    before = [row["out_channels_before"] for row in rows]
    after = [row["out_channels_after"] for row in rows]
    upper = max(max(before), max(after), 1) * 1.12
    axes[0].bar(indices, before, color="blue", width=0.82, label="Before Pruning")
    axes[1].bar(indices, after, color="red", width=0.82, label="After Pruning")
    axes[2].bar(indices, before, color="blue", width=0.82, label="Before Pruning")
    axes[2].bar(indices, after, color="red", width=0.82, label="After Pruning")
    for ax, heading in zip(axes, ("Before Pruning", "After Pruning", "Channel Comparison")):
        ax.set_title(heading)
        ax.set_ylabel("Output channels")
        ax.set_ylim(0, upper)
        ax.set_xlim(-0.8, count - 0.2)
        ax.grid(axis="y", alpha=0.2)
        ax.set_axisbelow(True)
        ax.legend(loc="upper right")
        ax.tick_params(axis="x", labelbottom=True)
        ax.set_xticks(indices, [row["layer_name"] for row in rows], rotation=90, fontsize=6 if count > 32 else 8)
        ax.set_xlabel("Conv module (baseline order)")
    fig.suptitle(title, y=1.005, fontsize=14)
    fig.tight_layout()
    return fig


def _eda_channel_plots(conv_rows, folder, title):
    _eda_write_csv(folder / "channel_plot_data.csv", conv_rows)
    files = [_eda_save_figure(make_channel_figure(conv_rows, title), folder / "channels_overview")]
    page_size = min(32, max(1, int(globals().get("EDA_MAX_LAYERS_PER_PAGE", 32))))
    for offset in range(0, len(conv_rows), page_size):
        page = offset // page_size + 1
        files.append(_eda_save_figure(make_channel_figure(conv_rows[offset:offset + page_size],
                     f"{title} — layers {offset + 1}–{min(offset + page_size, len(conv_rows))}"),
                     folder / f"channels_detail_{page:02d}"))
    fig, ax = _eda_plt.subplots(figsize=(max(14, min(32, len(conv_rows) * 0.16)), 5))
    ax.bar(range(len(conv_rows)), [row["out_channels_reduction_pct"] for row in conv_rows], color="red")
    ax.set_xticks(range(len(conv_rows)), [row["layer_name"] for row in conv_rows], rotation=90, fontsize=6)
    ax.set_ylabel("Output channels removed (%)")
    ax.set_title(title + " — physical Conv-output reduction")
    ax.set_ylim(0, 100)
    fig.tight_layout()
    files.append(_eda_save_figure(fig, folder / "channel_reduction_percent"))
    return {"status": "MEASURED", "conv_layers": len(conv_rows), "detail_page_size": page_size, "files": files}


def _eda_mapping_plots(audit, conv_rows, folder):
    mapping = audit.get("channel_mapping")
    if not mapping:
        return {"status": "UNAVAILABLE", "reason": "Original channel indices were not recorded; no reconstruction attempted."}
    _eda_write_json(folder / "channel_mapping.json", mapping)
    _eda_write_csv(folder / "channel_mapping.csv", mapping)
    masks = audit.get("channel_masks")
    if masks:
        _eda_write_json(folder / "channel_masks.json", masks)
    by_layer = {row["layer"]: row for row in mapping if row.get("module_type") == "Conv2d"
                and row.get("dimension") in {"out", "out_channels", "output", "output_channels"}}
    selected = []
    for row in conv_rows:
        item = by_layer.get(row["layer_name"])
        if item is None:
            return {"status": "UNAVAILABLE", "reason": "Mapping does not cover every Conv output.", "layer": row["layer_name"]}
        before = int(item["channels_before"])
        kept, removed = item.get("kept_original_indices"), item.get("removed_original_indices")
        if not isinstance(kept, list) or not isinstance(removed, list):
            raise TypeError(f"Missing original-index ledger for {row['layer_name']}.")
        if (len(set(kept)) != len(kept) or len(set(removed)) != len(removed)
                or set(kept) & set(removed) or set(kept) | set(removed) != set(range(before))
                or len(kept) != item["channels_after"]
                or before != row["out_channels_before"] or len(kept) != row["out_channels_after"]):
            raise ValueError(f"Original-index ledger disagrees with measured channels at {row['layer_name']}.")
        selected.append((row["layer_name"], before, set(removed)))
    page_size = min(32, max(1, int(globals().get("EDA_MAX_LAYERS_PER_PAGE", 32))))
    files = []
    for offset in range(0, len(selected), page_size):
        page = selected[offset:offset + page_size]
        width = max(item[1] for item in page)
        data = _eda_np.full((len(page), width), _eda_np.nan)
        for index, (_name, count, removed) in enumerate(page):
            data[index, :count] = 0
            if removed:
                data[index, sorted(removed)] = 1
        fig, ax = _eda_plt.subplots(figsize=(16, max(4, len(page) * 0.25)))
        cmap = _EdaColors(["#4477cc", "#ee3333"]).with_extremes(bad="#eeeeee")
        plot = ax.imshow(data, interpolation="nearest", aspect="auto", cmap=cmap, vmin=0, vmax=1)
        ax.set_yticks(range(len(page)), [item[0] for item in page], fontsize=7)
        ax.set_xlabel("Original Conv-output channel index (before any pruning)")
        ax.set_title("Original-index mask: blue kept, red removed, grey outside layer width")
        fig.colorbar(plot, ax=ax, ticks=[0, 1]).ax.set_yticklabels(["Kept", "Removed"])
        fig.tight_layout()
        files.append(_eda_save_figure(fig, folder / f"channel_mask_{offset // page_size + 1:02d}"))
    return {"status": "MEASURED", "conv_layers": len(selected), "files": files,
            "meaning": "Physical Conv-output masks, including coupled dimensions; never summed as independent channels."}


def _eda_lamp_plots(audit, folder):
    scores = audit.get("lamp_scores")
    if not scores:
        return {"status": "UNAVAILABLE", "reason": "Finite pre-cap LAMP scores were not recorded."}
    if any(not _eda_finite(row.get("score")) for row in scores):
        raise ValueError("LAMP EDA requires finite original scores; capacity sentinels must not be plotted.")
    _eda_write_csv(folder / "lamp_scores.csv", scores)
    _eda_write_json(folder / "lamp_scores.json", scores)
    fig, ax = _eda_plt.subplots(figsize=(10, 5))
    for removed, color, label in ((False, "blue", "Kept"), (True, "red", "Removed")):
        values = [row["score"] for row in scores if bool(row.get("removed")) == removed]
        if values:
            ax.hist(values, bins=min(50, max(5, int(len(values) ** 0.5))), alpha=0.55, color=color, label=label)
    ax.set_xlabel("Finite LAMP score before capacity mask")
    ax.set_ylabel("Independent root channels")
    ax.set_title("LAMP selection — descriptive score distribution")
    ax.legend()
    fig.tight_layout()
    files = [_eda_save_figure(fig, folder / "lamp_score_distribution")]
    capacities = {}
    for row in scores:
        group = capacities.setdefault(row["group_id"], {"channels": 0, "capacity_eligible": 0, "removed": 0})
        group["channels"] += 1
        group["capacity_eligible"] += bool(row.get("capacity_eligible"))
        group["removed"] += bool(row.get("removed"))
    capacity_rows = [{"group_id": key, **value} for key, value in capacities.items()]
    _eda_write_csv(folder / "lamp_group_capacity.csv", capacity_rows)
    if any(row["removed"] > row["capacity_eligible"] for row in capacity_rows):
        raise ValueError("LAMP selection removed channels outside the audited capacity mask.")
    fig, ax = _eda_plt.subplots(figsize=(max(12, min(24, len(capacity_rows) * .45)), 6))
    positions = _eda_np.arange(len(capacity_rows))
    blocked = [row["channels"] - row["capacity_eligible"] for row in capacity_rows]
    retained = [row["capacity_eligible"] - row["removed"] for row in capacity_rows]
    removed = [row["removed"] for row in capacity_rows]
    ax.bar(positions, blocked, color="grey", label="Protected by capacity cap")
    ax.bar(positions, retained, bottom=blocked, color="blue", label="Capacity eligible, kept")
    ax.bar(positions, removed, bottom=_eda_np.array(blocked) + retained, color="red", label="Removed")
    ax.set_xticks(positions, [row["group_id"] for row in capacity_rows], rotation=90, fontsize=7)
    ax.set_ylabel("Independent root channels")
    ax.set_title("Group capacity and actual LAMP selection")
    ax.legend()
    fig.tight_layout()
    files.append(_eda_save_figure(fig, folder / "lamp_group_capacity"))
    return {"status": "MEASURED", "finite_scores": len(scores),
            "capacity_eligible": sum(bool(row.get("capacity_eligible")) for row in scores),
            "removed": sum(bool(row.get("removed")) for row in scores), "files": files}


def _eda_efficiency_plots(stages, result, folder):
    files = []
    for metric, label in (("physical_conv_outputs", "Physical Conv-output channels"),
                          ("parameters", "Parameters (Conv + BN)"),
                          ("estimated_flops", "Estimated layer FLOPs (Conv + BN)")):
        fig, ax = _eda_plt.subplots(figsize=(14, 5))
        positions = _eda_np.arange(len(stages))
        ax.bar(positions - .2, [row[metric + "_before"] for row in stages], width=.4, color="blue", label="Before")
        ax.bar(positions + .2, [row[metric + "_after"] for row in stages], width=.4, color="red", label="After")
        ax.set_xticks(positions, [str(row["stage"]) for row in stages], rotation=90)
        ax.set_ylabel(label)
        ax.set_title("Measured stage inventory — " + label)
        ax.legend()
        fig.tight_layout()
        files.append(_eda_save_figure(fig, folder / ("stage_" + metric)))
    baseline, pruned = result.get("baseline", {}), result.get("pruned", {})
    summary = {"independent_eligible_channels_before": result.get("eligible_channels_before"),
               "independent_eligible_channels_after": result.get("eligible_channels_after"),
               "actual_independent_channel_ratio": result.get("actual_eligible_channel_ratio"),
               "physical_conv_outputs_before": sum(row["physical_conv_outputs_before"] for row in stages),
               "physical_conv_outputs_after": sum(row["physical_conv_outputs_after"] for row in stages),
               "baseline": {key: baseline.get(key) for key in ("parameters", "gflops", "checkpoint_bytes", "latency")},
               "pruned": {key: pruned.get(key) for key in ("parameters", "gflops", "checkpoint_bytes", "latency")},
               "comparison": result.get("comparison", {}),
               "latency_scope": "raw model forward, not end-to-end",
               "layer_flops_scope": "Conv/BN estimate; model GFLOPs also counts functional attention"}
    _eda_write_json(folder / "efficiency_summary.json", summary)
    return {"status": "MEASURED", "files": files, **summary}


def _eda_evaluation_plots(result, folder):
    statuses, files = {}, []
    for identity in ("baseline", "pruned"):
        validation = (result.get(identity) or {}).get("validation")
        if not validation:
            statuses[identity] = {"status": "NOT_RUN", "reason": "Dataset validation was not run."}
            continue
        target = folder / identity
        target.mkdir(parents=True, exist_ok=True)
        per_class = validation.get("per_class") or []
        if per_class:
            _eda_write_csv(target / "per_class_metrics.csv", per_class)
            labels = [row.get("name", str(row.get("class_id"))) for row in per_class]
            fig, axes = _eda_plt.subplots(1, 3, figsize=(21, 5))
            for key, label in (("ap50", "AP50"), ("ap50_95", "AP50–95")):
                indices = [index for index, row in enumerate(per_class) if _eda_finite(row.get(key))]
                if indices:
                    axes[0].plot(indices, [per_class[index][key] for index in indices], "o-", label=label)
            for key, label in (("precision", "Precision"), ("recall", "Recall"), ("f1", "F1")):
                indices = [index for index, row in enumerate(per_class) if _eda_finite(row.get(key))]
                if indices:
                    axes[1].plot(indices, [per_class[index][key] for index in indices], "o-", label=label)
            support_indices = [index for index, row in enumerate(per_class) if _eda_finite(row.get("support"))]
            if support_indices:
                axes[2].bar(support_indices, [per_class[index]["support"] for index in support_indices], color="grey")
            else:
                axes[2].text(.5, .5, "Class support unavailable", ha="center", va="center", transform=axes[2].transAxes)
            for ax in axes:
                ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right")
                if ax.lines:
                    ax.legend()
            for ax in axes[:2]:
                ax.set_ylim(0, 1)
                ax.set_ylabel("Measured metric")
            axes[2].set_ylabel("Ground-truth instance support")
            fig.suptitle(identity + " — measured class metrics; missing values are omitted")
            fig.tight_layout()
            files.append(_eda_save_figure(fig, target / "class_metrics"))
        curves = validation.get("curves")
        if curves:
            _eda_write_json(target / "curves.json", curves)
        curve_status = _eda_curve_plots(curves, target) if curves else {"status": "UNAVAILABLE"}
        confusion = validation.get("confusion_matrix") or {}
        confusion_status = {"status": "UNAVAILABLE", "reason": "No measured confusion matrix was collected."}
        if (confusion.get("measured") is True or confusion.get("status") == "MEASURED") and confusion.get("matrix") is not None:
            matrix = _eda_np.asarray(confusion["matrix"], dtype=float)
            labels = confusion.get("labels", [])
            if matrix.ndim != 2 or matrix.shape != (len(labels), len(labels)) or not _eda_np.isfinite(matrix).all():
                raise ValueError("Confusion matrix dimensions/labels/values are invalid.")
            _eda_write_json(target / "confusion_matrix.json", confusion)
            _eda_write_csv(target / "confusion_matrix.csv", [
                {"predicted_class": labels[index], **{label: float(value) for label, value in zip(labels, row)}}
                for index, row in enumerate(matrix)])
            fig, ax = _eda_plt.subplots(figsize=(9, 8))
            plotted = ax.imshow(matrix, cmap="Blues")
            ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right")
            ax.set_yticks(range(len(labels)), labels)
            ax.set_xlabel(confusion.get("x_label", "True class"))
            ax.set_ylabel(confusion.get("y_label", "Predicted class"))
            conf = confusion.get("confidence_threshold", confusion.get("conf"))
            matching_iou = confusion.get("matching_iou_threshold", confusion.get("iou"))
            ax.set_title(f"{identity}: measured confusion matrix\nconf={conf}, matching IoU={matching_iou}")
            fig.colorbar(plotted, ax=ax)
            fig.tight_layout()
            files.append(_eda_save_figure(fig, target / "confusion_matrix"))
            confusion_status = {"status": "MEASURED", "confidence_threshold": conf, "matching_iou_threshold": matching_iou,
                                "validation_iou": confusion.get("validation_iou")}
        statuses[identity] = {"status": "MEASURED", "per_class": "MEASURED" if per_class else "UNAVAILABLE",
                              "curves": curve_status, "confusion_matrix": confusion_status}
    return {"status": "MEASURED" if any(row["status"] == "MEASURED" for row in statuses.values()) else "NOT_RUN",
            "measurements": statuses, "files": files,
            "interpretation": "Descriptive association; layer changes do not establish a cause of mAP changes."}


def _eda_curve_plots(curves, folder):
    entries = curves if isinstance(curves, list) else curves.get("curves", curves.get("items", []))
    count, files = 0, []
    for entry in entries:
        if not isinstance(entry, dict) or entry.get("status") in {"UNAVAILABLE", "NOT_RUN"}:
            continue
        x, y = entry.get("x"), entry.get("y")
        if x is None or y is None:
            continue
        xs, ys = _eda_np.asarray(x, dtype=float), _eda_np.asarray(y, dtype=float)
        if xs.ndim != 1 or not xs.size or not _eda_np.isfinite(xs).all() or not _eda_np.isfinite(ys).all():
            continue
        if ys.ndim == 1:
            ys = ys[None, :]
        if ys.ndim != 2 or ys.shape[-1] != len(xs):
            continue
        fig, ax = _eda_plt.subplots(figsize=(10, 6))
        labels = entry.get("labels") or entry.get("class_names") or []
        for index, values in enumerate(ys):
            ax.plot(xs, values, label=str(labels[index]) if index < len(labels) else f"series {index}", alpha=.75)
        ax.set_xlabel(entry.get("x_label", entry.get("xlabel", "x")))
        ax.set_ylabel(entry.get("y_label", entry.get("ylabel", "y")))
        ax.set_title(entry.get("name", "Measured validation curve"))
        if len(ys) <= 20:
            ax.legend()
        fig.tight_layout()
        files.append(_eda_save_figure(fig, folder / f"evaluation_curve_{count:02d}"))
        count += 1
    return {"status": "MEASURED" if count else "UNAVAILABLE", "curve_count": count, "files": files}


def _build_run_eda(result, run_dir, inventory_before, inventory_after):
    """Export ordered EDA from actual run records; failures propagate to run reports."""
    folder = _EdaPath(run_dir) / "eda"
    folder.mkdir(parents=True, exist_ok=True)
    rows = layer_eda_rows(inventory_before, inventory_after)
    conv_rows = [row for row in rows if row["module_type"] == "Conv2d"]
    stages = stage_eda_rows(rows)
    _eda_write_csv(folder / "layer_eda.csv", rows)
    _eda_write_csv(folder / "stage_eda.csv", stages)
    audit = result.get("audit") or {}
    title = f"{result.get('model_name', 'YOLO26' + result.get('model_variant', 'n'))} {result.get('policy', '')} pr{result.get('ratio', 0):g}"
    report = {"status": "MEASURED", "model_variant": result.get("model_variant", "n"),
              "model_name": result.get("model_name", "YOLO26n"), "policy": result.get("policy"), "ratio": result.get("ratio"),
              "section_order": ["identity_coverage", "channels", "masks_lamp", "efficiency", "validation_tradeoffs"],
              "identity_coverage": {"provenance": result.get("provenance"), "conv_layers": len(conv_rows),
                                    "bn_layers": sum(row["module_type"] == "BatchNorm2d" for row in rows),
                                    "eligible_layers": sum(bool(row["eligible"]) for row in rows),
                                    "protected_layers": sum(bool(row["protection_reason"]) for row in rows),
                                    "group_count": result.get("pruned_group_count"),
                                    "protection_reasons": sorted({str(row["protection_reason"]) for row in rows if row["protection_reason"]})},
              "channels": _eda_channel_plots(conv_rows, folder, title),
              "channel_mapping": _eda_mapping_plots(audit, conv_rows, folder),
              "lamp_scores": _eda_lamp_plots(audit, folder),
              "efficiency": _eda_efficiency_plots(stages, result, folder),
              "validation_tradeoffs": _eda_evaluation_plots(result, folder),
              "notes": ["Channel figures include all Conv outputs in original module order.",
                        "Independent root-channel counts differ from physical coupled Conv-output changes.",
                        "No activation feature maps, simulated accuracy, or reconstructed historical channel indices."]}
    _eda_write_json(folder / "eda_status.json", report)
    return report


def build_run_eda(result, run_dir, inventory_before, inventory_after):
    """Persist an explicit failed EDA report before propagating analysis errors."""
    try:
        return _build_run_eda(result, run_dir, inventory_before, inventory_after)
    except Exception as error:
        _eda_write_json(_EdaPath(run_dir) / "eda/eda_status.json",
                        {"status": "FAIL", "model_variant": result.get("model_variant", "n"),
                         "policy": result.get("policy"), "ratio": result.get("ratio"),
                         "error": {"type": type(error).__name__, "message": str(error)}})
        raise


def plot_model_comparison(suites, output_dir):
    """Compare N/S only where each point has measured accuracy and efficiency."""
    folder = _EdaPath(output_dir)
    folder.mkdir(parents=True, exist_ok=True)
    suites = list(suites.values()) if isinstance(suites, dict) else list(suites)
    points = []
    for suite in suites:
        variant = suite.get("model_variant", "n")
        name = suite.get("model_name", "YOLO26" + variant)
        baseline = suite.get("baseline") or {}
        for ratio, policy, metrics in [(0., "baseline", baseline)] + [
                (run["ratio"], run.get("policy", "extended"), run.get("pruned") or {})
                for run in suite.get("results", [])]:
            validation = metrics.get("validation") or {}
            latency = metrics.get("latency") or {}
            points.append({"model_variant": variant, "model_name": name, "ratio": ratio, "policy": policy,
                           "map50_95": validation.get("map50_95"), "parameters": metrics.get("parameters"),
                           "gflops": metrics.get("gflops"),
                           "latency_ms": latency.get("median_ms") if latency.get("status") in {"PASS", "MEASURED"} else None,
                           "latency_p05_ms": latency.get("p05_ms") if latency.get("status") in {"PASS", "MEASURED"} else None,
                           "latency_p95_ms": latency.get("p95_ms") if latency.get("status") in {"PASS", "MEASURED"} else None})
    _eda_write_csv(folder / "model_comparison.csv", points)
    statuses = []
    for metric, label in (("parameters", "Parameters"), ("gflops", "GFLOPs"), ("latency_ms", "Median raw model forward latency (ms)")):
        valid = [row for row in points if _eda_finite(row.get(metric)) and _eda_finite(row.get("map50_95"))]
        if not valid:
            statuses.append({"plot": "map_vs_" + metric, "status": "NOT_RUN", "reason": "Paired measured accuracy/efficiency is unavailable."})
            continue
        fig, ax = _eda_plt.subplots(figsize=(10, 7))
        for variant in dict.fromkeys(row["model_variant"] for row in valid):
            rows = [row for row in valid if row["model_variant"] == variant]
            ax.scatter([row[metric] for row in rows], [row["map50_95"] for row in rows], label="YOLO26" + variant)
            for row in rows:
                if (metric == "latency_ms" and _eda_finite(row.get("latency_p05_ms"))
                        and _eda_finite(row.get("latency_p95_ms"))):
                    ax.hlines(row["map50_95"], row["latency_p05_ms"], row["latency_p95_ms"], alpha=.5)
                ax.annotate(f"{row['model_name']} {row['policy']} pr{row['ratio']:g}", (row[metric], row["map50_95"]),
                            xytext=(5, 5), textcoords="offset points", fontsize=8)
        ax.set_xlabel(label)
        ax.set_ylabel("Measured mAP50–95")
        ax.set_title("N/S compression–accuracy comparison")
        ax.legend()
        fig.tight_layout()
        files = _eda_save_figure(fig, folder / ("map_vs_" + metric))
        statuses.append({"plot": "map_vs_" + metric, "status": "MEASURED", "points": len(valid), "files": files})
    report = {"plots": statuses, "point_count": len(points), "scope": "Each model uses its own measured baseline; no missing accuracy is replaced with zero."}
    _eda_write_json(folder / "comparison_status.json", report)
    return report

## 07 — Implementasi LAMP dan Pencarian Target GFLOPs
Kandidat target mempertahankan proteksi head, attention, split CSP dan antarmuka fitur pada seluruh tier cap.
Pencarian mencatat GFLOPs aktual dan target yang melampaui kapasitas.

Feasibility dihitung per model dari GFLOPs maksimum pada ketiga tier sebelum kandidat target dievaluasi.
Target di atas batas terukur diberi `TARGET_NOT_REACHED` dengan GFLOPs dan speed-up aktual.
Baris dengan plan identik menunjuk ke `artifact_configuration_id` yang sama.
Kandidat maksimum memakai ID `target-1.70x` bila berbagi plan dengan target lain yang tidak tercapai.
Gunakan checkpoint kanonis satu kali jika menjalankan recovery terpisah.
Untuk grid sembilan target, gunakan folder suite baru.

In [ ]:
"""GFLOPs-target planner and tensor oracle embedded after the V2 policy."""
# ruff: noqa: F821 -- The standalone notebook loads V2 declarations before this cell.

import math
from contextlib import nullcontext
from copy import deepcopy
from types import SimpleNamespace

import torch
import torch_pruning as tp

GFLOPS_CAP_TIERS = ((25, 10), (35, 20), (50, 50))
GFLOPS_INTERFACE_STAGES = (0, 1, 3, 4, 5, 6, 7, 13, 16, 17, 19, 20, 22)
GFLOPS_P5_ROOT = "model.7.conv"


def _gflops_rank_space(audit, caps, protect_interfaces, exclude_p5):
    """Keep original LAMP scores and bound capacity before ranking candidates."""
    if len(caps) != 2 or any(type(cap) is not int or not 0 <= cap <= 50 for cap in caps):
        raise ValueError("Cap harus berisi dua persentase integer antara 0 dan 50.")
    structure = audit["structure"]
    interfaces = set()
    if protect_interfaces:
        for stage in GFLOPS_INTERFACE_STAGES:
            block = structure["modules"][f"model.{stage}"]
            interface = getattr(block, "cv2", block).conv
            if not isinstance(interface, torch.nn.Conv2d):
                raise TypeError(f"Output antarmuka YOLO26 tidak didukung: model.{stage}.")
            interfaces.add(interface)
    if exclude_p5:
        interfaces.add(structure["modules"][GFLOPS_P5_ROOT])

    ranked, capacities, eligible_channels = [], {}, 0
    for group in audit["eligible"].values():
        if interfaces.intersection(group["output_dimensions"]):
            continue
        width = group["initial_channels"]
        percent = caps[0] if group["root"] in structure["conservative"] else caps[1]
        capacity = min(width - 1, *(size * percent // 100 for size in group["output_dimensions"].values()))
        root = group["name"]
        scores = group["initial_lamp_scores"]
        if len(scores) != width or not all(math.isfinite(score) for score in scores):
            raise ValueError(f"Skor LAMP tidak valid: {root}.")
        eligible_channels += width
        capacities[root] = capacity
        ranked.extend(sorted((float(score), root, index) for index, score in enumerate(scores))[:capacity])
    return sorted(ranked), capacities, eligible_channels


def gflops_ranked_candidates(audit, caps, protect_interfaces=True, exclude_p5=False):
    return _gflops_rank_space(audit, caps, protect_interfaces, exclude_p5)[0]


def _gflops_prefix_plan(ranked, count):
    if type(count) is not int or not 0 <= count <= len(ranked):
        raise ValueError(f"Target {count} channel melampaui kapasitas {len(ranked)}.")
    plan = {}
    for _, root, index in ranked[:count]:
        plan.setdefault(root, []).append(index)
    return {root: sorted(indices) for root, indices in sorted(plan.items())}


def gflops_plan_key(plan):
    """Canonical identity for caching and deduplicating equal physical plans."""
    return tuple((root, tuple(sorted(indices))) for root, indices in sorted(plan.items()) if indices)


def p5_control_plans(audit, extended_10_history):
    """Use the historical 10% mask itself as the source of the P5-only dose."""
    groups = [entry for entry in audit["eligible"].values() if GFLOPS_P5_ROOT in entry["signature"]]
    if len(groups) != 1 or groups[0]["name"] != GFLOPS_P5_ROOT:
        raise ValueError("Root canonical P5 harus unik dan cocok dengan model.7.conv.")
    history = {entry["root"]: entry["removed_original_indices"] for entry in extended_10_history}
    if len(history) != len(extended_10_history) or not history.get(GFLOPS_P5_ROOT):
        raise ValueError("Kontrol extended 10% harus mencatat satu penghapusan group P5.")
    count = sum(len(indices) for indices in history.values())
    candidates = gflops_ranked_candidates(audit, (50, 50), protect_interfaces=False, exclude_p5=True)
    return {"p5-only": {GFLOPS_P5_ROOT: sorted(history[GFLOPS_P5_ROOT])},
            "p5-protected": _gflops_prefix_plan(candidates, count)}


def apply_exact_gflops_plan(pruner, audit, plan, caps=None):
    """Run the existing dependency verifier and ledger with exact finite selections."""
    groups = {entry["name"]: entry for entry in audit["eligible"].values()}
    expected_caps = {name: _v2_capacity(entry) for name, entry in groups.items()}
    if caps is not None:
        if any(name not in groups or type(cap) is not int or not 0 <= cap <= expected_caps[name]
               for name, cap in caps.items()):
            raise ValueError("Cap plan tidak sesuai dengan dependency group sumber.")
        expected_caps = {name: caps.get(name, 0) for name in groups}
        eligible_channels = sum(groups[name]["initial_channels"] for name in caps)
        audit["report"]["target_policy_eligible_channels"] = eligible_channels

    graph = pruner.DG
    selections = gflops_plan_key(plan)
    if not selections:
        raise ValueError("Plan pruning harus memiliki setidaknya satu channel.")
    for name, indices in selections:
        entry = groups.get(name)
        if (entry is None or len(indices) > expected_caps[name] or len(indices) != len(set(indices))
                or any(type(index) is not int or not 0 <= index < entry["initial_channels"] for index in indices)):
            raise ValueError(f"Plan channel invalid atau melebihi cap: {name}.")
        if entry["root"].out_channels != entry["initial_channels"]:
            raise ValueError(f"Plan harus dimulai dari indeks baseline: {name}.")

    audit["report"]["ranking"] = "Finite LAMP scores sorted by (score, root, original index), exact integer budget"
    audit["report"]["planned_original_indices"] = {name: list(indices) for name, indices in selections}
    audit["report"]["planned_group_capacities"] = expected_caps
    if caps is not None:
        permitted = {}
        for name, entry in groups.items():
            scores = entry["initial_lamp_scores"]
            permitted[name] = set(sorted(range(len(scores)), key=lambda index: (scores[index], index))[:expected_caps[name]])
        for row in audit["report"]["lamp_scores"]:
            row["capacity_eligible"] = row["channel_idx"] in permitted[row["root"]]
        if any(not set(indices).issubset(permitted[name]) for name, indices in selections):
            raise ValueError("Plan memilih channel di luar peringkat LAMP dalam batas cap.")

    def step(interactive=True):
        if not interactive:
            raise ValueError("Pruning memerlukan pemeriksaan dependency interaktif.")
        for name, indices in selections:
            entry = groups[name]
            group = graph.get_pruning_group(entry["root"], tp.prune_conv_out_channels, list(indices))
            # All coupled Conv outputs must lose exactly the same number of channels.
            for dependency, mapped in group:
                dimension = _v2_dependency_dimension(dependency, graph)
                if dimension is not None and dimension[1] == "out" and len(mapped) != len(indices):
                    raise ValueError(f"Jumlah survivor output residual tidak konsisten: {name}.")
            yield group

    history = apply_policy_pruning(SimpleNamespace(DG=graph, step=step), audit)
    applied = {entry["root"]: entry["removed_original_indices"] for entry in history}
    if gflops_plan_key(applied) != selections:
        raise ValueError("Mask aktual berbeda dari plan indeks asli.")
    return history


def search_gflops_targets(audit, baseline_gflops, measure_plan,
                         targets=(1.02, 1.05, 1.10, 1.20, 1.70), *, profile_cache=None, progress_callback=None):
    """Original finite LAMP prefix search, with resumable profiling and isolated failures."""
    if not math.isfinite(baseline_gflops) or baseline_gflops <= 0:
        raise ValueError("GFLOPs baseline harus finite dan positif.")
    targets = tuple(targets)
    if not targets or len(targets) != len(set(targets)) or any(not math.isfinite(t) or t <= 1 for t in targets):
        raise ValueError("Target GFLOPs harus unik, finite, dan lebih dari satu.")
    cached = {} if profile_cache is None else profile_cache
    cached[repr(())] = float(baseline_gflops)
    measurements, tiers, spaces, errors = {}, [], {}, []

    def profile(ranked, count, tier):
        plan = _gflops_prefix_plan(ranked, count)
        key = repr(gflops_plan_key(plan))
        if key not in cached:
            measured = float(measure_plan(plan))
            if not math.isfinite(measured) or measured <= 0 or measured > baseline_gflops + 1e-12:
                raise ValueError(f"GFLOPs tidak valid untuk tier {tier}, {count} channel.")
            cached[key] = measured
        value = cached[key]
        if not math.isfinite(value) or not 0 < value <= baseline_gflops + 1e-12:
            raise ValueError("Cache profil GFLOPs tidak valid.")
        measurements[tier, count] = {"tier": tier, "channels": count, "gflops": value,
                                     "speed_up": baseline_gflops / value}
        if progress_callback:
            progress_callback(cached, list(measurements.values()))
        return value, plan

    for tier, cap in enumerate(GFLOPS_CAP_TIERS, 1):
        ranked, capacities, eligible = _gflops_rank_space(audit, cap, True, False)
        entry = {"tier": tier, "hidden_cap_pct": cap[0], "other_cap_pct": cap[1],
                 "eligible_channels": eligible, "capacity_channels": len(ranked), "status": "NOT_RUN"}
        try:
            measured, plan = profile(ranked, len(ranked), tier)
            entry.update(status="PASS", max_gflops_speed_up=baseline_gflops / measured, gflops_at_capacity=measured)
            spaces[tier] = ranked, capacities, measured, plan
        except Exception as error:
            entry.update(status="FAIL", error=str(error))
            errors.append({"tier": tier, "stage": "capacity", "error": str(error)})
        tiers.append(entry)

    rows = []
    for target in targets:
        try:
            feasible = next((entry for entry in tiers if entry["status"] == "PASS"
                             and entry["max_gflops_speed_up"] >= target), None)
            if feasible is None:
                valid = [entry for entry in tiers if entry["status"] == "PASS"]
                if len(valid) != len(tiers):
                    raise ValueError("Kapasitas belum dapat dibuktikan karena ada profil tier yang gagal.")
                best = max(valid, key=lambda entry: (entry["max_gflops_speed_up"], -entry["tier"]))
                ranked, capacities, measured, plan = spaces[best["tier"]]
                rows.append({"target": target, "status": "TARGET_NOT_REACHED", "tier": best["tier"],
                             "channels": len(ranked), "gflops": measured, "speed_up": baseline_gflops / measured,
                             "plan": plan, "caps": capacities})
                continue
            tier = feasible["tier"]
            ranked, capacities, _, _ = spaces[tier]
            checkpoints = sorted({min(len(ranked), (feasible["eligible_channels"] * pct + 99) // 100)
                                  for pct in range(1, 51)} | {len(ranked)})
            previous, previous_flops = 0, baseline_gflops
            for checkpoint in checkpoints:
                measured, plan = profile(ranked, checkpoint, tier)
                if measured > previous_flops + 1e-12:
                    raise ValueError("GFLOPs bertambah pada prefix dalam tier.")
                if baseline_gflops / measured >= target:
                    break
                previous, previous_flops = checkpoint, measured
            for count in range(previous + 1, checkpoint + 1):
                measured, plan = profile(ranked, count, tier)
                if measured > previous_flops + 1e-12:
                    raise ValueError("GFLOPs bertambah pada refinement integer.")
                if baseline_gflops / measured >= target:
                    break
                previous_flops = measured
            rows.append({"target": target, "status": "REACHED", "tier": tier, "channels": count,
                         "gflops": measured, "speed_up": baseline_gflops / measured,
                         "plan": plan, "caps": capacities})
        except Exception as error:
            rows.append({"target": target, "status": "FAIL", "error": str(error), "plan": None})
    return {"targets": rows, "tiers": tiers, "errors": errors,
            "measurements": [measurements[key] for key in sorted(measurements)]}


def _gflops_checked_masks(audit):
    mappings = {}
    for row in audit["report"]["channel_mapping"]:
        key = row["layer"], row["dimension"]
        kept = row["kept_original_indices"]
        removed = row["removed_original_indices"]
        if (key in mappings or len(kept) != row["channels_after"] or not kept
                or kept != sorted(kept) or removed != sorted(removed)
                or sorted([*kept, *removed]) != list(range(row["channels_before"]))):
            raise ValueError(f"Ledger indeks asli tidak valid: {key}.")
        mappings[key] = row
    return mappings


def _gflops_index(tensor, axis, indices):
    return tensor.index_select(axis, torch.tensor(indices, device=tensor.device, dtype=torch.long))


def verify_gflops_oracle(baseline, pruned, image, audit):
    """Verify exact parameter slicing and functional parity against a dense oracle."""
    source, target = dict(baseline.named_modules()), dict(pruned.named_modules())
    if source.keys() != target.keys() or baseline.training or pruned.training:
        raise ValueError("Oracle memerlukan module tree yang sama dalam mode eval.")
    mappings = _gflops_checked_masks(audit)
    candidate_device = next(pruned.parameters()).device
    masked = deepcopy(baseline).to(candidate_device).eval()
    masked_modules = dict(masked.named_modules())
    verified, changed_modules, masked_inputs = 0, 0, 0
    with torch.no_grad():
        for name, module in source.items():
            current = target[name]
            if type(module) is not type(current):
                raise ValueError(f"Jenis modul berubah: {name}.")
            if isinstance(module, torch.nn.Conv2d):
                inputs, outputs = (mappings.pop((name, dimension)) for dimension in ("in", "out"))
                attributes = ("groups", "kernel_size", "stride", "padding", "dilation", "padding_mode")
                if any(getattr(module, key) != getattr(current, key) for key in attributes):
                    raise ValueError(f"Konfigurasi Conv berubah: {name}.")
                if ((inputs["channels_before"], outputs["channels_before"]) != (module.in_channels, module.out_channels)
                        or (inputs["channels_after"], outputs["channels_after"]) != (current.in_channels, current.out_channels)):
                    raise ValueError(f"Lebar Conv berbeda dari ledger: {name}.")
                dropped = inputs["removed_original_indices"]
                sliced = bool(dropped or outputs["removed_original_indices"])
                if sliced and (module.groups != 1 or current.groups != 1):
                    raise ValueError(f"Oracle tidak mendukung perubahan grouped/depthwise Conv: {name}.")
                weight, bias = module.weight, module.bias
                if sliced:
                    weight = _gflops_index(weight, 0, outputs["kept_original_indices"])
                    weight = _gflops_index(weight, 1, inputs["kept_original_indices"])
                    if bias is not None:
                        bias = _gflops_index(bias, 0, outputs["kept_original_indices"])
                    changed_modules += 1
                if dropped:
                    consumer = masked_modules[name]
                    consumer.weight.index_fill_(1, torch.tensor(dropped, device=candidate_device, dtype=torch.long), 0)
                    masked_inputs += 1
                tensors = [("weight", weight), ("bias", bias)]
            elif isinstance(module, torch.nn.BatchNorm2d):
                channels = mappings.pop((name, "channel"))
                if (channels["channels_before"] != module.num_features
                        or channels["channels_after"] != current.num_features or module.training or current.training):
                    raise ValueError(f"Lebar atau mode BN berbeda dari ledger: {name}.")
                if any(getattr(module, key) != getattr(current, key) for key in ("eps", "affine", "track_running_stats")):
                    raise ValueError(f"Konfigurasi BN berubah: {name}.")
                sliced = bool(channels["removed_original_indices"])
                changed_modules += int(sliced)
                tensors = []
                for key in ("weight", "bias", "running_mean", "running_var", "num_batches_tracked"):
                    value = getattr(module, key)
                    if sliced and value is not None and key != "num_batches_tracked":
                        value = _gflops_index(value, 0, channels["kept_original_indices"])
                    tensors.append((key, value))
            else:
                continue
            for key, value in tensors:
                actual = getattr(current, key)
                if ((value is None) != (actual is None)
                        or (value is not None and (value.dtype != actual.dtype or not torch.equal(value.to(actual.device), actual)))):
                    raise ValueError(f"Irisan tensor berbeda dari baseline: {name}.{key}.")
                verified += int(value is not None)
        if mappings:
            raise ValueError("Ledger memuat dimensi tanpa modul Conv/BN yang sesuai.")

        example = image.to(candidate_device)
        cudnn = torch.backends.cudnn
        # Dense and compact kernels can choose different accumulation paths. Use
        # FP32 for the comparison on CUDA and restore the caller's backend flags.
        precision = (cudnn.flags(enabled=cudnn.enabled, benchmark=cudnn.benchmark,
                                 deterministic=cudnn.deterministic, allow_tf32=False)
                     if candidate_device.type == "cuda" else nullcontext())
        with precision:
            oracle_outputs = _v2_trace_outputs(masked(example))
            compact_outputs = _v2_trace_outputs(pruned(example))
        metrics = {}
        names = ("one2many.boxes", "one2many.scores", "one2one.boxes", "one2one.scores")
        for name, reference, actual in zip(names, oracle_outputs, compact_outputs):
            if (reference.shape != actual.shape or not torch.isfinite(reference).all()
                    or not torch.isfinite(actual).all()
                    or not torch.allclose(reference, actual, rtol=1e-4, atol=1e-5)):
                raise ValueError(f"Forward raw berbeda dari oracle masked: {name}.")
            metrics[name] = {"shape": list(actual.shape), "max_abs_diff": float((reference - actual).abs().max())}
    return {"status": "PASS", "conv_bn_tensors_verified": verified, "changed_conv_bn_modules": changed_modules,
            "masked_conv_consumers": masked_inputs, "raw_outputs": metrics, "rtol": 1e-4, "atol": 1e-5,
            "scope": "Exact baseline slices and dense consumer-column mask parity on the supplied FP32 input."}

## 08 — Eksperimen Independen dan Kontrol P5
Setiap konfigurasi memulai dari checkpoint baseline variannya.

| Konfigurasi | Pembanding |
| --- | --- |
| Extended 10–50% | Baseline dari varian yang sama |
| P5-only | Baseline, dengan indeks P5 diambil dari extended 10% |
| P5-protected | Extended 10% dengan jumlah channel independen yang dibuang sama |
| Target 1,02× / 1,05× / 1,10× / 1,20× / 1,30× / 1,40× / 1,50× / 1,60× / 1,70× | Baseline dan GFLOPs aktual kandidat |

## 09 — Simpan Checkpoint dan Fresh Reload
Uji muat ulang objek DetectionModel FP32 pada proses baru dan bandingkan output kedua cabang Detect.

In [ ]:
RELOAD_SCRIPT = r'''
import json, sys
from pathlib import Path
import numpy as np
import torch
from ultralytics import YOLO

checkpoint, reference, result, device = sys.argv[1:]
torch.set_num_threads(2)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision("highest")
payload = torch.load(reference, map_location="cpu", weights_only=True)
wrapper = YOLO(checkpoint, task="detect")
core = wrapper.model.to(device).float().eval()
with torch.no_grad():
    output = core(payload["image"].to(device))
snapshot = {"decoded": output[0].detach().cpu()}
for branch in ("one2many", "one2one"):
    for key in ("boxes", "scores"):
        snapshot[branch + "/" + key] = output[1][branch][key].detach().cpu()
for key, expected in payload["expected"].items():
    if not torch.isfinite(snapshot[key]).all():
        raise RuntimeError("Non-finite output after reload: " + key)
    torch.testing.assert_close(snapshot[key], expected, rtol=1e-4, atol=1e-5)
if sum(p.numel() for p in core.parameters()) != payload["parameters"]:
    raise RuntimeError("Parameter count changed after reload")
if not all(p.dtype == torch.float32 for p in core.parameters()):
    raise RuntimeError("Reload is not FP32")
torch.save(snapshot, result)
# Fresh wrapper protects the raw unfused model from inference backend fusion.
predictor = YOLO(checkpoint, task="detect")
size = payload["image"].shape[-1]
predictions = predictor.predict(source=np.zeros((size, size, 3), dtype=np.uint8),
    imgsz=size, device=device, conf=0.001, iou=0.7, max_det=300,
    quantize=None, nms=None, verbose=False, save=False)
if len(predictions) != 1 or predictions[0].boxes is None:
    raise RuntimeError("No valid detection Results after reload")
boxes = predictions[0].boxes.data.detach().cpu()
if boxes.ndim != 2 or boxes.shape[1] != 6 or not torch.isfinite(boxes).all():
    raise RuntimeError("Invalid detection tensor")
nc = len(core.names)
if boxes.numel() and (((boxes[:, 5] < 0) | (boxes[:, 5] >= nc) | (boxes[:, 5] != boxes[:, 5].floor())).any()
                     or ((boxes[:, 4] < 0) | (boxes[:, 4] > 1)).any()):
    raise RuntimeError("Invalid detection class/confidence")
Path(result).with_suffix(".json").write_text(json.dumps({"prediction_count": len(boxes),
    "parameters": payload["parameters"], "direct_yolo_load": True,
    "raw_parity": True, "prediction_shape": list(boxes.shape)}), encoding="utf-8")
'''


def save_fp32_checkpoint(core, path, provenance):
    """Simpan arsitektur hasil pruning memakai format model-object Ultralytics."""
    model = deepcopy(core).cpu().float().eval().requires_grad_(False)
    for module in model.modules():
        module._forward_hooks.clear()
        module._forward_pre_hooks.clear()
        module._backward_hooks.clear()
    model.args = {"task": "detect", "imgsz": IMGSZ, "data": str(DATA_YAML), "nms": None, "quantize": None}
    torch.save({"model": model, "ema": None, "optimizer": None, "epoch": -1,
                "train_args": dict(model.args), "version": ultralytics.__version__,
                "date": datetime.now(timezone.utc).isoformat(), "lamp_format_version": 1,
                "lamp_provenance": provenance, "recovery_enabled": False}, path)
    return {"path": str(Path(path).resolve()), "sha256": sha256_file(path), "file_bytes": Path(path).stat().st_size}


def verify_fresh_reload(checkpoint, core, image, output_dir, *, timeout=300):
    """Restore di interpreter baru; exit code/error tidak boleh disembunyikan."""
    output_dir = Path(output_dir).resolve()
    script = output_dir / "verify_reload.py"
    reference = output_dir / "reload_reference.pt"
    result = output_dir / "reload_output.pt"
    script.write_text(RELOAD_SCRIPT, encoding="utf-8")
    expected = forward_snapshot(core, image)
    torch.save({"image": image.detach().cpu(), "expected": expected,
                "parameters": sum(p.numel() for p in core.parameters())}, reference)
    child_env = dict(os.environ)
    child_env.pop("PYTHONPATH", None)
    child_env["WANDB_MODE"] = "disabled"
    child_env["YOLO_AUTOINSTALL"] = "false"
    completed = subprocess.run([sys.executable, "-B", str(script), str(Path(checkpoint).resolve()),
                                str(reference), str(result), str(next(core.parameters()).device)],
                               cwd=output_dir, env=child_env, capture_output=True, text=True,
                               encoding="utf-8", errors="replace", timeout=timeout)
    (output_dir / "reload_stdout.txt").write_text(completed.stdout, encoding="utf-8")
    (output_dir / "reload_stderr.txt").write_text(completed.stderr, encoding="utf-8")
    if completed.returncode:
        raise RuntimeError(f"Fresh-process reload gagal (exit {completed.returncode}): {completed.stderr[-6000:]}")
    actual = torch.load(result, map_location="cpu", weights_only=True)
    for key in expected:
        torch.testing.assert_close(actual[key], expected[key], rtol=1e-4, atol=1e-5)
    report = json.loads(result.with_suffix(".json").read_text(encoding="utf-8"))
    report["max_absolute_difference"] = {key: float((actual[key] - expected[key]).abs().max()) for key in expected}
    report["process"] = "fresh Python, no repository PYTHONPATH"
    return report

## 10 — Validasi Baseline dan Pruned
Ukur baseline dan pruned dengan split, citra, serta konfigurasi evaluator yang sama.
Structural PASS, reload PASS, evaluasi, dan latency memiliki status terpisah.

In [ ]:
def prepare_validation_dataset(data_yaml, dataset_root, output_dir):
    """Validasi dataset lokal tanpa download atau normalisasi label otomatis."""
    source = Path(data_yaml).expanduser()
    if not source.is_file():
        return None
    raw = yaml.safe_load(source.read_text(encoding="utf-8"))
    if not isinstance(raw, dict) or not raw.get("val") or not raw.get("names"):
        raise ValueError("DATA_YAML tersedia tetapi tidak berisi val/names yang valid.")
    raw.pop("download", None)
    root = Path(dataset_root).expanduser().resolve() if dataset_root is not None else source.resolve().parent
    if not root.is_dir():
        raise FileNotFoundError(f"DATASET_ROOT tidak tersedia: {root}")
    raw["path"] = str(root)
    copy_path = Path(output_dir) / "evaluation_data.yaml"
    copy_path.write_text(yaml.safe_dump(raw, sort_keys=False, allow_unicode=True), encoding="utf-8")
    info = check_det_dataset(str(copy_path), autodownload=False)
    splits = info["val"] if isinstance(info["val"], list) else [info["val"]]
    if any(not Path(split).exists() for split in splits):
        raise FileNotFoundError("Validation split tidak tersedia.")
    return {"yaml": str(copy_path.resolve()), "info": info, "source_yaml": str(source.resolve()),
            "source_yaml_sha256": sha256_file(source), "evaluation_yaml_sha256": sha256_file(copy_path),
            "val_paths": [str(split) for split in splits], "class_names": ordered_names(info["names"])}


def validation_arguments(data_yaml, output_dir, label):
    """Satu sumber pengaturan evaluasi untuk baseline dan pruned model."""
    return {"data": str(data_yaml), "split": "val", "imgsz": IMGSZ, "batch": BATCH,
            "workers": WORKERS, "device": str(TARGET_DEVICE), "conf": 0.001,
            "iou": 0.7, "max_det": 300, "quantize": None, "nms": None,
            "augment": False, "plots": False, "verbose": False, "save_json": False,
            "project": str(output_dir), "name": label, "exist_ok": False}


def evaluate_checkpoint(path, dataset, output_dir, label):
    """Evaluasi wrapper baru agar fusion validator tidak mengubah model pengukuran."""
    wrapper = YOLO(str(path), task="detect")
    assert_class_match(wrapper.names, dataset["info"]["names"])
    metric = wrapper.val(**validation_arguments(dataset["yaml"], output_dir, label))
    box = metric.box
    per_class = [{"class_id": i, "name": name, "ap50": None, "ap50_95": None}
                 for i, name in enumerate(dataset["class_names"])]
    for row, class_id in enumerate(box.ap_class_index):
        per_class[int(class_id)].update({"ap50": float(box.ap50[row]), "ap50_95": float(box.ap[row])})
    result = {"map50_95": float(box.map), "map50": float(box.map50),
              "precision": float(box.mp), "recall": float(box.mr), "per_class": per_class,
              "split": "val", "precision_mode": "FP32", "nms": None}
    if any(not math.isfinite(result[key]) for key in ("map50_95", "map50", "precision", "recall")):
        raise ValueError("Metrik evaluasi mengandung NaN/Inf.")
    return result


def cuda_latency(core, image, warmup=50, repeats=200):
    """Latency forward model batch satu; tidak mencakup preprocessing/NMS/disk IO."""
    if image.device.type != "cuda":
        return {"status": "NOT_RUN", "reason": "CUDA tidak tersedia; tidak mengklaim benchmark GPU."}
    core = deepcopy(core).eval().requires_grad_(False)
    samples = []
    with torch.inference_mode():
        for _ in range(warmup):
            core(image)
        torch.cuda.synchronize(image.device)
        for _ in range(repeats):
            torch.cuda.synchronize(image.device)
            started = time.perf_counter()
            core(image)
            torch.cuda.synchronize(image.device)
            samples.append((time.perf_counter() - started) * 1000)
    return {"status": "PASS", "median_ms": float(np.median(samples)), "mean_ms": float(np.mean(samples)),
            "p05_ms": float(np.percentile(samples, 5)), "p95_ms": float(np.percentile(samples, 95)),
            "warmup": warmup, "repeats": repeats, "batch": 1, "imgsz": list(image.shape[-2:]),
            "precision": "FP32", "device": str(image.device), "fusion": False, "scope": "raw model forward"}

In [ ]:
"""Validation declarations embedded verbatim in standalone LAMP V2 notebooks.

The source never imports repository modules at notebook runtime. Numerical exports
come from the actual Ultralytics validator; missing measurements remain null.
"""
# ruff: noqa: F821 -- Notebook configuration/helpers provide these runtime globals.


def validation_arguments(data_yaml, output_dir, label):
    """Identical FP32 protocol for baseline/pruned; plots also collect confusion counts."""
    return {"data": str(data_yaml), "split": "val", "imgsz": IMGSZ, "batch": BATCH,
            "workers": WORKERS, "device": str(TARGET_DEVICE), "conf": 0.001,
            "iou": 0.7, "max_det": 300, "quantize": None, "nms": None,
            "augment": False, "plots": True, "verbose": False, "save_json": False,
            "project": str(output_dir), "name": label, "exist_ok": False}


def _validation_number(value):
    """JSON scalar: unsupported/nonfinite auxiliary metrics are unavailable, not zero."""
    if value is None:
        return None
    try:
        number = float(value)
    except (TypeError, ValueError, OverflowError):
        return None
    return number if math.isfinite(number) else None


def _validation_array(value):
    import numpy as np

    if hasattr(value, "detach"):
        value = value.detach().cpu().numpy()
    return np.asarray(value, dtype=float)


def _validation_json_array(value):
    import numpy as np

    array = _validation_array(value)
    return np.where(np.isfinite(array), array, None).tolist()


def _validation_per_class(metric, class_names):
    """Map SDK result rows via ap_class_index, never by names or array position."""
    import numpy as np

    box = metric.box
    support = getattr(metric, "nt_per_class", None)
    if support is not None:
        support = _validation_array(support)
        if (support.shape != (len(class_names),) or not np.isfinite(support).all()
                or (support < 0).any() or not np.equal(support, np.floor(support)).all()):
            raise ValueError("Support kelas SDK tidak valid atau tidak cocok dengan jumlah kelas.")
    rows = [{"class_id": i, "name": name, "ap50": None, "ap50_95": None,
             "precision": None, "recall": None, "f1": None,
             "support": int(support[i]) if support is not None else None}
            for i, name in enumerate(class_names)]
    indices = list(getattr(box, "ap_class_index", []))
    validated_indices = []
    for value in indices:
        scalar = _validation_number(value)
        if scalar is None or scalar != int(scalar) or not 0 <= int(scalar) < len(rows):
            raise ValueError("Indeks kelas SDK tidak valid.")
        validated_indices.append(int(scalar))
    if len(set(validated_indices)) != len(validated_indices):
        raise ValueError("Indeks kelas SDK duplikat.")
    for offset, class_id in enumerate(validated_indices):
        for key, attribute in (("ap50", "ap50"), ("ap50_95", "ap"),
                               ("precision", "p"), ("recall", "r"), ("f1", "f1")):
            values = getattr(box, attribute, None)
            if values is not None and len(values) > offset:
                rows[class_id][key] = _validation_number(values[offset])
    return rows, validated_indices


def _validation_curves(metric, class_names, class_ids):
    """Keep SDK axes and class mapping, replacing nonfinite samples with JSON null."""
    import numpy as np

    try:
        curve_values = metric.curves_results
    except (AttributeError, TypeError, ValueError) as error:
        return {"status": "UNAVAILABLE", "reason": str(error), "curves": []}
    names = getattr(metric, "curves", [])
    records = []
    for index, values in enumerate(curve_values):
        record = {"name": names[index] if index < len(names) else f"SDK curve {index}",
                  "class_ids": list(class_ids), "class_names": [class_names[i] for i in class_ids],
                  "status": "UNAVAILABLE", "x": [], "y": []}
        try:
            x_value, y_value, x_label, y_label = values
            x, y = _validation_array(x_value), _validation_array(y_value)
            record.update({"x_label": str(x_label), "y_label": str(y_label)})
            if y.ndim == 1 and len(class_ids) == 1:
                y = y[None, :]
            if (not class_ids or x.ndim != 1 or not x.size or y.ndim != 2
                    or y.shape != (len(class_ids), x.size)):
                record["reason"] = "Dimensi kurva tidak memiliki pemetaan kelas SDK yang valid."
            else:
                record.update({"x": _validation_json_array(x), "y": _validation_json_array(y),
                               "status": "MEASURED" if np.isfinite(x).all() and np.isfinite(y).all()
                               else "PARTIAL"})
                if record["status"] == "PARTIAL":
                    record["reason"] = "Sampel SDK NaN/Inf disimpan sebagai null."
        except (TypeError, ValueError, OverflowError) as error:
            record["reason"] = str(error)
        records.append(record)
    states = {item["status"] for item in records}
    status = "MEASURED" if states == {"MEASURED"} else (
        "PARTIAL" if states & {"MEASURED", "PARTIAL"} else "UNAVAILABLE")
    return {"status": status, "curves": records}


def _validation_confusion_matrix(validator, class_names, arguments):
    """Export actual counts and the distinct matching/NMS IoU thresholds."""
    import inspect

    import numpy as np

    matrix_object = getattr(validator, "confusion_matrix", None)
    result = {"status": "UNAVAILABLE", "labels": [*class_names, "background"],
              "orientation": "rows=predicted, columns=true", "matrix": None,
              "normalized_by_true": None, "confidence_threshold": None,
              "matching_iou_threshold": None, "validation_iou": arguments["iou"],
              "normalization": "Divide each true-class column by its sum; empty columns stay zero."}
    if matrix_object is None or getattr(matrix_object, "matrix", None) is None:
        result["reason"] = "Validator tidak menyediakan confusion matrix."
        return result
    matrix = _validation_array(matrix_object.matrix)
    expected = len(class_names) + 1
    if (matrix.shape != (expected, expected) or not np.isfinite(matrix).all() or (matrix < 0).any()):
        raise ValueError("Confusion matrix SDK tidak valid atau class order berbeda.")
    threshold = _validation_number(getattr(validator, "confusion_matrix_conf", None))
    matching_iou = None
    try:
        parameter = inspect.signature(matrix_object.process_batch).parameters.get("iou_thres")
        if parameter is not None and parameter.default is not inspect.Parameter.empty:
            matching_iou = _validation_number(parameter.default)
    except (AttributeError, TypeError, ValueError):
        pass
    totals = matrix.sum(axis=0, keepdims=True)
    normalized = np.divide(matrix, totals, out=np.zeros_like(matrix), where=totals != 0)
    result.update({"status": "MEASURED", "matrix": matrix.tolist(),
                   "normalized_by_true": normalized.tolist(),
                   "zero_support_columns": [int(i) for i in np.flatnonzero(totals[0] == 0)],
                   "confidence_threshold": threshold, "matching_iou_threshold": matching_iou,
                   "matching_iou_source": "Signature default of actual ConfusionMatrix.process_batch; validator calls without iou_thres.",
                   "confidence_source": "Actual DetectionValidator.confusion_matrix_conf"})
    return result


def _validation_metric_semantics(metric):
    """Record the SDK best-F1 reporting point without confusing it with CM confidence."""
    import numpy as np
    from ultralytics.utils.metrics import smooth

    result = {"precision_recall_f1": "SDK values at the global confidence maximizing smoothed mean F1 (IoU=0.50).",
              "reporting_confidence": None,
              "reporting_confidence_source": "SDK px[smooth(f1_curve.mean(0), 0.1).argmax()]",
              "map_iou_thresholds": [round(0.5 + 0.05 * i, 2) for i in range(10)],
              "absent_class": "Support 0 denotes no ground truth; class metrics without an SDK row remain null.",
              "confusion_matrix": "A separate operating point: see confidence_threshold and matching_iou_threshold."}
    box = metric.box
    try:
        px, f1 = _validation_array(box.px), _validation_array(box.f1_curve)
        if (px.ndim == 1 and f1.ndim == 2 and f1.shape[0] > 0 and f1.shape[1] == len(px)
                and np.isfinite(px).all() and np.isfinite(f1).all()):
            result["reporting_confidence"] = float(px[int(smooth(f1.mean(0), 0.1).argmax())])
    except (AttributeError, TypeError, ValueError):
        pass
    return result


def evaluate_checkpoint(path, dataset, output_dir, label):
    """Evaluate a fresh wrapper; preserve SDK metrics, curves and populated confusion counts."""
    wrapper = YOLO(str(path), task="detect")
    assert_class_match(wrapper.names, dataset["info"]["names"])
    class_names = list(dataset["class_names"])
    assert_class_match(wrapper.names, class_names)
    arguments = gflops_validation_arguments(dataset, output_dir, label)
    captured = {}
    capture_validator = None
    if hasattr(wrapper, "add_callback"):
        # Model.val() keeps its validator local in Ultralytics 8.4.155.
        # Read the actual instance at callback time rather than assuming an attribute.
        def capture_validator(validator):
            captured["validator"] = validator

        wrapper.add_callback("on_val_end", capture_validator)
    try:
        with gflops_require_complete_nms(), gflops_disable_label_cache():
            metric = wrapper.val(**arguments)
    finally:
        # Avoid a callback -> capture -> validator cycle retaining GPU allocations
        # between baseline/ratio runs. Removing it also affects the shared callbacks.
        callbacks = getattr(wrapper, "callbacks", {}).get("on_val_end", [])
        if capture_validator in callbacks:
            callbacks.remove(capture_validator)
    box = metric.box
    per_class, class_ids = _validation_per_class(metric, class_names)
    result = {"map50_95": _validation_number(box.map), "map50": _validation_number(box.map50),
              "precision": _validation_number(box.mp), "recall": _validation_number(box.mr),
              "per_class": per_class, "split": "val", "precision_mode": "FP32", "nms": None}
    if any(result[key] is None for key in ("map50_95", "map50", "precision", "recall")):
        raise ValueError("Metrik evaluasi mengandung NaN/Inf atau tidak tersedia.")
    result["curves"] = _validation_curves(metric, class_names, class_ids)
    validator = captured.get("validator", getattr(wrapper, "validator", None))
    result.update(gflops_validation_metadata(validator, arguments, dataset))
    result["confusion_matrix"] = _validation_confusion_matrix(validator, class_names, arguments)
    result["metric_semantics"] = _validation_metric_semantics(metric)
    export_dir = Path(output_dir) / label
    export_dir.mkdir(parents=True, exist_ok=True)
    write_csv(export_dir / "per_class_metrics.csv", per_class)
    write_json(export_dir / "curves.json", result["curves"])
    write_json(export_dir / "confusion_matrix.json", result["confusion_matrix"])
    write_json(export_dir / "metric_semantics.json", result["metric_semantics"])
    result["exports"] = {name: str(export_dir / name) for name in (
        "per_class_metrics.csv", "curves.json", "confusion_matrix.json", "metric_semantics.json")}
    return result

In [ ]:
"""Dataset and student-selection declarations embedded in notebook 05."""
# ruff: noqa: F821 -- Notebook runtime supplies the V2 evaluator and helpers.

import hashlib
import json
import logging
import math
from contextlib import contextmanager
from pathlib import Path
from unittest.mock import patch

POLTRADA_V3_ID = "ilhamrafiqin/poltrada-v3-yolo26-clean"
POLTRADA_V3_MOUNT = Path("/kaggle/input/datasets") / POLTRADA_V3_ID


def _gflops_split_images(root, entries):
    """Resolve directories and image lists using dataset-relative identities."""
    from ultralytics.data.utils import IMG_FORMATS

    if isinstance(entries, str):
        entries = [entries]
    if not isinstance(entries, list) or not entries:
        raise ValueError("Split Poltrada harus berupa direktori atau daftar gambar lokal.")
    images = []
    for entry in entries:
        if not isinstance(entry, str) or not entry.strip():
            raise ValueError("Path split kosong atau tidak valid.")
        path = (root / entry).resolve()
        if (not path.exists() or not path.is_relative_to(root)) and entry.startswith("../"):
            # Roboflow exports commonly use ../valid/images from data.yaml.
            path = (root / entry.removeprefix("../")).resolve()
        if not path.is_relative_to(root):
            raise ValueError(f"Path split berada di luar DATASET_ROOT: {path}")
        if path.is_dir():
            files = (item for item in path.rglob("*") if item.is_file())
        elif path.is_file() and path.suffix.lower() == ".txt":
            files = (path.parent / line.strip() for line in path.read_text(encoding="utf-8-sig").splitlines()
                     if line.strip())
        else:
            raise FileNotFoundError(f"Direktori atau daftar gambar split tidak tersedia: {path}")
        for image in files:
            if image.suffix.lower().removeprefix(".") not in IMG_FORMATS:
                continue
            image = image.resolve()
            if not image.is_file() or not image.is_relative_to(root):
                raise ValueError(f"Gambar tidak tersedia atau berada di luar DATASET_ROOT: {image}")
            images.append(image)
    if not images:
        raise ValueError("Split Poltrada tidak berisi gambar yang dapat dievaluasi.")
    if len(set(images)) != len(images):
        raise ValueError("Split Poltrada memuat path gambar duplikat.")
    return sorted(images, key=lambda image: image.relative_to(root).as_posix())


def _gflops_fingerprint(root, images, split, names):
    """Hash every selected image and label; missing and empty labels stay distinct."""
    from ultralytics.data.utils import img2label_paths

    records = []
    for image, label in zip(images, img2label_paths([str(image) for image in images]), strict=True):
        label = Path(label).resolve()
        if not label.is_relative_to(root):
            raise ValueError(f"Label berada di luar DATASET_ROOT: {label}")
        records.append({"image": image.relative_to(root).as_posix(), "image_sha256": sha256_file(image),
                        "label": label.relative_to(root).as_posix(),
                        "label_sha256": sha256_file(label) if label.is_file() else None})
    # Encoding and sorted path order make the digest independent of the mount point.
    payload = json.dumps({"split": split, "class_names": names, "records": records},
                         ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(payload).hexdigest(), records


def gflops_prepare_dataset(mode, data_yaml, dataset_root, suite_dir, *, seed,
                           expected_fingerprint=None, dataset_id=None):
    """Freeze the input set before evaluating any model in the suite."""
    import yaml

    if mode not in {"smoke", "full_val", "test"}:
        raise ValueError("Mode evaluasi harus smoke, full_val, atau test.")
    if expected_fingerprint is not None:
        if (not isinstance(expected_fingerprint, str) or len(expected_fingerprint) != 64
                or any(char not in "0123456789abcdefABCDEF" for char in expected_fingerprint)):
            raise ValueError("Fingerprint full val harus berupa SHA-256 heksadesimal.")
        expected_fingerprint = expected_fingerprint.lower()
    source = Path(data_yaml).expanduser().resolve()
    if not source.is_file():
        return None
    root = Path(dataset_root).expanduser().resolve() if dataset_root is not None else source.parent
    if not root.is_dir():
        raise FileNotFoundError(f"DATASET_ROOT tidak tersedia: {root}")
    raw = yaml.safe_load(source.read_text(encoding="utf-8-sig"))
    if not isinstance(raw, dict) or not isinstance(raw.get("names"), (dict, list)) or not raw["names"]:
        raise ValueError("DATA_YAML harus memuat names Poltrada yang berurutan.")
    class_names = ordered_names(raw["names"])
    if raw.get("nc", len(class_names)) != len(class_names):
        raise ValueError("nc di DATA_YAML berbeda dari jumlah names.")

    split = "test" if mode == "test" else "val"
    val_images = _gflops_split_images(root, raw.get("val"))
    full_val_fingerprint = None
    if mode != "smoke" or expected_fingerprint is not None:
        full_val_fingerprint, val_records = _gflops_fingerprint(root, val_images, "val", class_names)
        if expected_fingerprint is not None and full_val_fingerprint != expected_fingerprint:
            raise ValueError("Fingerprint full val tidak cocok dengan konfigurasi yang dikunci.")

    official_mount = root == POLTRADA_V3_MOUNT.resolve() and source.is_relative_to(root)
    verified_copy = dataset_id == POLTRADA_V3_ID and expected_fingerprint is not None
    provenance_status = "VERIFIED" if official_mount or verified_copy else "UNVERIFIED"
    provenance_basis = ("KAGGLE_MOUNT" if official_mount else
                        "EXPECTED_FULL_VAL_FINGERPRINT" if verified_copy else "USER_PATH_ONLY")
    if mode == "test":
        if not raw.get("test") or not raw.get("train"):
            return None
        images = _gflops_split_images(root, raw["test"])
        fingerprint, records = _gflops_fingerprint(root, images, split, class_names)
        test_hashes = {row["image_sha256"] for row in records}
        if any(row["image_sha256"] in test_hashes for row in val_records):
            raise ValueError("Split test berbagi konten gambar dengan full val.")
        train_images = _gflops_split_images(root, raw["train"])
        if any(sha256_file(image) in test_hashes for image in train_images):
            raise ValueError("Split test berbagi konten gambar dengan train.")
    elif mode == "smoke":
        ranked = sorted(val_images, key=lambda image: (
            hashlib.sha256(f"{seed}:{image.relative_to(root).as_posix()}".encode()).hexdigest(),
            image.relative_to(root).as_posix()))
        images = sorted(ranked[:128], key=lambda image: image.relative_to(root).as_posix())
        fingerprint, records = _gflops_fingerprint(root, images, split, class_names)
    else:
        images = val_images
        fingerprint, records = full_val_fingerprint, val_records
    if any(image.with_suffix(".npy").is_file() for image in images):
        raise ValueError("Cache gambar .npy berada di samping citra evaluasi. Gunakan salinan dataset tanpa cache.")

    folder = Path(suite_dir).resolve()
    folder.mkdir(parents=True, exist_ok=True)
    paths = folder / f"evaluation_{mode}_images.txt"
    paths.write_text("".join(f"{image.as_posix()}\n" for image in images), encoding="utf-8")
    # This descriptor permits no download or implicit split fallback. An unused train
    # key is required by Ultralytics even though this notebook only calls val().
    descriptor = {"path": str(root), "train": str(paths), "val": str(paths),
                  "names": class_names, "nc": len(class_names)}
    if split == "test":
        descriptor["test"] = str(paths)
    evaluation_yaml = folder / f"evaluation_{mode}_data.yaml"
    evaluation_yaml.write_text(yaml.safe_dump(descriptor, sort_keys=False, allow_unicode=True), encoding="utf-8")
    manifest = {"mode": mode, "split": split, "seed": seed, "image_count": len(images),
                "full_val_image_count": len(val_images), "fingerprint": fingerprint,
                "full_val_fingerprint": full_val_fingerprint, "provenance_status": provenance_status,
                "provenance_basis": provenance_basis,
                "dataset_id": POLTRADA_V3_ID if official_mount else dataset_id,
                "source_yaml": str(source), "source_yaml_sha256": sha256_file(source),
                "evaluation_yaml_sha256": sha256_file(evaluation_yaml), "class_names": class_names,
                "test_content_disjoint_from_train_val": True if mode == "test" else None}
    manifest_path = folder / f"evaluation_{mode}_manifest.json"
    write_json(manifest_path, {**manifest, "files": records})
    return {**manifest, "manifest_path": str(manifest_path), "yaml": str(evaluation_yaml),
            "info": descriptor, "val_paths": [str(paths)], "image_files": [str(image) for image in images]}


def gflops_validation_arguments(dataset, output_dir, label):
    """Preserve notebook 03 settings and explicitly select the prepared split."""
    arguments = validation_arguments(dataset["yaml"], output_dir, label)
    arguments.update({"split": dataset["split"], "fraction": 1.0})
    return arguments


@contextmanager
def gflops_disable_label_cache():
    """Force label parsing from the fingerprinted files without changing the input dataset."""
    from ultralytics.data import dataset as sdk_dataset

    def set_in_memory_version(_prefix, _path, records, version):
        # get_labels() expects the writer's in-memory version field.
        records["version"] = version

    with (patch.object(sdk_dataset, "load_dataset_cache_file", side_effect=FileNotFoundError),
          patch.object(sdk_dataset, "save_dataset_cache_file", side_effect=set_in_memory_version)):
        yield


@contextmanager
def gflops_require_complete_nms():
    """Fail validation if the SDK stops processing a batch because of its NMS time limit."""
    from ultralytics.utils import LOGGER

    timeouts = []

    class TimeoutHandler(logging.Handler):
        def emit(self, record):
            message = record.getMessage()
            if "NMS time limit" in message and "exceeded" in message:
                timeouts.append(message)

    handler = TimeoutHandler()
    LOGGER.addHandler(handler)
    try:
        yield
    finally:
        LOGGER.removeHandler(handler)
    if timeouts:
        raise RuntimeError(f"Validasi parsial karena NMS timeout: {timeouts[0]}")


def gflops_validation_metadata(validator, arguments, dataset):
    """Verify the actual SDK backend and loaded images before exporting mAP."""
    if validator is None or getattr(validator, "args", None) is None:
        raise ValueError("Validator SDK aktual tidak terekam.")
    fields = ("split", "imgsz", "batch", "workers", "device", "conf", "iou", "max_det",
              "quantize", "nms", "augment", "plots", "fraction", "rect", "data")
    effective = {field: getattr(validator.args, field, None) for field in fields}
    effective["device"] = str(validator.device)
    effective["data"] = str(effective["data"])
    effective["end2end"] = getattr(validator, "end2end", None)
    if (effective["split"] != dataset["split"] or effective["end2end"] is not False
            or effective["nms"] is not None or effective["quantize"] is not None):
        raise ValueError("Validator SDK tidak memakai split, FP32, atau mode one-to-many dengan NMS yang ditetapkan.")
    loaded = getattr(getattr(validator.dataloader, "dataset", None), "im_files", ())
    if sorted(str(Path(path).resolve()) for path in loaded) != sorted(dataset["image_files"]):
        raise ValueError("Daftar gambar validator berbeda dari manifest evaluasi.")
    effective["dataloader_batch"] = validator.dataloader.batch_size
    if effective["batch"] != arguments["batch"] or effective["dataloader_batch"] != min(arguments["batch"], len(loaded)):
        raise ValueError("Batch validator aktual berbeda dari protokol (atau batas ukuran subset).")
    return {"split": dataset["split"], "evaluation_mode": dataset["mode"],
            "dataset_fingerprint": dataset["fingerprint"],
            "dataset_provenance_status": dataset["provenance_status"],
            "validation_image_count": len(loaded), "validation_arguments_requested": arguments,
            "validation_arguments_effective": effective}


def _gflops_complete_student_row(row):
    if (row.get("evaluation_mode") != "full_val" or row.get("dataset_provenance_status") != "VERIFIED"
            or any(row.get(key) != "PASS" for key in ("structural_status", "reload_status", "validation_status"))
            or row.get("latency_status") not in {"PASS", "MEASURED"}):
        return False
    fields = ("target_speedup", "gflops_speedup", "map50_95", "baseline_map50_95", "gflops",
              "parameters", "latency_median_ms")
    if any(not isinstance(row.get(key), (float, int)) or isinstance(row[key], bool)
           or not math.isfinite(row[key]) for key in fields):
        return False
    return (row.get("model_variant") in {"n", "s"} and row["target_speedup"] in {1.02, 1.05, 1.10, 1.20, 1.70}
            and row.get("target_status", "REACHED") == "REACHED"
            and row.get("baseline_validation_parity_status", "PASS") == "PASS"
            and row["gflops_speedup"] >= row["target_speedup"]
            and 0 <= row["map50_95"] <= 1 and 0 <= row["baseline_map50_95"] <= 1
            and row["baseline_map50_95"] - row["map50_95"] <= 0.02 + 1e-12
            and all(row[key] > 0 for key in ("gflops", "parameters", "latency_median_ms")))


def gflops_select_student(rows, mode):
    """Select one comparable full-validation student from both variants."""
    if mode != "full_val":
        return {"status": "NOT_RUN", "reason": "Student dipilih hanya dari full_val.", "selected": None}
    eligible = [row for row in rows if _gflops_complete_student_row(row)]
    if not eligible:
        return {"status": "NO_QUALIFYING_STUDENT", "selected": None}
    fingerprints = {row.get("dataset_fingerprint") for row in eligible}
    if len(fingerprints) != 1 or not next(iter(fingerprints)):
        raise ValueError("Kandidat student berasal dari fingerprint full val berbeda atau tidak tercatat.")
    best = min(eligible, key=lambda row: (-row["target_speedup"], -row["map50_95"], row["latency_median_ms"],
                                          row["parameters"], row["gflops"], row["model_variant"],
                                          row.get("checkpoint_sha256") or ""))
    return {"status": "SELECTED", "selected": dict(best)}


def _gflops_verified_checkpoint(path, expected):
    checkpoint = Path(path).resolve()
    if not checkpoint.is_file() or sha256_file(checkpoint) != expected:
        raise ValueError(f"Checkpoint berbeda dari SHA-256 yang dikunci: {checkpoint}")
    return checkpoint


def gflops_select_candidate_files(paths):
    """Combine independent N/S full-val exports under the same measurement contract."""
    files = [Path(path).resolve() for path in paths]
    if len(files) != len(set(files)):
        raise ValueError("File kandidat yang sama tidak boleh digabung dua kali.")
    variants, fingerprints, contracts, rows = set(), set(), [], []
    hashes = {str(path): sha256_file(path) for path in files}
    for path in files:
        record = json.loads(path.read_text(encoding="utf-8"))
        if (record.get("format_version") not in {1, 2} or record.get("mode") != "full_val"
                or record.get("status") not in {"PASS", "PARTIAL"}
                or record.get("evaluation_status") not in {"PASS", "PARTIAL"}
                or record.get("dataset_provenance_status") != "VERIFIED"):
            raise ValueError("Gabungan student memerlukan ekspor full_val VERIFIED yang selesai PASS.")
        members = set(record.get("model_variants", []))
        if not members or not members <= {"n", "s"} or variants.intersection(members):
            raise ValueError("Setiap varian harus mempunyai tepat satu sumber full_val.")
        variants.update(members)
        fingerprints.add(record.get("dataset_fingerprint"))
        contract = record.get("comparison_contract")
        if not isinstance(contract, dict) or not contract:
            raise ValueError("Kontrak evaluasi dan perangkat latency tidak tercatat.")
        contracts.append(contract)
        for source in record.get("rows", []):
            if source.get("model_variant") not in members:
                raise ValueError("Varian baris kandidat berbeda dari deklarasi ekspor.")
            row = dict(source)
            for key in ("checkpoint", "baseline_checkpoint"):
                relative = row.get(key + "_relative_path")
                if relative:
                    relocated = (path.parent / relative).resolve()
                    if not relocated.is_relative_to(path.parent):
                        raise ValueError("Path checkpoint kandidat keluar dari folder ekspor.")
                    row[key + "_path"] = str(relocated)
            row["selection_source_sha256"] = hashes
            rows.append(row)
    if len(fingerprints) > 1 or (fingerprints and not next(iter(fingerprints))):
        raise ValueError("Fingerprint full validation N/S berbeda atau kosong.")
    if contracts and any(contract != contracts[0] for contract in contracts[1:]):
        raise ValueError("Protokol, implementasi, atau perangkat latency N/S berbeda.")
    if variants != {"n", "s"}:
        return {"status": "NOT_RUN", "selected": None,
                "reason": "Pemilihan akhir menunggu hasil full_val dari N dan S.",
                "available_variants": sorted(variants)}
    result = gflops_select_student(rows, "full_val")
    if result.get("selected"):
        selected = result["selected"]
        _gflops_verified_checkpoint(selected["checkpoint_path"], selected["checkpoint_sha256"])
        _gflops_verified_checkpoint(selected["baseline_checkpoint_path"], selected["baseline_checkpoint_sha256"])
    return {**result, "source_files": hashes, "model_variants": sorted(variants)}


def gflops_lock_student(selection, path):
    """Write an immutable-by-convention full-val lock, verified again before test."""
    if selection.get("status") != "SELECTED":
        return None
    student = selection["selected"]
    if not _gflops_complete_student_row(student) or not student.get("dataset_fingerprint"):
        raise ValueError("Lock student memerlukan hasil full val yang lolos seleksi.")
    _gflops_verified_checkpoint(student["checkpoint_path"], student["checkpoint_sha256"])
    _gflops_verified_checkpoint(student["baseline_checkpoint_path"], student["baseline_checkpoint_sha256"])
    lock = Path(path)
    record = {"format_version": 1, "status": "LOCKED", "selected": student}
    if lock.exists():
        if json.loads(lock.read_text(encoding="utf-8")) != record:
            raise FileExistsError(f"Lock student sudah berisi pilihan berbeda: {lock}")
    else:
        # Exclusive creation prevents another run replacing the frozen selection.
        with lock.open("x", encoding="utf-8") as stream:
            json.dump(record, stream, ensure_ascii=False, indent=2, allow_nan=False)
    return record


def gflops_run_locked_test(lock_path, dataset, output_dir, *, checkpoint_path=None, baseline_checkpoint_path=None):
    """Evaluate the fixed baseline/student pair only on an independent test split."""
    if dataset is None:
        return {"status": "NOT_RUN", "reason": "Split test/train independen belum tersedia."}
    if (dataset.get("mode") != "test" or dataset.get("split") != "test"
            or dataset.get("provenance_status") != "VERIFIED"
            or dataset.get("test_content_disjoint_from_train_val") is not True):
        raise ValueError("Test memerlukan split independen yang provenance-nya VERIFIED.")
    record = json.loads(Path(lock_path).read_text(encoding="utf-8"))
    student = record.get("selected", {})
    if (record.get("status") != "LOCKED" or record.get("format_version") != 1
            or not _gflops_complete_student_row(student)
            or student["dataset_fingerprint"] != dataset.get("full_val_fingerprint")):
        raise ValueError("Lock student tidak cocok dengan full val Poltrada yang telah diverifikasi.")
    baseline = _gflops_verified_checkpoint(baseline_checkpoint_path or student["baseline_checkpoint_path"],
                                           student["baseline_checkpoint_sha256"])
    pruned = _gflops_verified_checkpoint(checkpoint_path or student["checkpoint_path"], student["checkpoint_sha256"])
    before = evaluate_checkpoint(baseline, dataset, output_dir, "baseline_test")
    after = evaluate_checkpoint(pruned, dataset, output_dir, "student_test")
    return {"status": "PASS", "baseline": before, "student": after,
            "delta_map50_95_pp": 100 * (after["map50_95"] - before["map50_95"]),
            "test_fingerprint": dataset["fingerprint"], "checkpoint_sha256": student["checkpoint_sha256"]}

In [ ]:
"""Cross-configuration analysis for notebook 05, following notebook 03's EDA.

Consumes completed run records and their original inventories. It never prunes,
validates, selects a student, or supplies a value for a missing measurement.
"""
# ruff: noqa: F821 -- Shared EDA and evaluation declarations precede this cell.

import pandas as pd


def _gflops_eda_csv(path):
    if not path.is_file() or not path.stat().st_size or not path.read_text(encoding="utf-8").strip():
        return []
    frame = pd.read_csv(path)
    return frame.astype(object).where(pd.notna(frame), None).to_dict("records")


def _gflops_eda_delta(after, before, scale=1):
    return scale * (after - before) if _eda_finite(after) and _eda_finite(before) else None


def gflops_eda_tables(bundle, suite_dir):
    """Join classes by ID and keep model baselines, denominators and statuses explicit."""
    tables = {key: [] for key in (
        "metrics", "sources", "architecture", "coverage", "layers", "stages", "groups", "ledger",
        "per_class_metrics", "validation_protocol", "verification", "target_tiers", "target_search",
        "search_profiles", "p5_contrasts", "p5_per_class", "student_eligibility",
    )}
    for model in bundle.get("models", []):
        identity = {"model": model["model_name"], "model_variant": model["model_variant"]}
        folder = _EdaPath(suite_dir) / model["model_name"].lower()
        baseline_report = model.get("baseline_report", {})
        baseline = baseline_report.get("reserialized", model.get("baseline", {}))
        baseline_val = baseline.get("validation", {}) if baseline.get("validation_status") == "PASS" else {}
        baseline_classes = {row["class_id"]: row for row in baseline_val.get("per_class", [])}
        source = model.get("source", {})
        tables["sources"].append({**identity, **source,
                                   "baseline_validation_parity_status": baseline_report.get("validation_parity_status", "NOT_RUN")})
        for row in baseline_report.get("architecture", {}).get("stages", []):
            tables["architecture"].append({**identity, **row})
        records = {row["configuration_id"]: row for row in model.get("rows", [])}
        processed = set()
        stages = [(f"baseline-{kind}", "baseline", baseline_report.get(kind, {}), None)
                  for kind in ("source", "reserialized")]
        stages.extend((run["configuration_id"], "pruned", run.get("pruned", {}), run)
                      for run in model.get("results", []))
        for config_id, stage, metric, run in stages:
            if not metric and run is None:
                continue
            processed.add(config_id)
            audit = (run or {}).get("audit", {})
            info = {**identity, "configuration_id": config_id, "stage": stage,
                    "policy": run["policy"] if run else "baseline", "ratio": run.get("ratio") if run else 0}
            validation_status = run.get("validation_status", "NOT_RUN") if run else metric.get("validation_status", "NOT_RUN")
            val = metric.get("validation", {}) if validation_status == "PASS" else {}
            latency = metric.get("latency", {})
            row = {**records.get(config_id, {}), **info,
                   **{key: metric.get(key) for key in ("parameters", "gflops", "checkpoint_bytes")},
                   **{key: val.get(key) for key in ("map50_95", "map50", "precision", "recall")},
                   "baseline_map50_95": baseline_val.get("map50_95"), "validation_status": validation_status,
                   "evaluation_mode": val.get("evaluation_mode", bundle.get("mode")),
                   "dataset_fingerprint": val.get("dataset_fingerprint"),
                   "dataset_provenance_status": val.get("dataset_provenance_status"),
                   "validation_image_count": val.get("validation_image_count"),
                   "latency_status": latency.get("status", "NOT_RUN"),
                   "latency_median_ms": latency.get("median_ms") if latency.get("status") in {"PASS", "MEASURED"} else None,
                   "latency_p95_ms": latency.get("p95_ms") if latency.get("status") in {"PASS", "MEASURED"} else None,
                   "gflops_speedup": baseline.get("gflops") / metric["gflops"]
                   if _eda_finite(metric.get("gflops")) and metric["gflops"] > 0 and baseline.get("gflops") else None,
                   "latency_speedup": baseline["latency"]["median_ms"] / latency["median_ms"]
                   if latency.get("status") in {"PASS", "MEASURED"} and latency.get("median_ms", 0) > 0
                   and baseline.get("latency", {}).get("status") in {"PASS", "MEASURED"}
                   and baseline["latency"].get("median_ms") else None,
                   "delta_map50_95_pp": _gflops_eda_delta(val.get("map50_95"), baseline_val.get("map50_95"), 100),
                   "pruned_group_count": len((run or {}).get("groups", [])) if run else 0}
            if run is None:
                row.update(structural_status="BASELINE", actual_eligible_ratio=0, eda_status="NOT_RUN")
                reload = baseline_report.get("reload", {})
                row["reload_status"] = ("PASS" if config_id == "baseline-reserialized"
                                        and reload.get("direct_yolo_load") is True and reload.get("raw_parity") is True
                                        else row.get("reload_status", "NOT_RUN"))
            for key, label in (("parameters", "parameter"), ("gflops", "gflops"), ("checkpoint_bytes", "checkpoint")):
                delta = _gflops_eda_delta(metric.get(key), baseline.get(key), -100)
                row[label + "_reduction_pct"] = delta / baseline[key] if delta is not None and baseline.get(key) else None
            delta = _gflops_eda_delta(val.get("map50_95"), baseline_val.get("map50_95"), 100)
            row["relative_map50_95_pct"] = delta / baseline_val["map50_95"] if delta is not None and baseline_val.get("map50_95") else None
            tables["metrics"].append(row)
            protocol = {key: val.get(key) for key in (
                "split", "evaluation_mode", "dataset_fingerprint", "dataset_provenance_status", "validation_image_count",
            )}
            tables["validation_protocol"].append({**info, "validation_status": validation_status, **protocol,
                                                   **val.get("validation_arguments_effective", {})})
            # The reserialized baseline is the one common reference for all pruned students.
            if config_id != "baseline-source":
                for entry in val.get("per_class", []):
                    prior = baseline_classes.get(entry["class_id"], {})
                    if prior and prior.get("name") != entry.get("name"):
                        raise ValueError(f"Nama/ID kelas tidak konsisten pada {model['model_name']} / {config_id}.")
                    tables["per_class_metrics"].append({**info, **entry,
                        "validation_status": validation_status, "evaluation_mode": row["evaluation_mode"],
                        "baseline_ap50_95": prior.get("ap50_95"), "baseline_ap50": prior.get("ap50"),
                        **{f"delta_{key}_pp": _gflops_eda_delta(entry.get(key), prior.get(key), 100)
                           for key in ("ap50_95", "ap50", "precision", "recall", "f1")}})
            if run is None:
                continue
            eda_dir = folder / run.get("artifact_configuration_id", config_id) / "eda"
            layers = _gflops_eda_csv(eda_dir / "layer_eda.csv")
            convs = [layer for layer in layers if layer["module_type"] == "Conv2d"]
            denominator = audit.get("target_policy_eligible_channels", audit.get("independent_channels_before"))
            removed = audit.get("independent_channels_removed")
            coverage = {**info, "graph_eligible_groups": len(audit.get("eligible_groups", [])),
                        "graph_rejected_roots": len(audit.get("rejected_groups", [])),
                        "selected_groups": len(run.get("groups", [])),
                        "graph_independent_channels_before": audit.get("independent_channels_before"),
                        "policy_eligible_channels": denominator, "independent_channels_removed": removed,
                        "policy_eligible_channels_after": denominator - removed
                        if _eda_finite(denominator) and _eda_finite(removed) else None,
                        "actual_eligible_ratio": run.get("actual_eligible_channel_ratio"),
                        "physical_conv_outputs_removed": audit.get("physical_conv_output_channels_removed"),
                        "inventory_status": "MEASURED" if layers else "UNAVAILABLE",
                        "conv_layers": len(convs) if layers else None,
                        "changed_conv_outputs": sum(layer["out_channels_removed"] > 0 for layer in convs) if layers else None}
            tables["coverage"].append(coverage)
            tables["layers"].extend({**info, **layer} for layer in layers)
            tables["stages"].extend({**info, **item} for item in _gflops_eda_csv(eda_dir / "stage_eda.csv"))
            tables["ledger"].extend({**info, **item} for item in audit.get("channel_mapping", []))
            scores = {}
            for item in audit.get("lamp_scores", []):
                scores.setdefault(item["root"], []).append(item)
            for entry in audit.get("eligible_groups", []):
                values = scores.get(entry["name"], [])
                finite = sorted(item["score"] for item in values if _eda_finite(item.get("score")))
                tables["groups"].append({**info, "root": entry["name"], "group_id": entry["group_id"],
                    "decision": "GRAPH_ELIGIBLE", "initial_channels": entry["initial_channels"],
                    "capacity_eligible": sum(bool(item.get("capacity_eligible")) for item in values),
                    "removed": sum(bool(item.get("removed")) for item in values),
                    "score_min": finite[0] if finite else None,
                    "score_median": float(_eda_np.median(finite)) if finite else None,
                    "score_max": finite[-1] if finite else None,
                    "coupled_output_layers": entry.get("signature"), "reason": ""})
            tables["groups"].extend({**info, **entry, "decision": "GRAPH_REJECTED"}
                                    for entry in audit.get("rejected_groups", []))
            tables["verification"].append({**info,
                "structural_status": run.get("structural_smoke_status", "NOT_RUN"),
                "oracle_status": run.get("oracle", {}).get("status", "NOT_RUN"),
                "reproducibility_status": run.get("reproducibility_status", "NOT_RUN"),
                "reload_status": run.get("reload_status", "NOT_RUN"), "validation_status": validation_status,
                "baseline_validation_parity_status": baseline_report.get("validation_parity_status", "NOT_RUN"),
                "latency_status": row["latency_status"], "eda_status": run.get("eda_status", "NOT_RUN"),
                "checkpoint_sha256": run.get("saved_pruned", {}).get("sha256")})
            tables["student_eligibility"].append({**info,
                "target_speedup": row.get("target_speedup"), "target_status": row.get("target_status"),
                "gflops_speedup": row.get("gflops_speedup"), "delta_map50_95_pp": row["delta_map50_95_pp"],
                "evaluation_mode": row["evaluation_mode"], "dataset_provenance_status": row["dataset_provenance_status"],
                "latency_status": row["latency_status"], "qualifies": _gflops_complete_student_row(row)})
        tables["metrics"].extend({**row, "stage": "pruned"} for config_id, row in records.items() if config_id not in processed)
        search = model.get("target_search", {})
        for key, source_key in (("target_tiers", "tiers"), ("search_profiles", "measurements"), ("target_search", "targets")):
            tables[key].extend({**identity, **{k: v for k, v in item.items() if k not in {"plan", "caps"}}}
                               for item in search.get(source_key, []))
        for key in ("p5_contrasts", "p5_per_class"):
            tables[key].extend({**identity, **entry} for entry in _gflops_eda_csv(folder / f"{key}.csv"))
    return tables


def _gflops_eda_finish(fig, path, count, reason="Pengukuran berpasangan belum tersedia."):
    files = _eda_save_figure(fig, path)
    return {"plot": path.name, "status": "MEASURED" if count else "NOT_RUN",
            "measured_values": count, "reason": None if count else reason, **files}


def _gflops_eda_panel(ax, rows, x, y, xlabel, ylabel):
    valid = [row for row in rows if _eda_finite(row.get(x)) and _eda_finite(row.get(y))]
    if valid:
        ax.plot([row[x] for row in valid], [row[y] for row in valid], "o-")
    else:
        ax.text(.5, .5, "NOT_RUN", ha="center", transform=ax.transAxes)
    ax.set(xlabel=xlabel, ylabel=ylabel)
    ax.grid(alpha=.2)
    return len(valid)


def gflops_eda_model_plots(tables, model_name, folder):
    """Notebook 03 sweep/AP plots, with computational targets shown separately."""
    folder = _EdaPath(folder)
    rows = [row for row in tables["metrics"] if row["model"] == model_name and row["configuration_id"] != "baseline-source"]
    protocol = next((row for row in tables["validation_protocol"] if row["model"] == model_name
                     and row.get("validation_status") == "PASS"), {})
    scope = " · " + str(protocol.get("evaluation_mode", "NOT_RUN"))
    if protocol.get("validation_image_count") is not None:
        scope += f" · {protocol['validation_image_count']} citra"
    if protocol.get("imgsz") is not None:
        scope += f" · {protocol['imgsz']} px"
    statuses = []
    controls = sorted((row for row in rows if row["policy"] in {"baseline", "extended"}), key=lambda row: row["ratio"])
    fig, axes = _eda_plt.subplots(1, 3, figsize=(17, 5))
    count = sum(_gflops_eda_panel(ax, controls, "ratio", metric, "Extended eligible-channel ratio", label)
                for ax, metric, label in zip(axes, ("map50_95", "parameter_reduction_pct", "gflops_reduction_pct"),
                                            ("mAP50–95", "Parameter reduction (%)", "GFLOPs reduction (%)")))
    fig.suptitle(model_name + scope + "\nKontrol extended 10–50%; target GFLOPs dianalisis terpisah")
    fig.tight_layout()
    statuses.append(_gflops_eda_finish(fig, folder / "extended_sweep", count))

    fig, axes = _eda_plt.subplots(2, 2, figsize=(17, 10))
    count = 0
    for ax, metric in zip(axes.flat, ("map50_95", "map50", "precision", "recall")):
        measured = [(index, row[metric]) for index, row in enumerate(rows) if _eda_finite(row.get(metric))]
        if measured:
            ax.scatter(*zip(*measured))
        else:
            ax.text(.5, .5, "NOT_RUN", ha="center", transform=ax.transAxes)
        ax.set_xticks(range(len(rows)), [row["configuration_id"] for row in rows], rotation=65, ha="right", fontsize=8)
        ax.set(ylabel=metric, ylim=(0, 1))
        ax.grid(axis="y", alpha=.2)
        if metric == "map50_95" and rows and _eda_finite(rows[0].get("baseline_map50_95")):
            ax.axhline(rows[0]["baseline_map50_95"] - .02, color="red", linestyle="--", label="baseline − 2 pp")
            ax.legend()
        count += len(measured)
    fig.suptitle(model_name + scope + "\nMetrik setiap konfigurasi")
    fig.tight_layout()
    statuses.append(_gflops_eda_finish(fig, folder / "configuration_metrics", count))

    classes = [row for row in tables["per_class_metrics"] if row["model"] == model_name]
    class_names = dict(sorted({row["class_id"]: row["name"] for row in classes}.items()))
    configs = [row["configuration_id"] for row in rows]
    fig, axes = _eda_plt.subplots(1, 2, figsize=(max(14, len(class_names) * 2.5), max(5, len(configs) * .45)))
    count = 0
    for ax, metric, label in zip(axes, ("ap50_95", "delta_ap50_95_pp"), ("AP50–95", "Delta AP50–95 vs baseline (pp)")):
        values = {(row["configuration_id"], row["class_id"]): row.get(metric) for row in classes}
        data = _eda_np.array([[values.get((config, class_id)) if _eda_finite(values.get((config, class_id))) else _eda_np.nan
                              for class_id in class_names] for config in configs], dtype=float)
        measured = int(_eda_np.isfinite(data).sum())
        if measured:
            options = {"vmin": 0, "vmax": 1, "cmap": "Blues"} if metric == "ap50_95" else {
                "vmin": -max(1, float(_eda_np.nanmax(_eda_np.abs(data)))),
                "vmax": max(1, float(_eda_np.nanmax(_eda_np.abs(data)))), "cmap": "RdBu"}
            plot = ax.imshow(_eda_np.ma.masked_invalid(data), aspect="auto", **options)
            fig.colorbar(plot, ax=ax)
            for i, j in zip(*_eda_np.where(_eda_np.isfinite(data))):
                ax.text(j, i, f"{data[i, j]:.3f}" if metric == "ap50_95" else f"{data[i, j]:+.2f}",
                        ha="center", va="center", fontsize=7)
        else:
            ax.text(.5, .5, "NOT_RUN", ha="center", transform=ax.transAxes)
        ax.set_xticks(range(len(class_names)), list(class_names.values()), rotation=45, ha="right")
        ax.set_yticks(range(len(configs)), configs)
        ax.set_title(label + " · sel kosong = tidak tersedia")
        count += measured
    fig.suptitle(model_name + scope + "\nAP per kelas untuk seluruh konfigurasi")
    fig.tight_layout()
    statuses.append(_gflops_eda_finish(fig, folder / "per_class_ap_all", count))

    for metric, label in (("parameters", "Parameters"), ("gflops", "GFLOPs"), ("latency_median_ms", "Median latency raw forward (ms)")):
        fig, ax = _eda_plt.subplots(figsize=(11, 6))
        valid = [row for row in rows if _eda_finite(row.get(metric)) and _eda_finite(row.get("map50_95"))]
        for row in valid:
            ax.scatter(row[metric], row["map50_95"])
            ax.annotate(row["configuration_id"], (row[metric], row["map50_95"]), xytext=(4, 4), textcoords="offset points", fontsize=8)
        if not valid:
            ax.text(.5, .5, "NOT_RUN", ha="center", transform=ax.transAxes)
        ax.set(xlabel=label, ylabel="mAP50–95", title=model_name + scope + "\nAkurasi dan efisiensi")
        ax.grid(alpha=.2)
        fig.tight_layout()
        statuses.append(_gflops_eda_finish(fig, folder / ("tradeoff_" + metric), len(valid)))

    profiles = [row for row in tables["search_profiles"] if row["model"] == model_name]
    tiers = [row for row in tables["target_tiers"] if row["model"] == model_name]
    fig, axes = _eda_plt.subplots(1, 2, figsize=(15, 5))
    for tier in sorted({row["tier"] for row in profiles}):
        values = sorted((row for row in profiles if row["tier"] == tier), key=lambda row: row["channels"])
        axes[0].plot([row["channels"] for row in values], [row["speed_up"] for row in values], ".-", label=f"tier {tier}")
    if profiles:
        axes[0].legend()
    else:
        axes[0].text(.5, .5, "NOT_RUN", ha="center", transform=axes[0].transAxes)
    tiers = [row for row in tiers if _eda_finite(row.get("max_gflops_speed_up"))]
    if tiers:
        axes[1].bar([str(row["tier"]) for row in tiers], [row["max_gflops_speed_up"] for row in tiers])
        for target in sorted({row["target"] for row in tables["target_search"] if row["model"] == model_name}):
            axes[1].axhline(target, linestyle="--", label=f"target {target:g}×")
        axes[1].legend()
    else:
        axes[1].text(.5, .5, "NOT_RUN", ha="center", transform=axes[1].transAxes)
    axes[0].set(xlabel="Channel independen dihapus", ylabel="GFLOPs speed-up", title="Kandidat pencarian, tanpa evaluasi mAP")
    axes[1].set(xlabel="Tier cap", ylabel="Maksimum GFLOPs speed-up", title="Kapasitas dengan proteksi aktif")
    fig.suptitle(model_name + scope + "\nPencarian target komputasi")
    fig.tight_layout()
    statuses.append(_gflops_eda_finish(fig, folder / "target_search_capacity", len(profiles) + len(tiers)))
    return statuses


def _gflops_build_analysis(bundle, suite_dir):
    """Export analysis for every completed configuration independently of detail display."""
    folder = _EdaPath(suite_dir) / "comparison/analysis"
    folder.mkdir(parents=True, exist_ok=True)
    tables = gflops_eda_tables(bundle, suite_dir)
    for key, rows in tables.items():
        _eda_write_csv(folder / f"{key}.csv", rows)
        _eda_write_json(folder / f"{key}.json", rows)
    plots = []
    for model in bundle.get("models", []):
        plots.extend({"model": model["model_name"], **row}
                     for row in gflops_eda_model_plots(tables, model["model_name"], folder / model["model_name"].lower()))
    report = {"status": "PASS" if bundle.get("models") else "NOT_RUN", "mode": bundle.get("mode"),
              "table_rows": {key: len(rows) for key, rows in tables.items()}, "plots": plots,
              "interpretation": "Smoke checks the pipeline. Full validation selects a student. Layer distributions are descriptive.",
              "selection": bundle.get("selection"),
              "display_configurations": list(globals().get("EDA_CONFIGURATIONS", ())),
              "detail_configurations": list(globals().get("EDA_DETAIL_CONFIGURATIONS", ())), "tables": tables}
    _eda_write_json(folder / "analysis_status.json", {key: value for key, value in report.items() if key != "tables"})
    return report


def gflops_build_analysis(bundle, suite_dir):
    path = _EdaPath(suite_dir) / "comparison/analysis/analysis_status.json"
    _eda_write_json(path, {"status": "RUNNING"})
    try:
        return _gflops_build_analysis(bundle, suite_dir)
    except Exception as error:
        _eda_write_json(path, {"status": "FAIL", "error": str(error), "type": type(error).__name__})
        raise

In [ ]:
"""Notebook 05 runtime, embedded after the shared LAMP helpers."""
# ruff: noqa: F821 -- Runtime globals are supplied by preceding notebook cells.
# ruff: noqa: BLE001 -- Independent jobs persist stage-specific failures before continuing.


def gflops_load_source(path, provenance, variant):
    if sha256_file(path) != SOURCE_CHECKPOINT_SHA256[variant] or provenance["sha256"] != SOURCE_CHECKPOINT_SHA256[variant]:
        raise ValueError(f"Checkpoint sumber {variant} tidak cocok dengan SHA-256 yang dikunci.")
    core = YOLO(str(path), task="detect").model.to(TARGET_DEVICE).float().eval()
    core.end2end = True
    architecture_audit(core, variant)
    return core


def gflops_config():
    if MODE not in {"smoke", "full_val", "test"}:
        raise ValueError("MODE harus smoke, full_val, atau test.")
    if len(MODEL_VARIANTS) != len(set(MODEL_VARIANTS)) or not set(MODEL_VARIANTS) <= {"n", "s"} or not MODEL_VARIANTS:
        raise ValueError("MODEL_VARIANTS harus memuat n dan/atau s tanpa duplikasi.")
    if NOTEBOOK_VARIANT not in {"n", "s"} or MODEL_VARIANTS != [NOTEBOOK_VARIANT]:
        raise ValueError("Setiap notebook hanya menjalankan varian yang sesuai judulnya.")
    if MODE == "test" and STUDENT_LOCK_PATH is None:
        raise ValueError("MODE test memerlukan STUDENT_LOCK_PATH.")
    return {**configuration(), "mode": MODE, "targets": GFLOPS_TARGETS, "dataset_id": DATASET_ID,
            "expected_dataset_fingerprint": EXPECTED_DATASET_FINGERPRINT, "source_sha256": SOURCE_CHECKPOINT_SHA256,
            "eda_configurations": list(EDA_CONFIGURATIONS), "eda_detail_configurations": list(EDA_DETAIL_CONFIGURATIONS),
            "notebook_variant": NOTEBOOK_VARIANT,
            "comparison_candidate_paths": [str(path) for path in COMPARISON_CANDIDATE_PATHS],
            "implementation_sha256": IMPLEMENTATION_SHA256,
            "platform": PLATFORM, "run_diagnostic_controls": RUN_DIAGNOSTIC_CONTROLS,
            "trace_mode": "unfused dual-head FP32", "validation_mode": "one-to-many NMS FP32",
            "latency_mode": "unfused dual-head raw forward FP32, batch 1", "training_epochs": 0}


def gflops_latency(core, image):
    if not RUN_LATENCY_BENCHMARK:
        return {"status": "NOT_RUN", "reason": "Benchmark dimatikan."}
    try:
        return cuda_latency(core, image, LATENCY_WARMUP, LATENCY_REPEATS)
    except Exception as error:
        return {"status": "FAIL", "error": safe_error(error)}


def gflops_validation(row, checkpoint, dataset, folder):
    if dataset is None:
        return
    try:
        row["validation"] = evaluate_checkpoint(checkpoint, dataset, folder, "validation")
        row["validation_status"] = "PASS"
    except Exception as error:
        row["validation_status"] = "FAIL"
        row["errors"].append({"stage": "validation", "error": safe_error(error)})


def gflops_baseline(path, provenance, variant, image, folder, dataset):
    folder.mkdir(parents=True, exist_ok=True)
    previous_path = folder / "baseline_report.json"
    previous = json.loads(previous_path.read_text(encoding="utf-8")) if previous_path.is_file() else {}
    core = gflops_load_source(path, provenance, variant)
    forward_snapshot(core, image)
    measurements = model_measurements(core, IMGSZ)
    latency = previous.get("reserialized", {}).get("latency", {})
    measurements["latency"] = latency if latency.get("status") in {"PASS", "MEASURED"} else gflops_latency(core, image)
    source = previous.get("source") or {**measurements, "checkpoint_bytes": Path(path).stat().st_size,
                                       "validation_status": "NOT_RUN", "errors": []}
    if source["validation_status"] != "PASS":
        gflops_validation(source, path, dataset, folder / "source")
    saved = workflow_save_checkpoint(core, folder / "baseline_fp32.pt", provenance)
    reload_report = previous.get("reload")
    if not reload_report:
        reload_report = verify_fresh_reload(saved["path"], core, image, folder, timeout=RELOAD_TIMEOUT_SECONDS)
    clone = gflops_reload(saved["path"], core)
    if model_state_sha256(core) != model_state_sha256(clone):
        raise ValueError("Baseline hasil reload mengubah state sumber.")
    baseline = previous.get("reserialized") or {**measurements, "checkpoint_bytes": saved["file_bytes"],
                                                "validation_status": "NOT_RUN", "errors": []}
    source["latency"] = baseline["latency"] = measurements["latency"]
    if baseline["validation_status"] != "PASS":
        gflops_validation(baseline, saved["path"], dataset, folder / "reserialized")
    parity = "NOT_RUN"
    if dataset is not None:
        fields = ("map50_95", "map50", "precision", "recall")
        parity = ("PASS" if source["validation_status"] == baseline["validation_status"] == "PASS"
                  and all(abs(source["validation"][key] - baseline["validation"][key]) <= 1e-6 for key in fields)
                  else "FAIL")
    result = {"source": source, "reserialized": baseline, "validation_parity_status": parity,
              "state_sha256": model_state_sha256(core), "checkpoint": saved, "reload": reload_report,
              "architecture": architecture_audit(core, variant)["report"]}
    write_json(previous_path, result)
    return result


def gflops_reload(path, reference):
    model = YOLO(str(path), task="detect").model.to(TARGET_DEVICE).float().eval()
    model.end2end = reference.end2end
    return model


def gflops_make_graph(core, image, variant, ratio):
    return build_policy_pruner(core, image, "extended", ratio, variant)


def gflops_plan_from_history(history):
    return {item["root"]: sorted(item["removed_original_indices"]) for item in history}


def gflops_result_row(result, baseline, dataset):
    after = result["pruned"]
    before = baseline["reserialized"]
    latency = after.get("latency", {})
    before_latency = before["latency"].get("median_ms")
    median = latency.get("median_ms")
    checkpoint = result.get("saved_pruned", {})
    validation = after.get("validation", {})
    return {
        "model": result["model_name"], "model_variant": result["model_variant"],
        "configuration_id": result["configuration_id"], "policy": result["policy"], "ratio": result["ratio"],
        "artifact_configuration_id": result.get("artifact_configuration_id", result["configuration_id"]),
        "actual_eligible_ratio": result.get("actual_eligible_channel_ratio"),
        "target_speedup": result.get("target_speedup"), "target_status": result.get("target_status"),
        "gflops_speedup": before["gflops"] / after["gflops"] if after.get("gflops") else None,
        "latency_speedup": before_latency / median if before_latency and median else None,
        **{key: after.get(key) for key in ("parameters", "gflops", "checkpoint_bytes")},
        **{key: validation.get(key) for key in ("map50_95", "map50", "precision", "recall")},
        "baseline_map50_95": (before.get("validation") or {}).get("map50_95"),
        "status": result.get("status", "PARTIAL"), "upload_status": result.get("upload_status", "NOT_RUN"),
        "raw_artifact_ref": result.get("raw_publication", {}).get("reference"),
        "raw_manifest_path": str(Path(checkpoint["path"]).parent / "raw_pruned_manifest.json") if checkpoint.get("path") else None,
        "latency_median_ms": median, "latency_p95_ms": latency.get("p95_ms"),
        "structural_status": result["structural_smoke_status"], "reload_status": result["reload_status"],
        "validation_status": result["validation_status"], "latency_status": latency.get("status", "NOT_RUN"),
        "eda_status": result["eda_status"], "baseline_validation_parity_status": baseline["validation_parity_status"],
        "checkpoint_path": checkpoint.get("path"), "checkpoint_sha256": checkpoint.get("sha256"),
        "baseline_checkpoint_path": baseline["checkpoint"]["path"],
        "baseline_checkpoint_sha256": baseline["checkpoint"]["sha256"],
        "evaluation_mode": MODE, "dataset_provenance_status": (dataset or {}).get("provenance_status"),
        "dataset_fingerprint": (dataset or {}).get("fingerprint"), **result["comparison"],
    }


def gflops_experiment(path, provenance, variant, image, baseline, dataset, folder, job):
    folder.mkdir(parents=True, exist_ok=True)
    workflow_bind(folder, "job_contract.json", {"id": job["id"], "policy": job["policy"],
        "plan": job.get("plan"), "caps": job.get("caps"), "ratio": job.get("ratio"),
        "source": provenance["sha256"], "baseline_state": baseline["state_sha256"]})
    report_path = folder / "pruning_manifest.json"
    ratio = job.get("ratio", 0.10)
    result = json.loads(report_path.read_text(encoding="utf-8")) if report_path.is_file() else {
        "format_version": 2, "stage": "raw_pruned", "configuration_id": job["id"],
        "model_name": f"YOLO26{variant}", "model_variant": variant,
        "policy": job["policy"], "ratio": ratio, "target_speedup": job.get("target"),
        "target_status": job.get("status"), "tier": job.get("tier"), "provenance": provenance,
        "baseline": baseline["reserialized"], "pruned": {}, "comparison": {}, "errors": [],
        "structural_smoke_status": "NOT_RUN", "reload_status": "NOT_RUN",
        "validation_status": "NOT_RUN", "eda_status": "NOT_RUN", "upload_status": "NOT_RUN"}
    write_json(report_path, result)
    stage = "structural_smoke"
    try:
        saved = result.get("saved_pruned")
        ready = result["structural_smoke_status"] == "PASS" and saved is not None
        if not ready:
            seed_environment(SEED)
            core = gflops_load_source(path, provenance, variant)
            if model_state_sha256(core) != baseline["state_sha256"]:
                raise ValueError("State kandidat sebelum pruning berbeda dari baseline.")
            original = deepcopy(core)
            protected = protected_policy_snapshot(core, variant)
            pruner, audit = gflops_make_graph(core, image, variant, ratio)
            inventory_before = measure_layer_inventory(core, image, audit, "extended")
            if job["policy"] == "extended":
                history = apply_policy_pruning(pruner, audit)
            else:
                history = apply_exact_gflops_plan(pruner, audit, job["plan"], caps=job.get("caps"))
            assert_layer_consistency(core)
            assert_policy_protected_unchanged(core, protected)
            forward_snapshot(core, image)
            result["oracle"] = verify_gflops_oracle(original, core, image, audit)
            del original
            selected = gflops_plan_from_history(history)
            repeat = gflops_load_source(path, provenance, variant)
            repeat_pruner, repeat_audit = gflops_make_graph(repeat, image, variant, ratio)
            if job["policy"] == "extended":
                replay = apply_policy_pruning(repeat_pruner, repeat_audit)
            else:
                replay = apply_exact_gflops_plan(repeat_pruner, repeat_audit, job["plan"], caps=job.get("caps"))
            if (gflops_plan_from_history(replay) != selected or model_state_sha256(core) != model_state_sha256(repeat)
                    or repeat_audit["report"]["channel_mapping"] != audit["report"]["channel_mapping"]):
                raise ValueError("Pengulangan tidak menghasilkan ledger dan state yang sama.")
            del repeat, repeat_pruner, repeat_audit
            result["state_sha256"] = model_state_sha256(core)
            result["reproducibility_status"] = "PASS"
            result["audit"] = audit["report"]
            result["groups"] = history
            result["pruned_group_count"] = len(history)
            result["selected_plan"] = selected
            result["actual_eligible_channel_ratio"] = sum(map(len, selected.values())) / audit["report"].get(
                "target_policy_eligible_channels", audit["independent_channels_before"])
            if job["policy"] != "extended":
                result["ratio"] = result["actual_eligible_channel_ratio"]
            write_json(folder / "channel_mapping.json", audit["report"]["channel_mapping"])
            write_json(folder / "channel_masks.json", audit["report"]["channel_masks"])
            write_json(folder / "dependency_audit.json", audit["report"])
            write_csv(folder / "lamp_scores.csv", audit["report"]["lamp_scores"])
            inventory_after = measure_layer_inventory(core, image, audit, "extended")
            write_json(folder / "inventories.json", {"before": inventory_before, "after": inventory_after})
            result["pruned"] = model_measurements(core, IMGSZ)
            if result["pruned"]["parameters"] >= result["baseline"]["parameters"]:
                raise ValueError("Parameter tidak berkurang.")
            if job.get("gflops") is not None and not math.isclose(result["pruned"]["gflops"], job["gflops"], rel_tol=1e-10):
                raise ValueError("GFLOPs kandidat berbeda dari profil pencarian.")
            result["structural_smoke_status"] = "PASS"
            stage = "save"
            saved = workflow_save_checkpoint(core, folder / "pruned_fp32.pt", provenance)
            result["saved_pruned"] = saved
            result["pruned"]["checkpoint_bytes"] = saved["file_bytes"]
            result["structure_signature"] = {
                "parameters": {name: list(value.shape) for name, value in core.named_parameters()},
                "conv_bn": {name: [module.in_channels, module.out_channels, module.groups]
                            if isinstance(module, torch.nn.Conv2d) else [module.num_features]
                            for name, module in core.named_modules()
                            if isinstance(module, (torch.nn.Conv2d, torch.nn.BatchNorm2d))}}
            write_json(report_path, result)
        else:
            checkpoint = folder / "pruned_fp32.pt"
            if sha256_file(checkpoint) != saved["sha256"]:
                raise ValueError("Raw checkpoint berubah; resume tidak boleh menimpanya.")
            saved["path"] = str(checkpoint.resolve())
            core = YOLO(str(checkpoint), task="detect").model.to(TARGET_DEVICE).float().eval()
            core.end2end = True
            if model_state_sha256(core) != result["state_sha256"]:
                raise ValueError("State raw resume berbeda dari ledger.")
            inventories = json.loads((folder / "inventories.json").read_text(encoding="utf-8"))
            inventory_before, inventory_after = inventories["before"], inventories["after"]
        stage = "reload"
        if result["reload_status"] != "PASS":
            result["reload"] = verify_fresh_reload(saved["path"], core, image, folder, timeout=RELOAD_TIMEOUT_SECONDS)
            reloaded = gflops_reload(saved["path"], core)
            if model_state_sha256(reloaded) != result["state_sha256"]:
                raise ValueError("State berubah setelah reload.")
            result["reload_status"] = "PASS"
            write_json(report_path, result)
        if result["validation_status"] != "PASS":
            stage = "validation"
            evaluated = {"validation_status": "NOT_RUN", "errors": result["errors"]}
            gflops_validation(evaluated, saved["path"], dataset, folder)
            result["validation_status"] = evaluated["validation_status"]
            if "validation" in evaluated:
                result["pruned"]["validation"] = evaluated["validation"]
            write_json(report_path, result)
        stage = "latency"
        if result["pruned"].get("latency", {}).get("status") not in {"PASS", "MEASURED"}:
            result["pruned"]["latency"] = gflops_latency(core, image)
        result["comparison"] = metric_comparison(result["baseline"], result["pruned"])
        write_json(report_path, result)
        stage = "eda"
        if result["eda_status"] != "PASS":
            try:
                result["eda"] = build_run_eda(result, folder, inventory_before, inventory_after)
                eda_status = result["eda"].get("status", "NOT_RUN")
                result["eda_status"] = "PASS" if eda_status == "MEASURED" else eda_status
            except Exception as error:
                result["eda_status"] = "FAIL"
                result["errors"].append({"stage": "eda", "error": safe_error(error)})
        audit_files = {name: sha256_file(folder / name) for name in ("dependency_audit.json", "inventories.json")
                       if (folder / name).is_file()}
        raw_manifest = {"format_version": 2, "stage": "raw_pruned", "model_variant": variant,
            "configuration_id": job["id"], "target_speedup": job.get("target"), "target_status": job.get("status"),
            "actual_gflops_speedup": baseline["reserialized"]["gflops"] / result["pruned"]["gflops"],
            "raw_checkpoint": {"relative_path": "pruned_fp32.pt", "sha256": saved["sha256"]},
            "baseline_checkpoint": {"relative_path": Path(os.path.relpath(baseline["checkpoint"]["path"], folder)).as_posix(),
                                    "artifact_path": "baseline_fp32.pt", "sha256": baseline["checkpoint"]["sha256"]},
            "state_sha256": result["state_sha256"], "structure_signature": result["structure_signature"],
            "source_baseline": provenance, "structural_status": result["structural_smoke_status"],
            "oracle_status": result.get("oracle", {}).get("status"), "reload_status": result["reload_status"],
            "evaluation_mode": MODE, "dataset": {key: (dataset or {}).get(key) for key in (
                "dataset_id", "fingerprint", "full_val_fingerprint", "train_fingerprint", "provenance_status",
                "class_names", "image_count", "split", "seed")},
            "baseline": baseline["reserialized"], "raw_pruned": result["pruned"], "audit_files": audit_files,
            "validation_status": result["validation_status"], "baseline_validation_parity_status": baseline["validation_parity_status"],
            "seed": SEED, "imgsz": IMGSZ, "platform": PLATFORM, "implementation_sha256": IMPLEMENTATION_SHA256}
        write_json(folder / "raw_pruned_manifest.json", raw_manifest)
        (folder / "RESTORE.md").write_text("Load native DetectionModel FP32 from pruned_fp32.pt; verify raw_pruned_manifest.json SHA-256. Do not rebuild from baseline YAML. Raw is the shared KD/fine-tuning initial state.", encoding="utf-8")
        stage = "upload"
        if WANDB_TRACKING:
            with workflow_tracking(folder, variant, job.get("target") or 1.0, source_ref=model_spec(variant)["artifact"]) as run:
                publication = workflow_publish(run, folder, "raw-pruned", {
                    "pruned_fp32.pt": saved["path"], "baseline_fp32.pt": baseline["checkpoint"]["path"],
                    "raw_pruned_manifest.json": folder / "raw_pruned_manifest.json",
                    **{name: folder / name for name in audit_files}},
                    metadata_info={"stage": "raw_pruned", "target_status": result["target_status"],
                                   "raw_sha256": saved["sha256"]}, aliases=["raw-pruned", "latest"])
                result["raw_publication"] = publication
                result["upload_status"] = "PASS" if publication["status"] == "CONFIRMED" else "NOT_RUN"
                write_json(report_path, result)
                row = gflops_result_row(result, baseline, dataset)
                run.log({"configuration": wandb.Table(dataframe=pd.DataFrame([row]))})
                workflow_publish(run, folder, "results", workflow_report_files(folder), kind="results")
    except Exception as error:
        result[stage + "_status"] = "FAIL"
        result["errors"].append({"stage": stage, "error": safe_error(error), "traceback": safe_error(traceback.format_exc())})
    statuses = [result.get(key, "NOT_RUN") for key in ("structural_smoke_status", "reload_status", "validation_status", "eda_status")]
    if job.get("status") == "TARGET_NOT_REACHED":
        statuses.append("PARTIAL")
    if RUN_LATENCY_BENCHMARK:
        statuses.append("PASS" if result["pruned"].get("latency", {}).get("status") in {"PASS", "MEASURED"} else "NOT_RUN")
    if WANDB_TRACKING:
        statuses.append(result["upload_status"])
    result["status"] = workflow_status(statuses, useful=bool(result.get("saved_pruned")))
    write_json(report_path, result)
    row = gflops_result_row(result, baseline, dataset)
    write_json(folder / "metrics.json", row)
    print(variant, job["id"], result["status"], row["structural_status"], row["validation_status"], flush=True)
    return result, row


def gflops_p5_comparison(results, folder):
    by_id = {result["configuration_id"]: result for result in results}
    summaries, classes = [], []
    for label, reference in (("p5-only", None), ("p5-protected", "extended-pr0.1")):
        if label not in by_id:
            continue
        result = by_id[label]
        before = by_id[reference]["pruned"] if reference else result["baseline"]
        after = result["pruned"]
        summaries.append({"contrast": label, "reference": reference or "baseline",
                          "reference_gflops": before.get("gflops"), "gflops": after.get("gflops"),
                          "validation_status": result["validation_status"],
                          **(metric_comparison(before, after) if before.get("checkpoint_bytes")
                             and after.get("checkpoint_bytes") else {})})
        base_classes = {row["class_id"]: row for row in before.get("validation", {}).get("per_class", [])}
        for row in after.get("validation", {}).get("per_class", []):
            prior = base_classes.get(row["class_id"], {})
            before_ap, after_ap = prior.get("ap50_95"), row.get("ap50_95")
            classes.append({"contrast": label, "reference": reference or "baseline", **row,
                            "reference_ap50_95": before_ap,
                            "delta_ap50_95_pp": 100 * (after_ap - before_ap)
                            if after_ap is not None and before_ap is not None else None})
    write_csv(folder / "p5_contrasts.csv", summaries)
    write_csv(folder / "p5_per_class.csv", classes)


def gflops_run_model(variant, folder, dataset):
    folder.mkdir(parents=True, exist_ok=True)
    manifest = folder / "model_manifest.json"
    suite = {"model_name": f"YOLO26{variant}", "model_variant": variant, "source": {},
             "baseline": {}, "baseline_report": {}, "results": [], "rows": [], "target_search": {},
             "status": "NOT_RUN", "errors": []}
    rows, results = suite["rows"], suite["results"]
    try:
        path, provenance = obtain_baseline(folder, variant)
        suite["source"] = provenance
        image = torch.rand(1, 3, IMGSZ, IMGSZ, generator=torch.Generator().manual_seed(SEED)).to(TARGET_DEVICE)
        baseline = gflops_baseline(path, provenance, variant, image, folder / "baseline", dataset)
        suite.update(baseline=baseline["reserialized"], baseline_report=baseline)
        for kind in ("source", "reserialized"):
            metrics = baseline[kind]
            rows.append({"model": f"YOLO26{variant}", "model_variant": variant,
                         "configuration_id": f"baseline-{kind}", "policy": "baseline", "ratio": 0,
                         "gflops_speedup": 1.0, "latency_speedup": 1.0 if metrics["latency"]["status"] == "PASS" else None,
                         **{key: metrics[key] for key in ("parameters", "gflops", "checkpoint_bytes")},
                         **{key: metrics.get("validation", {}).get(key) for key in ("map50_95", "map50", "precision", "recall")},
                         "latency_median_ms": metrics["latency"].get("median_ms"),
                         "latency_p95_ms": metrics["latency"].get("p95_ms"),
                         "structural_status": "PASS", "reload_status": "PASS" if kind == "reserialized" else "NOT_RUN",
                         "validation_status": metrics["validation_status"], "latency_status": metrics["latency"]["status"],
                         "eda_status": "NOT_RUN"})
        write_json(manifest, suite)

        def run(job):
            try:
                result, row = gflops_experiment(path, provenance, variant, image, baseline, dataset, folder / job["id"], job)
            except Exception as error:
                result = {"configuration_id": job["id"], "model_name": f"YOLO26{variant}", "model_variant": variant,
                          "policy": job["policy"], "ratio": job.get("ratio"), "target_speedup": job.get("target"),
                          "target_status": job.get("status"), "status": "FAIL", "baseline": baseline["reserialized"],
                          "pruned": {}, "comparison": {}, "structural_smoke_status": "FAIL", "reload_status": "NOT_RUN",
                          "validation_status": "NOT_RUN", "eda_status": "NOT_RUN", "errors": [{"error": safe_error(error)}]}
                row = gflops_result_row(result, baseline, dataset)
            results.append(result)
            rows.append(row)
            suite["status"] = "PARTIAL"
            write_json(manifest, suite)
            write_csv(folder / "metrics.csv", rows)
            import gc
            gc.collect()
            if TARGET_DEVICE.type == "cuda":
                torch.cuda.empty_cache()
            return result

        if RUN_DIAGNOSTIC_CONTROLS:
            controls = {ratio: run({"id": f"extended-pr{ratio:g}", "policy": "extended", "ratio": ratio})
                        for ratio in PRUNING_RATIOS}
        else:
            controls = {}
        graph_source = gflops_load_source(path, provenance, variant)
        _, graph_audit = gflops_make_graph(graph_source, image, variant, 0.10)
        if controls.get(0.10, {}).get("structural_smoke_status") == "PASS":
            try:
                for name, plan in p5_control_plans(graph_audit, controls[0.10]["groups"]).items():
                    run({"id": name, "policy": name, "plan": plan})
                gflops_p5_comparison(results, folder)
            except Exception as error:
                suite["errors"].append({"stage": "p5", "error": safe_error(error)})

        def profile(plan):
            probe = gflops_load_source(path, provenance, variant)
            pruner, audit = gflops_make_graph(probe, image, variant, 0.10)
            apply_exact_gflops_plan(pruner, audit, plan)
            return model_measurements(probe, IMGSZ)["gflops"]

        cache_path = folder / "profile_cache.json"
        cache = json.loads(cache_path.read_text(encoding="utf-8")) if cache_path.is_file() else {}
        def progress(values, points):
            write_json(cache_path, values)
            write_json(folder / "profile_progress.json", points)
        search = search_gflops_targets(graph_audit, baseline["reserialized"]["gflops"], profile,
                                      tuple(GFLOPS_TARGETS), profile_cache=cache, progress_callback=progress)
        suite["target_search"] = search
        write_json(folder / "target_search.json", search)
        write_json(manifest, suite)
        unique = {}
        # Feasibility and the requested grid are already recorded in target_search.json.
        # Run the original 1.70× diagnostic before other infeasible targets so
        # identical maximum plans reuse its checkpoint and recovery manifest.
        ordered = sorted(search["targets"], key=lambda row: (
            row["status"] == "TARGET_NOT_REACHED" and row["target"] != 1.70))
        for target in ordered:
            label = f"target-{target['target']:.2f}x"
            if target["status"] == "FAIL":
                failed = {"configuration_id": label, "model_name": f"YOLO26{variant}", "model_variant": variant,
                    "policy": "gflops-target", "ratio": None, "target_speedup": target["target"],
                    "target_status": "FAIL", "baseline": baseline["reserialized"], "pruned": {}, "comparison": {},
                    "structural_smoke_status": "FAIL", "reload_status": "NOT_RUN", "validation_status": "NOT_RUN",
                    "eda_status": "NOT_RUN", "status": "FAIL", "errors": [{"stage": "search", "error": target["error"]}]}
                results.append(failed)
                rows.append(gflops_result_row(failed, baseline, dataset))
                write_json(manifest, suite)
                continue
            key = gflops_plan_key(target["plan"])
            if key in unique:
                original = unique[key]
                alias = deepcopy(original)
                alias.update(configuration_id=label, target_speedup=target["target"], target_status=target["status"],
                             artifact_configuration_id=original["configuration_id"])
                if target["status"] != "REACHED" and alias["status"] == "PASS":
                    alias["status"] = "PARTIAL"
                results.append(alias)
                rows.append(gflops_result_row(alias, baseline, dataset))
                write_json(manifest, suite)
                continue
            result = run({**target, "id": label, "policy": "gflops-target", "ratio": .10})
            unique[key] = result
        # Preserve ascending target order in the tables and manifests.
        rows.sort(key=lambda row: row.get("target_speedup") or 0)
        results.sort(key=lambda row: row.get("target_speedup") or 0)
        suite["status"] = workflow_status([result.get("status", "FAIL") for result in results], useful=bool(baseline))
    except Exception as error:
        suite["errors"].append({"stage": "model", "error": safe_error(error), "traceback": safe_error(traceback.format_exc())})
        suite["status"] = "PARTIAL" if suite["baseline"] or results else "FAIL"
        for target in GFLOPS_TARGETS:
            label = f"target-{target:.2f}x"
            if not any(row.get("configuration_id") == label for row in rows):
                rows.append({"model": f"YOLO26{variant}", "model_variant": variant, "configuration_id": label,
                             "policy": "gflops-target", "target_speedup": target, "target_status": "NOT_RUN",
                             "status": "NOT_RUN", "structural_status": "NOT_RUN", "reload_status": "NOT_RUN",
                             "validation_status": "NOT_RUN", "latency_status": "NOT_RUN", "eda_status": "NOT_RUN"})
    finally:
        write_json(manifest, suite)
        write_csv(folder / "metrics.csv", rows)
    return suite


def gflops_execute(folder):
    report = {"format_version": 2, "status": "NOT_RUN", "mode": MODE, "models": [], "errors": []}
    manifest = folder / "suite_manifest.json"
    prior = json.loads(manifest.read_text(encoding="utf-8")) if manifest.is_file() else None
    bound = False
    dataset = None
    try:
        expected = EXPECTED_DATASET_FINGERPRINT
        portable = None
        if DATASET_MANIFEST_PATH is not None:
            portable = json.loads(Path(DATASET_MANIFEST_PATH).read_text(encoding="utf-8"))
            if portable.get("dataset_id") != DATASET_ID or portable.get("provenance_status") != "VERIFIED":
                raise ValueError("Manifest dataset sumber belum VERIFIED atau ID berbeda.")
            expected = portable["val_fingerprint"]
        dataset = gflops_prepare_dataset(MODE, DATA_YAML, DATASET_ROOT, folder / "dataset_checks" / uuid.uuid4().hex[:8], seed=SEED,
                                         expected_fingerprint=expected, dataset_id=DATASET_ID)
        dataset_record = None
        if dataset is not None and (EXPORT_DATASET_MANIFEST or DATASET_MANIFEST_ONLY):
            dataset_record, _, _ = workflow_dataset_manifest(dataset, DATA_YAML, DATASET_ROOT, Path(dataset["manifest_path"]).parent)
            if portable is not None and dataset_record["train_fingerprint"] != portable["train_fingerprint"]:
                raise ValueError("Fingerprint train Filkom berbeda dari sumber.")
            dataset["train_fingerprint"] = dataset_record["train_fingerprint"]
        workflow_bind(folder, "resume_contract.json", {
            "implementation": IMPLEMENTATION_SHA256, "variant": NOTEBOOK_VARIANT, "platform": PLATFORM,
            "targets": list(GFLOPS_TARGETS), "imgsz": IMGSZ, "batch": BATCH, "seed": SEED,
            "diagnostics": RUN_DIAGNOSTIC_CONTROLS, "diagnostic_ratios": list(PRUNING_RATIOS) if RUN_DIAGNOSTIC_CONTROLS else [],
            "dataset_fingerprint": (dataset or {}).get("fingerprint"), "mode": MODE,
            "train_fingerprint": (dataset or {}).get("train_fingerprint"),
            "dataset_provenance_status": (dataset or {}).get("provenance_status"),
            "device": str(TARGET_DEVICE), "gpu": ENVIRONMENT.get("gpu"), "cuda_build": ENVIRONMENT.get("cuda_build"),
            "packages": ENVIRONMENT.get("packages"), "validation_workers": WORKERS,
            "run_latency": RUN_LATENCY_BENCHMARK, "latency_warmup": LATENCY_WARMUP, "latency_repeats": LATENCY_REPEATS})
        bound = True
        write_json(folder / "config.json", gflops_config())
        if dataset_record is not None:
            write_json(folder / "dataset_manifest.json", dataset_record)
        if DATASET_MANIFEST_ONLY:
            report["reason"] = "Preflight dataset saja; pruning tidak dijalankan."
            return report
        if MODE == "test":
            report["test"] = gflops_run_locked_test(Path(STUDENT_LOCK_PATH), dataset, folder / "test",
                                                    checkpoint_path=TEST_STUDENT_CHECKPOINT_PATH,
                                                    baseline_checkpoint_path=TEST_BASELINE_CHECKPOINT_PATH)
            report["status"] = report["test"]["status"]
            return report
        for variant in MODEL_VARIANTS:
            model = gflops_run_model(variant, folder / f"yolo26{variant}", dataset)
            report["models"].append(model)
            write_json(manifest, report)
        rows = [row for model in report["models"] for row in model["rows"]]
        write_csv(folder / "metrics.csv", rows)
        if any(model["results"] for model in report["models"]):
            plot_model_comparison(report["models"], folder / "comparison")
        report["status"] = workflow_status([model["status"] for model in report["models"]],
                                            useful=any(model["baseline"] for model in report["models"]))
        report["evaluation_status"] = workflow_status([row.get("validation_status", "NOT_RUN") for row in rows])
        report["selection"] = {"status": "NOT_RUN", "selected": None,
                                "reason": "Gabungkan full validation N/S; hanya raw dengan metrik lengkap yang eligible."}
        gflops_export_candidates(report, dataset, folder)
    except (Exception, KeyboardInterrupt) as error:
        if prior is not None and not bound:
            write_json(folder / "resume_rejection.json", {"status": "FAIL", "error": safe_error(error)})
            return {**prior, "status": "PARTIAL", "resume_rejected": safe_error(error)}
        if isinstance(error, KeyboardInterrupt) and not report["models"]:
            for variant in MODEL_VARIANTS:
                partial = folder / f"yolo26{variant}" / "model_manifest.json"
                if partial.is_file():
                    report["models"].append(json.loads(partial.read_text(encoding="utf-8")))
        report["status"] = "PARTIAL" if report["models"] else "FAIL"
        report["errors"].append({"error": safe_error(error), "traceback": safe_error(traceback.format_exc())})
    finally:
        if bound or prior is None:
            write_json(manifest, report)
    return report


def gflops_export_candidates(report, dataset, folder):
    """Export the measured rows and a cross-session comparison contract."""
    folder = Path(folder).resolve()
    rows = []
    for model in report["models"]:
        for source in model["rows"]:
            row = dict(source)
            for key in ("checkpoint", "baseline_checkpoint"):
                if row.get(key + "_path"):
                    row[key + "_relative_path"] = Path(row[key + "_path"]).resolve().relative_to(folder).as_posix()
            rows.append(row)
    contract = {"implementation_sha256": IMPLEMENTATION_SHA256, "source_sha256": SOURCE_CHECKPOINT_SHA256,
                "platform": PLATFORM,
                "seed": SEED, "imgsz": IMGSZ, "validation_batch": BATCH,
                "validation_mode": "one-to-many NMS FP32", "latency_mode": "unfused dual-head FP32 batch 1",
                "latency_warmup": LATENCY_WARMUP, "latency_repeats": LATENCY_REPEATS,
                **{key: ENVIRONMENT.get(key) for key in ("gpu", "cuda_build", "packages", "flops_method")}}
    record = {"format_version": 2, "mode": MODE, "status": report["status"],
              "evaluation_status": report["evaluation_status"],
              "model_variants": [model["model_variant"] for model in report["models"]],
              "dataset_provenance_status": (dataset or {}).get("provenance_status"),
              "dataset_fingerprint": (dataset or {}).get("fingerprint"),
              "comparison_contract": contract, "rows": rows}
    write_json(folder / "student_candidates.json", record)


def gflops_finalize_student(folder, comparison_paths):
    """Can run again after attaching the other notebook's complete output folder."""
    if MODE != "full_val" or not comparison_paths:
        return {"status": "NOT_RUN", "selected": None,
                "reason": "Isi COMPARISON_CANDIDATE_PATHS dengan ekspor full_val varian lainnya."}
    selection = gflops_select_candidate_files([Path(folder) / "student_candidates.json", *comparison_paths])
    if selection.get("selected"):
        if ":v" not in (selection["selected"].get("raw_artifact_ref") or ""):
            return {"status": "NOT_RUN", "selected": None, "provisional": selection["selected"],
                    "reason": "Publikasi raw terpilih belum dikonfirmasi; retry upload sebelum mengunci student."}
        gflops_lock_student(selection, Path(folder) / "student_lock.json")
    write_json(Path(folder) / "comparison/final_selection.json", selection)
    return selection

In [ ]:
BUNDLE = gflops_execute(SUITE_DIR)
print("Output", SUITE_DIR, "| Status", BUNDLE["status"])

## 11 — EDA Identitas, Cakupan, dan Channel
Analisis mengikuti urutan notebook 03 dan 04. Rekap mencakup **semua konfigurasi model pada notebook ini**.
Overview channel ditampilkan untuk setiap konfigurasi yang EDA-nya berhasil.
`EDA_CONFIGURATIONS=()` berarti semua hasil; isi tuple ID untuk membatasi tampilan.
Detail layer, ledger, mask, distribusi LAMP, dan kurva individual mengikuti
`EDA_DETAIL_CONFIGURATIONS`, default `()` untuk semua konfigurasi. Gunakan `()` pada keduanya untuk
menampilkan seluruh detail. Pengaturan ini hanya mengatur tampilan; ekspor analisis tetap lengkap.
Sel 11–14 dapat dijalankan ulang dari `BUNDLE` tanpa pruning atau validasi ulang.

### 11.1 Identitas eksperimen dan cakupan arsitektur
Periksa checkpoint, hash, pasangan baseline, jumlah citra, fingerprint, dan mode evaluasi.
Tabel cakupan membedakan group eligible dari graph, kapasitas policy target, channel independen,
dan perubahan fisik Conv-output. Group yang ditolak graph beserta alasannya tetap tersedia.
Layer pada group graph yang diterima dapat mempunyai kapasitas nol karena proteksi target tambahan.

In [ ]:
from IPython.display import Image as NotebookImage
from IPython.display import Markdown

try:
    ANALYSIS = gflops_build_analysis(BUNDLE, SUITE_DIR)
except Exception as error:
    print("EDA suite PARTIAL:", safe_error(error))
    ANALYSIS = {"status": "PARTIAL", "plots": [], "tables": gflops_eda_tables(BUNDLE, SUITE_DIR)}
TABLES = ANALYSIS["tables"]
SUMMARY = pd.DataFrame(TABLES["metrics"])
EDA_RUNS = []


def show_eda_table(rows, columns=None):
    frame = pd.DataFrame(rows)
    if frame.empty:
        print("UNAVAILABLE — lihat status pengukuran pada tabel ringkasan.")
        return
    if columns:
        frame = frame.reindex(columns=columns)
    with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", 100):
        display(frame)


def show_eda_images(folder, *patterns):
    for pattern in patterns:
        for path in sorted(folder.glob(pattern)):
            display(NotebookImage(filename=str(path)))


show_eda_table(TABLES["sources"], ["model", "requested_artifact", "sha256", "source_verification",
                                  "baseline_validation_parity_status"])
show_eda_table(TABLES["metrics"], ["model", "configuration_id", "policy", "stage", "ratio", "actual_eligible_ratio",
    "pruned_group_count", "evaluation_mode", "validation_image_count", "dataset_provenance_status",
    "structural_status", "reload_status", "validation_status", "eda_status"])
show_eda_table(TABLES["architecture"], ["model", "index", "type", "from", "region", "parameters"])
show_eda_table(TABLES["coverage"])

### 11.2 Perbandingan channel sebelum dan sesudah pruning
Diagram **Before Pruning** biru, **After Pruning** merah, dan **Channel Comparison** overlay
memakai urutan seluruh Conv dan batas sumbu yang sama. Layer yang tetap utuh ikut ditampilkan.
Bandingkan pola extended, P5-only, P5-protected, dan kandidat GFLOPs pada model yang sama.
Jumlah Conv-output yang terhapus dapat melebihi channel independen karena residual mengikat
beberapa output. Konsentrasi penghapusan merupakan observasi distribusi; kontras P5 di bagian 13
memerlukan evaluasi mAP untuk menguji hipotesis sensitivitas.

In [ ]:
for model in BUNDLE["models"]:
    model_name = model["model_name"]
    available = {result["configuration_id"]: result for result in model["results"]
                 if result["eda_status"] == "PASS"}
    if not available:
        print(f"{model_name} — EDA belum tersedia. Lihat status tiap konfigurasi pada tabel.")
    for key, result in available.items():
        if EDA_CONFIGURATIONS and key not in EDA_CONFIGURATIONS:
            continue
        folder = SUITE_DIR / model_name.lower() / result.get("artifact_configuration_id", key) / "eda"
        EDA_RUNS.append((model_name, key, result, folder))
        display(Markdown(f"#### {model_name} · {key}"))
        show_eda_images(folder, "channels_overview.png", "channel_reduction_percent.png")
EDA_DETAILS = [entry for entry in EDA_RUNS if not EDA_DETAIL_CONFIGURATIONS or entry[1] in EDA_DETAIL_CONFIGURATIONS]
for model_name, key, result, folder in EDA_DETAILS:
    display(Markdown(f"#### Detail layer · {model_name} · {key}"))
    selected = [row for row in TABLES["layers"] if row["model"] == model_name and row["configuration_id"] == key]
    show_eda_table(selected, ["layer_name", "module_type", "region", "in_channels_before", "in_channels_after",
        "out_channels_before", "out_channels_after", "out_channels_removed", "out_channels_reduction_pct",
        "parameters_before", "parameters_after", "estimated_flops_before", "estimated_flops_after",
        "dependency_group", "protection_reason"])
    show_eda_images(folder, "channels_detail_*.png")

### 11.3 Keputusan LAMP, kapasitas group, dan ledger indeks asli
Lihat group yang diterima/ditolak, rentang skor finite, kapasitas, dan jumlah channel yang dipilih.
Skor `+inf` dari pembatas kapasitas tidak diperlakukan sebagai importance.
Heatmap memakai indeks asli sebelum pruning. Ledger juga mencatat kolom input konsumen dan BN,
termasuk offset concat dan seluruh konsumen fan-out. Tabel lengkap seluruh konfigurasi diekspor
ke `comparison/analysis/groups.csv` dan `ledger.csv`. Heatmap menggambarkan channel bobot.

In [ ]:
for model_name, key, result, folder in EDA_DETAILS:
    display(Markdown(f"#### LAMP dan dependency · {model_name} · {key}"))
    selected = [row for row in TABLES["groups"] if row["model"] == model_name and row["configuration_id"] == key]
    show_eda_table(selected, ["root", "group_id", "decision", "initial_channels", "capacity_eligible",
        "removed", "score_min", "score_median", "score_max", "coupled_output_layers", "reason"])
    changed = [row for row in TABLES["ledger"] if row["model"] == model_name and row["configuration_id"] == key
               and row.get("removed_original_indices")]
    show_eda_table(changed, ["layer", "module_type", "dimension", "channels_before", "channels_after",
                            "kept_original_indices", "removed_original_indices", "dependency_group_ids"])
    show_eda_images(folder, "channel_mask_*.png", "lamp_score_distribution.png", "lamp_group_capacity.png")

## 12 — EDA Efisiensi dan Metrik Deteksi
### 12.1 Efisiensi per model, layer, dan stage
Ringkasan memuat seluruh konfigurasi, masing-masing dibandingkan dengan baseline variannya.
GFLOPs speed-up dan latency speed-up memakai baseline yang sama untuk varian tersebut.
Latency adalah raw dual-head forward FP32 batch satu, dengan p50 dan p95, di luar preprocessing,
NMS, dan I/O. Pengukuran CUDA yang belum dijalankan tetap `NOT_RUN`.
Estimasi operasi per layer/stage mencakup Conv/BN; total GFLOPs model juga memasukkan attention.
Ukuran checkpoint pascapruning dibandingkan dengan baseline FP32 yang disimpan ulang.
Ukuran file baseline sumber ditampilkan terpisah karena format penyimpanannya dapat berbeda.

In [ ]:
show_eda_table(TABLES["metrics"], [
    "model", "configuration_id", "stage", "parameters", "parameter_reduction_pct",
    "gflops", "gflops_reduction_pct", "gflops_speedup", "checkpoint_bytes", "checkpoint_reduction_pct",
    "latency_median_ms", "latency_p95_ms", "latency_speedup", "latency_status",
])
for model_name, key, result, folder in EDA_DETAILS:
    display(Markdown(f"#### Efisiensi stage · {model_name} · {key}"))
    selected = [row for row in TABLES["stages"] if row["model"] == model_name and row["configuration_id"] == key]
    show_eda_table(selected)
    show_eda_images(folder, "stage_physical_conv_outputs.png", "stage_parameters.png", "stage_estimated_flops.png")

### 12.2 Akurasi keseluruhan dan AP/P/R/F1 per kelas untuk seluruh konfigurasi
`delta_map50_95_pp = 100 × (mAP_pruned − mAP_baseline)` menyatakan poin persentase.
Perubahan relatif menggunakan mAP baseline sebagai denominator dan tidak terdefinisi jika baseline nol.
Rekap per kelas mengikuti notebook 03, dilengkapi selisih AP terhadap baseline dengan ID kelas yang sama.
Support menunjukkan jumlah ground-truth; kelas tanpa metrik tetap kosong.
Pada `smoke`, semua angka hanya memeriksa pipeline di subset tetap. Interpretasi akurasi penelitian
serta pemilihan student membutuhkan seluruh validation Poltrada V3.

In [ ]:
show_eda_table(TABLES["metrics"], [
    "model", "configuration_id", "stage", "map50_95", "map50", "precision", "recall",
    "delta_map50_95_pp", "relative_map50_95_pct", "validation_status", "evaluation_mode", "validation_image_count",
])
PER_CLASS = pd.DataFrame(TABLES["per_class_metrics"])
for model in BUNDLE["models"]:
    model_name = model["model_name"]
    display(Markdown(f"#### Semua konfigurasi · {model_name}"))
    selected = [row for row in TABLES["per_class_metrics"] if row["model"] == model_name]
    show_eda_table(selected, ["configuration_id", "stage", "class_id", "name", "support", "ap50_95", "ap50",
                            "precision", "recall", "f1", "delta_ap50_95_pp", "delta_ap50_pp"])
    show_eda_images(SUITE_DIR / "comparison/analysis" / model_name.lower(),
                    "configuration_metrics.png", "per_class_ap_all.png")

### 12.3 Kurva PR/F1/confidence, confusion matrix, dan protokol efektif
Tampilkan kurva baseline dan pruned dari validator aktual untuk konfigurasi detail.
Precision/recall/F1 per kelas berasal dari titik confidence terbaik menurut smoothed mean F1 SDK
pada IoU 0,50. Confusion matrix memakai **baris predicted, kolom true**, termasuk background.
Threshold confidence dan IoU pencocokan matrix berbeda dari IoU NMS validator; baca metadata aktual.
Seluruh konfigurasi menyimpan kurva dan matrix di folder EDA masing-masing saat evaluasi tersedia.
Tabel protokol mengungkap split, jumlah citra, FP32, one-to-many/NMS, dan fingerprint setiap evaluasi.

In [ ]:
show_eda_table(TABLES["validation_protocol"], [
    "model", "configuration_id", "validation_status", "evaluation_mode", "split", "validation_image_count",
    "dataset_fingerprint", "dataset_provenance_status", "imgsz", "batch", "half", "conf", "iou", "max_det",
    "nms", "end2end", "augment", "rect", "device",
])
for model_name, key, result, folder in EDA_DETAILS:
    display(Markdown(f"#### Kurva dan matrix · {model_name} · {key}"))
    for stage in ("baseline", "pruned"):
        metrics = result[stage].get("validation") or {}
        display(Markdown(f"##### {stage.capitalize()}"))
        if not metrics:
            print("NOT_RUN — metrik deteksi belum tersedia.")
            continue
        display(metrics.get("metric_semantics", {}))
        matrix = metrics.get("confusion_matrix", {})
        display({field: matrix.get(field) for field in (
            "status", "labels", "confidence_threshold", "matching_iou_threshold", "validation_iou")})
        show_eda_images(folder / stage, "class_metrics.png", "evaluation_curve_*.png", "confusion_matrix.png")

## 13 — Analisis Trade-off Akurasi dan Efisiensi
### 13.1 Bukti awal extended dari tabel historis pengguna
Tabel 12 baris ini dipertahankan dengan provenance **UNVERIFIED**. Pada extended 10%, mAP50–95
yang dilaporkan turun dari 0,5261 ke 0,1692 pada N dan 0,5612 ke 0,2546 pada S; pada 20%,
keduanya di bawah 0,003. Status PASS pada tabel adalah status asli yang dilaporkan.
Artefak run, fingerprint, konfigurasi evaluator, dan counter perlu diverifikasi.
Tabel ini diarsipkan tersendiri dan tidak dimasukkan ke grafik hasil suite aktual.

In [ ]:
from io import StringIO
HISTORICAL_EXTENDED_CSV = 'index,model,model_variant,policy,ratio,parameters,gflops,checkpoint_bytes,map50_95,map50,precision,recall,parameter_reduction_pct,gflops_reduction_pct,delta_map50_95_pp,relative_map50_95_pct,actual_eligible_ratio,pruned_group_count,latency_median_ms,structural_status,validation_status,provenance\n0,YOLO26n,n,baseline,0.0,2506140,6.025830,10516163,5.261053e-01,0.742727,0.769929,0.690453,0.000000,0.000000,0.000000,0.000000,0.000000,0,13.690454,BASELINE,PASS,UNVERIFIED\n1,YOLO26n,n,extended,0.1,2236276,5.691846,9433063,1.692238e-01,0.286811,0.641445,0.269618,10.768114,5.542539,-35.688155,-67.834620,0.100195,7,13.545400,PASS,PASS,UNVERIFIED\n2,YOLO26n,n,extended,0.2,2013751,5.075542,8541159,1.987566e-03,0.006276,0.002626,0.066285,19.647306,15.770241,-52.411776,-99.622211,0.200389,10,13.722865,PASS,PASS,UNVERIFIED\n3,YOLO26n,n,extended,0.3,1800137,4.532680,7684519,8.236284e-04,0.003035,0.001787,0.051145,28.170932,24.779164,-52.528170,-99.843448,0.300097,16,14.463144,PASS,PASS,UNVERIFIED\n4,YOLO26n,n,extended,0.4,1625418,4.083136,6982951,3.451889e-07,0.000001,0.000131,0.000861,35.142570,32.239447,-52.610498,-99.999934,0.400292,27,13.931716,PASS,PASS,UNVERIFIED\n5,YOLO26n,n,extended,0.5,1536092,3.494195,6623847,0.000000e+00,0.000000,0.000000,0.000000,38.706856,42.013051,-52.610533,-100.000000,0.500000,35,13.696097,PASS,PASS,UNVERIFIED\n6,YOLO26s,s,baseline,0.0,9952508,23.006822,40378115,5.611909e-01,0.773451,0.775606,0.721401,0.000000,0.000000,0.000000,0.000000,0.000000,0,14.290742,BASELINE,PASS,UNVERIFIED\n7,YOLO26s,s,extended,0.1,8858156,21.711862,35995303,2.545885e-01,0.412033,0.674990,0.311926,10.995741,5.628591,-30.660239,-54.634238,0.100195,7,13.983301,PASS,PASS,UNVERIFIED\n8,YOLO26s,s,extended,0.2,7936472,19.764531,32304935,2.826255e-03,0.006314,0.164648,0.013613,20.256562,14.092738,-55.836468,-99.496383,0.200146,9,14.155988,PASS,PASS,UNVERIFIED\n9,YOLO26s,s,extended,0.3,7075142,17.591168,28855527,6.153192e-07,0.000002,0.000198,0.000842,28.910964,23.539341,-56.119032,-99.999890,0.300097,16,14.166117,PASS,PASS,UNVERIFIED\n10,YOLO26s,s,extended,0.4,6426986,15.373741,26256487,0.000000e+00,0.000000,0.000000,0.000000,35.423453,33.177470,-56.119094,-100.000000,0.400049,29,14.516314,PASS,PASS,UNVERIFIED\n11,YOLO26s,s,extended,0.5,6077564,13.034906,24857383,0.000000e+00,0.000000,0.000000,0.000000,38.934347,43.343303,-56.119094,-100.000000,0.500000,35,14.411619,PASS,PASS,UNVERIFIED\n'

HISTORICAL_EXTENDED = pd.read_csv(StringIO(HISTORICAL_EXTENDED_CSV))
show_eda_table(HISTORICAL_EXTENDED.to_dict("records"))
(SUITE_DIR / "comparison/analysis/historical_unverified.csv").write_text(HISTORICAL_EXTENDED_CSV, encoding="utf-8")

### 13.2 Kontrol extended dan pencarian target komputasi
Seperti notebook 03, lihat hubungan rasio kontrol dengan mAP dan reduksi parameter/GFLOPs.
Kandidat utama memakai target komputasi 1,02×/1,05×/1,10×/1,20×/1,30×/1,40×/1,50×/1,60×/1,70×. Tabel tier menunjukkan batas kapasitas
dan tabel target menunjukkan capaian atau `TARGET_NOT_REACHED`.
Profil pencarian hanya mengukur GFLOPs; mAP dinilai pada kandidat eksperimen yang dievaluasi.
Bandingkan GFLOPs speed-up dan latency speed-up yang terukur untuk menilai manfaat di perangkat.

In [ ]:
show_eda_table(TABLES["target_tiers"])
show_eda_table(TABLES["target_search"], ["model", "target", "status", "tier", "channels", "gflops", "speed_up"])
for model in BUNDLE["models"]:
    model_name = model["model_name"]
    display(Markdown(f"#### Sweep dan kapasitas · {model_name}"))
    show_eda_images(SUITE_DIR / "comparison/analysis" / model_name.lower(),
                    "extended_sweep.png", "target_search_capacity.png")

### 13.3 Trade-off seluruh konfigurasi model ini
Setiap titik berlabel ID konfigurasi. Grafik per varian memakai baseline variannya sendiri;
Hasil N/S berasal dari notebook terpisah. Penurunan mAP besar tetap merupakan temuan negatif
walaupun struktur dan reload lulus. Angka historis pengguna berprovenance `UNVERIFIED` disimpan
terpisah dari titik suite aktual. Satu seed tidak mengukur variasi antar-seed.

In [ ]:
show_eda_table(ANALYSIS["plots"], ["model", "plot", "status", "measured_values", "reason"])
for model in BUNDLE["models"]:
    display(Markdown(f"#### Trade-off · {model['model_name']}"))
    show_eda_images(SUITE_DIR / "comparison/analysis" / model["model_name"].lower(), "tradeoff_*.png")
show_eda_images(SUITE_DIR / "comparison", "map_vs_*.png")

### 13.4 Kontras sensitivitas P5 dan kelayakan student
**P5-only vs baseline** mengukur perubahan saat hanya channel P5 pilihan extended 10% yang dibuang.
**P5-protected vs extended 10%** membandingkan alokasi ulang dengan jumlah penghapusan independen yang sama.
Bandingkan delta mAP dan AP per kelas sambil membaca perbedaan GFLOPs kedua alokasi.
Distribusi channel sendiri belum membuktikan penyebab runtuhnya akurasi.

Kelayakan student mengikuti aturan full_val terverifikasi, target tercapai, kehilangan mAP50–95
maksimal dua poin, serta parameter/GFLOPs/latency yang lengkap. Pemilihan berasal dari pipeline full_val
dan mencakup gabungan N/S. Tabel EDA tidak mengganti atau mengunci ulang pilihan tersebut.

In [ ]:
show_eda_table(TABLES["p5_contrasts"])
show_eda_table(TABLES["p5_per_class"], ["model", "contrast", "reference", "class_id", "name", "support",
    "reference_ap50_95", "ap50_95", "delta_ap50_95_pp"])
show_eda_table(TABLES["student_eligibility"])
display(BUNDLE.get("selection", BUNDLE.get("test")))

## 14 — Ringkasan, Artefak, dan Reproducibility
Ringkasan mempertahankan status struktur, oracle, replay, reload, evaluasi, latency, dan EDA secara terpisah.
`PASS` pada struktur tidak menjadi penilaian akurasi. Metrik yang belum diukur tetap kosong atau `NOT_RUN`.
Seluruh tabel analisis tersedia sebagai CSV/JSON dan grafik sebagai PNG/SVG pada `comparison/analysis/`.
Folder EDA tiap konfigurasi berisi inventory, ledger, skor, diagram channel/stage, dan hasil validator.
Simpan juga manifest dataset, `target_search.json`, kontras P5, environment, checkpoint, dan hash.
Setiap notebook mengekspor `student_candidates.json`. Untuk pemilihan satu student N/S, simpan
folder output notebook pertama lalu lampirkan sebagai dataset pada notebook kedua. Isi
`COMPARISON_CANDIDATE_PATHS` dengan path file kandidat varian lain dan jalankan ulang sel terakhir.
Pemilihan memerlukan kedua full_val VERIFIED dengan baris kandidat lengkap, termasuk dari suite PARTIAL, fingerprint dataset yang sama,
serta protokol, versi implementasi, dan perangkat latency yang sama. Checkpoint terpilih beserta
baseline-nya wajib tersedia dan cocok dengan SHA-256 sebelum `student_lock.json` dibuat.
Gunakan objek DetectionModel FP32 beserta manifest untuk loader KD yang mempertahankan lebar pruned.

In [ ]:
print("Folder suite", SUITE_DIR)
for filename in ("suite_manifest.json", "config.json", "environment.json", "metrics.csv", "student_candidates.json", "student_lock.json"):
    path = SUITE_DIR / filename
    if path.is_file():
        print(path.name, path)
print("Analisis", SUITE_DIR / "comparison/analysis", "| Status ekspor", ANALYSIS["status"])
show_eda_table(TABLES["verification"])
show_eda_table(TABLES["metrics"])
if MODE == "full_val" and COMPARISON_CANDIDATE_PATHS:
    BUNDLE["selection"] = gflops_finalize_student(SUITE_DIR, COMPARISON_CANDIDATE_PATHS)
    write_json(SUITE_DIR / "suite_manifest.json", BUNDLE)
display(BUNDLE.get("selection", BUNDLE.get("test")))